10 GHSL Mumbai Metropolitan Production
Uses the frozen Chennai-validated baseline. Run one cell at a time. Cells 3-6 are gates; do not launch Cell 7 unless all pass.


In [1]:
# CELL 1: BOOTSTRAP
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess
drive.mount('/content/drive', force_remount=False)
D=Path('/content/drive/MyDrive'); R=D/'remote_exec.sh'; K=D/'hadoop-hdfs.pem'; L=Path('/content/hadoop-hdfs.pem')
if not R.is_file(): raise FileNotFoundError(R)
os.chmod(R,0o700)
if not L.is_file(): shutil.copy2(K,L)
os.chmod(L,0o600)
def run_remote(command,timeout=300,show_output=True):
    r=subprocess.run(['bash',str(R),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if r.returncode: raise RuntimeError(f'Remote command failed with return code {r.returncode}.')
    return r
print('CELL 1: SUCCESS')

Mounted at /content/drive
CELL 1: SUCCESS


In [2]:
# CELL 2: FROZEN MUMBAI CONFIGURATION

ACTIVE_CITY_SLUG = "mumbai_metro"

ACTIVE_CITY = {
    "city_name": "Mumbai Metropolitan Region",
    "city_slug": "mumbai_metro",
    "seed_lon": 72.8777,
    "seed_lat": 19.0760,
    "target_crs": "ESRI:54009",
    "expected_grid_cells": 4900,
    "expected_complete_cells": 4900,
    "expected_partial_cells": 0,
    "extent_mollweide": {
        "min_x": 7018000.0,
        "min_y": 2306000.0,
        "max_x": 7088000.0,
        "max_y": 2376000.0,
    },
    "hdfs_output_base": (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/cities/"
        "city=mumbai_metro"
    ),
    "remote_work_dir": (
        "/home/ubuntu/ghsl_multicity/mumbai_metro"
    ),
}

assert (
    ACTIVE_CITY["extent_mollweide"]["max_x"]
    - ACTIVE_CITY["extent_mollweide"]["min_x"]
    == 70000
)

print(ACTIVE_CITY)
print("CELL 2: SUCCESS")

{'city_name': 'Mumbai Metropolitan Region', 'city_slug': 'mumbai_metro', 'seed_lon': 72.8777, 'seed_lat': 19.076, 'target_crs': 'ESRI:54009', 'expected_grid_cells': 4900, 'expected_complete_cells': 4900, 'expected_partial_cells': 0, 'extent_mollweide': {'min_x': 7018000.0, 'min_y': 2306000.0, 'max_x': 7088000.0, 'max_y': 2376000.0}, 'hdfs_output_base': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=mumbai_metro', 'remote_work_dir': '/home/ubuntu/ghsl_multicity/mumbai_metro'}
CELL 2: SUCCESS


HDFS Startup - IP changed.

In [5]:
# ============================================================
# START AND VERIFY HDFS AFTER EC2 RESTART
#
# Safe:
#   - Starts existing HDFS daemons
#   - Does not format the NameNode
#   - Does not delete or modify stored HDFS datasets
# ============================================================

cmd = r"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "HDFS STARTUP AFTER EC2 RESTART"
echo "============================================================"

echo
echo "Current Java processes:"
jps || true

echo
echo "Starting HDFS daemons..."

start-dfs.sh

echo
echo "Waiting for HDFS startup..."

sleep 10

echo
echo "Java processes after startup:"

jps

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    if ! jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
    then
        echo "ERROR: $daemon is not running."
        exit 20
    fi
done

echo
echo "HDFS daemons: PASS"

hdfs dfsadmin -safemode get

if ! hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"
then
    echo
    echo "Waiting for HDFS safe mode to turn OFF..."

    for attempt in $(seq 1 12); do
        sleep 5

        if hdfs dfsadmin -safemode get \
            | grep -qi "Safe mode is OFF"
        then
            break
        fi
    done
fi

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS safe mode: OFF"

echo
echo "Checking existing curated city data..."

hdfs dfs -test -e \
    /data/ghsl/curated/cities/city=delhi_ncr

echo "Existing Delhi NCR HDFS data: PASS"

echo
echo "HDFS_STARTUP_SUCCESS"
"""

startup_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("HDFS STARTUP RESULT")
print("=" * 80)

print(
    "Return code:",
    startup_result.returncode,
)

print("NameNode formatted: NO")
print("HDFS data deleted: NO")
print("HDFS STARTUP: SUCCESS")

HDFS STARTUP AFTER EC2 RESTART

Current Java processes:
1733 Jps

Starting HDFS daemons...
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [ip-172-31-30-94]

Waiting for HDFS startup...

Java processes after startup:
2419 Jps
2266 SecondaryNameNode
2042 DataNode
1902 NameNode

HDFS daemons: PASS
Safe mode is ON

Waiting for HDFS safe mode to turn OFF...
HDFS safe mode: OFF

Checking existing curated city data...
Existing Delhi NCR HDFS data: PASS

HDFS_STARTUP_SUCCESS

REMOTE STDERR:


HDFS STARTUP RESULT
Return code: 0
NameNode formatted: NO
HDFS data deleted: NO
HDFS STARTUP: SUCCESS


In [6]:
# ============================================================
# CELL 3: BASELINE AND HDFS PREFLIGHT
#
# Corrected behavior:
#   - Verifies the frozen SHA-256 inventory
#   - Compiles baseline Python files into /tmp
#   - Does not create __pycache__ in the baseline
#   - Removes bytecode created by the previous Cell 3 attempt
#   - Verifies all permanent baseline files are read-only
#
# Does not:
#   - modify baseline source files
#   - launch Spark
#   - modify HDFS
# ============================================================

cmd = r"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

BASE="/home/ubuntu/ghsl_multicity_baseline"
COMPILE_ROOT="/tmp/ghsl_baseline_compile_check"

echo "============================================================"
echo "MUMBAI BASELINE AND HDFS PREFLIGHT"
echo "============================================================"


# ------------------------------------------------------------
# HDFS service validation.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Required baseline files.
# ------------------------------------------------------------

required_files=(
    "$BASE/baseline_manifest.json"
    "$BASE/SHA256SUMS"
    "$BASE/README.txt"
    "$BASE/source_city_config.json"
    "$BASE/batch/run_city_epochs.py"
    "$BASE/templates/population_epoch_template.py"
    "$BASE/templates/smod_epoch_template.py"
    "$BASE/templates/built_surface_epoch_template.py"
)

for file in "${required_files[@]}"; do
    if [ ! -f "$file" ]; then
        echo "ERROR: Required baseline file missing:"
        echo "$file"
        exit 20
    fi
done

echo "Required baseline files: PASS"


# ------------------------------------------------------------
# Verify the permanent baseline inventory before doing any
# compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Baseline SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Remove only generated Python bytecode created by the earlier
# preflight attempt.
#
# No source, manifest, checksum, config, or README is removed.
# ------------------------------------------------------------

find "$BASE" \
    -type d \
    -name "__pycache__" \
    -print \
    -exec rm -rf {} + \
    2>/dev/null \
    || true

find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    -print \
    -delete \
    2>/dev/null \
    || true

echo "Generated baseline bytecode removed: PASS"


# ------------------------------------------------------------
# Compile outside the frozen baseline.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

mkdir -p \
    "$COMPILE_ROOT/batch" \
    "$COMPILE_ROOT/templates"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile \
    "$BASE/batch/run_city_epochs.py"

for file in \
    "$BASE/templates/population_epoch_template.py" \
    "$BASE/templates/smod_epoch_template.py" \
    "$BASE/templates/built_surface_epoch_template.py"

do
    PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
    python3 -m py_compile "$file"
done

echo "Baseline Python compilation: PASS"


# ------------------------------------------------------------
# Confirm compilation created nothing inside the baseline.
# ------------------------------------------------------------

if find "$BASE" \
    -type d \
    -name "__pycache__" \
    | grep -q .
then
    echo "ERROR: Compilation created __pycache__ in baseline."
    exit 21
fi

if find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    | grep -q .
then
    echo "ERROR: Compilation created bytecode in baseline."
    exit 22
fi

echo "Baseline remains bytecode-free: PASS"


# ------------------------------------------------------------
# Verify all permanent baseline files are read-only.
#
# The baseline directories remain owner-writable so future
# city workspaces can be created elsewhere. Only files need
# to remain frozen.
# ------------------------------------------------------------

writable_files="$(
    find "$BASE" \
        -type f \
        -perm -u=w \
        -print
)"

if [ -n "$writable_files" ]; then
    echo "ERROR: Writable permanent baseline files found:"
    echo "$writable_files"
    exit 23
fi

echo "Baseline permanent files read-only: PASS"


# ------------------------------------------------------------
# Recheck the inventory after cleanup and compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Post-check SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Clean temporary compilation output.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

echo
echo "BASELINE_PREFLIGHT_SUCCESS"
"""

preflight_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 3 RESULT")
print("=" * 80)

print(
    "Return code:",
    preflight_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Baseline source files modified: NO")
print("CELL 3: SUCCESS")

MUMBAI BASELINE AND HDFS PREFLIGHT
HDFS services: PASS
Required baseline files: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Baseline SHA-256 inventory: PASS
Generated baseline bytecode removed: PASS
Baseline Python compilation: PASS
Baseline remains bytecode-free: PASS
Baseline permanent files read-only: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Post-check SHA-256 inventory: PASS

BASELINE_PREFLIGHT_SUCCESS

REMOTE STDERR:


CELL 3 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Baseline source files modified: NO
CELL 3: SUCCESS


In [7]:
# ============================================================
# CELL 4: CREATE MUMBAI WORKSPACE FROM FROZEN BASELINE
#
# Source:
#   /home/ubuntu/ghsl_multicity_baseline
#
# Destination:
#   /home/ubuntu/ghsl_multicity/mumbai_metro
#
# Safety:
#   - Refuses to overwrite an existing partial workspace
#   - Removes its own partial workspace if this build fails
#   - Does not change the frozen baseline
#   - Does not change Hyderabad, Pune, or Bengaluru
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1 and 2."
        )


if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


expected_workspace = (
    "/home/ubuntu/ghsl_multicity/mumbai_metro"
)

expected_hdfs_root = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/cities/"
    "city=mumbai_metro"
)

expected_extent = {
    "min_x": 7018000.0,
    "min_y": 2306000.0,
    "max_x": 7088000.0,
    "max_y": 2376000.0,
}


if (
    ACTIVE_CITY["remote_work_dir"]
    != expected_workspace
):
    raise RuntimeError(
        "Unexpected Mumbai workspace: "
        + ACTIVE_CITY["remote_work_dir"]
    )


if (
    ACTIVE_CITY["hdfs_output_base"]
    != expected_hdfs_root
):
    raise RuntimeError(
        "Unexpected Mumbai HDFS root: "
        + ACTIVE_CITY["hdfs_output_base"]
    )


if (
    ACTIVE_CITY["city_name"]
    != "Mumbai Metropolitan Region"
):
    raise RuntimeError(
        "Unexpected Mumbai city name."
    )


if (
    ACTIVE_CITY["extent_mollweide"]
    != expected_extent
):
    raise RuntimeError(
        "Unexpected Mumbai extent.\n"
        f"Expected: {expected_extent}\n"
        f"Actual:   "
        f"{ACTIVE_CITY['extent_mollweide']}"
    )


if (
    ACTIVE_CITY["expected_grid_cells"]
    != 4900
):
    raise RuntimeError(
        "Mumbai expected_grid_cells must be 4900."
    )


# ============================================================
# SERIALIZE MUMBAI CONFIGURATION
# ============================================================

config_json = (
    json.dumps(
        ACTIVE_CITY,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ============================================================
# BUILD REMOTE WORKSPACE CREATOR
# ============================================================

remote_builder = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    BASELINE = Path(
        "/home/ubuntu/ghsl_multicity_baseline"
    )

    TARGET = Path(
        "/home/ubuntu/ghsl_multicity/mumbai_metro"
    )

    CONFIG_PATH = Path(
        sys.argv[1]
    )

    TEMPLATE_NAMES = [
        "population_epoch_template.py",
        "smod_epoch_template.py",
        "built_surface_epoch_template.py",
    ]


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def compile_source(source, path):
        compile(
            source,
            str(path),
            "exec",
        )


    # ========================================================
    # LOAD CONFIGURATIONS
    # ========================================================

    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            CONFIG_PATH
        )

    config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    source_config_path = (
        BASELINE
        / "source_city_config.json"
    )

    if not source_config_path.is_file():
        raise FileNotFoundError(
            source_config_path
        )

    source_config = json.loads(
        source_config_path.read_text(
            encoding="utf-8"
        )
    )


    # ========================================================
    # VALIDATE MUMBAI CONFIGURATION
    # ========================================================

    expected_extent = {
        "min_x": 7018000.0,
        "min_y": 2306000.0,
        "max_x": 7088000.0,
        "max_y": 2376000.0,
    }

    if (
        config.get("city_slug")
        != "mumbai_metro"
    ):
        raise RuntimeError(
            "Mumbai configuration has the wrong slug."
        )

    if (
        config.get("city_name")
        != "Mumbai Metropolitan Region"
    ):
        raise RuntimeError(
            "Mumbai configuration has the wrong name."
        )

    if (
        config.get("remote_work_dir")
        != str(TARGET)
    ):
        raise RuntimeError(
            "Mumbai configuration has the wrong "
            "remote workspace."
        )

    if (
        config.get("hdfs_output_base")
        != (
            "hdfs://localhost:9000"
            "/data/ghsl/curated/cities/"
            "city=mumbai_metro"
        )
    ):
        raise RuntimeError(
            "Mumbai configuration has the wrong "
            "HDFS output root."
        )

    if (
        config.get("expected_grid_cells")
        != 4900
    ):
        raise RuntimeError(
            "Mumbai expected_grid_cells must be 4900."
        )

    if (
        config.get("extent_mollweide")
        != expected_extent
    ):
        raise RuntimeError(
            "Mumbai configuration has the wrong extent."
        )


    # ========================================================
    # VALIDATE BASELINE INPUTS
    # ========================================================

    baseline_driver = (
        BASELINE
        / "batch"
        / "run_city_epochs.py"
    )

    baseline_manifest = (
        BASELINE
        / "baseline_manifest.json"
    )

    baseline_checksums = (
        BASELINE
        / "SHA256SUMS"
    )

    required_baseline_files = [
        baseline_driver,
        baseline_manifest,
        baseline_checksums,
        source_config_path,
    ]

    for required_path in (
        required_baseline_files
    ):
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )

    baseline_templates = {}

    for template_name in TEMPLATE_NAMES:
        template_path = (
            BASELINE
            / "templates"
            / template_name
        )

        if not template_path.is_file():
            raise FileNotFoundError(
                template_path
            )

        baseline_templates[
            template_name
        ] = template_path


    # ========================================================
    # CHECK FOR AN EXISTING WORKSPACE
    # ========================================================

    workspace_manifest = (
        TARGET
        / "mumbai_workspace_manifest.json"
    )

    if TARGET.exists():
        if workspace_manifest.is_file():
            print("=" * 80)
            print("MUMBAI WORKSPACE CREATION")
            print("=" * 80)

            print(
                "Existing completed Mumbai "
                "workspace found."
            )

            print(
                "Workspace:",
                TARGET,
            )

            print(
                "Existing workspace was not "
                "overwritten."
            )

            print(
                "MUMBAI_WORKSPACE_ALREADY_EXISTS"
            )

            print(
                "MUMBAI_WORKSPACE_BUILD_SUCCESS"
            )

            raise SystemExit(0)

        raise RuntimeError(
            "An incomplete Mumbai workspace already "
            "exists and was not overwritten: "
            + str(TARGET)
        )


    # ========================================================
    # BUILD PARAMETER REPLACEMENTS
    # ========================================================

    source_root = str(
        source_config[
            "remote_work_dir"
        ]
    )

    target_root = str(
        config[
            "remote_work_dir"
        ]
    )

    source_slug = str(
        source_config[
            "city_slug"
        ]
    )

    target_slug = str(
        config[
            "city_slug"
        ]
    )

    source_name = str(
        source_config[
            "city_name"
        ]
    )

    target_name = str(
        config[
            "city_name"
        ]
    )

    source_extent = (
        source_config[
            "extent_mollweide"
        ]
    )

    target_extent = (
        config[
            "extent_mollweide"
        ]
    )

    replacements = [
        (
            source_root,
            target_root,
        ),
        (
            "city=" + source_slug,
            "city=" + target_slug,
        ),
        (
            source_slug,
            target_slug,
        ),
        (
            source_name,
            target_name,
        ),
    ]

    for key in [
        "min_x",
        "min_y",
        "max_x",
        "max_y",
    ]:
        replacements.append(
            (
                str(
                    float(
                        source_extent[key]
                    )
                ),
                str(
                    float(
                        target_extent[key]
                    )
                ),
            )
        )

        replacements.append(
            (
                str(
                    int(
                        source_extent[key]
                    )
                ),
                str(
                    int(
                        target_extent[key]
                    )
                ),
            )
        )


    def parameterize(source):
        output = source

        for old_value, new_value in replacements:
            output = output.replace(
                old_value,
                new_value,
            )

        return output


    # ========================================================
    # CREATE WORKSPACE
    # ========================================================

    try:
        (
            TARGET
            / "templates"
        ).mkdir(
            parents=True,
            exist_ok=False,
        )

        (
            TARGET
            / "programs"
        ).mkdir()

        (
            TARGET
            / "batch"
        ).mkdir()


        # ====================================================
        # PARAMETERIZE TEMPLATES
        # ====================================================

        template_hashes = {}

        for template_name in TEMPLATE_NAMES:
            source_path = (
                baseline_templates[
                    template_name
                ]
            )

            destination_path = (
                TARGET
                / "templates"
                / template_name
            )

            template_source = (
                source_path.read_text(
                    encoding="utf-8"
                )
            )

            template_source = parameterize(
                template_source
            )

            compile_source(
                template_source,
                destination_path,
            )

            destination_path.write_text(
                template_source,
                encoding="utf-8",
            )

            py_compile.compile(
                str(destination_path),
                doraise=True,
            )

            template_hashes[
                template_name
            ] = sha256_file(
                destination_path
            )


        # ====================================================
        # PARAMETERIZE DRIVER
        # ====================================================

        driver_destination = (
            TARGET
            / "batch"
            / "run_city_epochs.py"
        )

        driver_source = (
            baseline_driver.read_text(
                encoding="utf-8"
            )
        )

        driver_source = parameterize(
            driver_source
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS THROUGH AST
        # ====================================================

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found."
            )

        new_checksum_dictionary = {
            "population": (
                template_hashes[
                    "population_epoch_template.py"
                ]
            ),
            "smod": (
                template_hashes[
                    "smod_epoch_template.py"
                ]
            ),
            "built_surface": (
                template_hashes[
                    "built_surface_epoch_template.py"
                ]
            ),
        }

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                new_checksum_dictionary
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile_source(
            driver_source,
            driver_destination,
        )

        driver_destination.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(driver_destination),
            doraise=True,
        )


        # ====================================================
        # SAVE CONFIGURATION
        # ====================================================

        mumbai_config_path = (
            TARGET
            / "city_config.json"
        )

        mumbai_config_path.write_text(
            json.dumps(
                config,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # CODE-LEVEL PARAMETER AUDIT
        # ====================================================

        combined_source = (
            driver_source
        )

        for template_name in TEMPLATE_NAMES:
            combined_source += (
                TARGET
                / "templates"
                / template_name
            ).read_text(
                encoding="utf-8"
            )

        forbidden_values = [
            source_root,
            "city=" + source_slug,
            source_name,
        ]

        remaining_forbidden = [
            value
            for value in forbidden_values
            if value in combined_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Hyderabad leakage remains: "
                + repr(
                    remaining_forbidden
                )
            )

        required_code_values = [
            target_root,
            "city=" + target_slug,
            "7018000",
            "2306000",
            "7088000",
            "2376000",
        ]

        missing_code_values = [
            value
            for value
            in required_code_values
            if value
            not in combined_source
        ]

        if missing_code_values:
            raise RuntimeError(
                "Required Mumbai code parameters "
                "are missing: "
                + repr(
                    missing_code_values
                )
            )


        # ====================================================
        # CONFIGURATION-LEVEL AUDIT
        # ====================================================

        saved_config = json.loads(
            mumbai_config_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_config["city_name"]
            != "Mumbai Metropolitan Region"
        ):
            raise RuntimeError(
                "Mumbai city name is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["city_slug"]
            != "mumbai_metro"
        ):
            raise RuntimeError(
                "Mumbai city slug is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["extent_mollweide"]
            != expected_extent
        ):
            raise RuntimeError(
                "Mumbai extent is incorrect "
                "in city_config.json."
            )


        # ====================================================
        # VERIFY CHECKSUM DICTIONARY FROM SAVED DRIVER
        # ====================================================

        saved_driver_tree = ast.parse(
            driver_destination.read_text(
                encoding="utf-8"
            )
        )

        saved_checksum_dictionary = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                saved_checksum_dictionary = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if (
            saved_checksum_dictionary
            != new_checksum_dictionary
        ):
            raise RuntimeError(
                "Saved driver checksum dictionary "
                "does not match Mumbai templates."
            )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in TARGET.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # SAVE WORKSPACE MANIFEST
        # ====================================================

        manifest = {
            "city_slug": target_slug,
            "city_name": target_name,
            "source_baseline": str(
                BASELINE
            ),
            "workspace": str(
                TARGET
            ),
            "driver_sha256": sha256_file(
                driver_destination
            ),
            "template_sha256": (
                new_checksum_dictionary
            ),
            "extent_mollweide": (
                expected_extent
            ),
            "expected_grid_cells": 4900,
            "created_at_utc": (
                time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ",
                    time.gmtime(),
                )
            ),
        }

        workspace_manifest.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


    except Exception:
        # Do not leave behind a partial Mumbai workspace.
        if TARGET.exists():
            shutil.rmtree(
                TARGET
            )

        raise


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print("=" * 80)
    print("MUMBAI WORKSPACE CREATION")
    print("=" * 80)

    print(
        "Workspace:",
        TARGET,
    )

    print(
        "Driver:",
        TARGET
        / "batch"
        / "run_city_epochs.py",
    )

    print(
        "Configuration:",
        TARGET
        / "city_config.json",
    )

    print(
        "Manifest:",
        workspace_manifest,
    )

    print()
    print("Template parameterization: PASS")
    print("Template compilation: PASS")
    print("Driver parameterization: PASS")
    print("Driver compilation: PASS")
    print("Template checksum refresh: PASS")
    print("Saved checksum verification: PASS")
    print("Hyderabad leakage audit: PASS")
    print("Mumbai code-parameter audit: PASS")
    print("Mumbai configuration audit: PASS")
    print("Mumbai extent validation: PASS")
    print("Partial-build cleanup protection: PASS")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("MUMBAI_WORKSPACE_BUILD_SUCCESS")
    '''
).strip() + "\n"


# ============================================================
# LOCAL BUILDER VALIDATION
# ============================================================

compile(
    remote_builder,
    "build_mumbai_workspace.py",
    "exec",
)

builder_sha256 = hashlib.sha256(
    remote_builder.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI BUILDER COMPILE: PASS"
)

print(
    "Expected builder SHA-256:",
    builder_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

builder_b64 = base64.b64encode(
    remote_builder.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

BUILDER="/tmp/build_mumbai_workspace.py"
CONFIG="/tmp/mumbai_city_config.json"

EXPECTED_BUILDER_SHA256="{builder_sha256}"

echo "{builder_b64}" \
| base64 -d \
> "${{BUILDER}}"

echo "{config_b64}" \
| base64 -d \
> "${{CONFIG}}"

ACTUAL_BUILDER_SHA256="$(
    sha256sum "${{BUILDER}}" \
    | awk '{{print $1}}'
)"

echo "Remote builder SHA-256: ${{ACTUAL_BUILDER_SHA256}}"

if [ "${{ACTUAL_BUILDER_SHA256}}" != "${{EXPECTED_BUILDER_SHA256}}" ]; then
    echo "ERROR: Builder checksum mismatch."
    exit 20
fi

echo "REMOTE BUILDER CHECKSUM: PASS"

python3 -m py_compile "${{BUILDER}}"

echo "REMOTE BUILDER COMPILE: PASS"

python3 "${{BUILDER}}" "${{CONFIG}}"
"""


build_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 4 RESULT")
print("=" * 80)

print(
    "Return code:",
    build_result.returncode,
)

print(
    "Workspace:",
    ACTIVE_CITY["remote_work_dir"],
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 4: SUCCESS")

LOCAL MUMBAI BUILDER COMPILE: PASS
Expected builder SHA-256: 78ec31b92218789af4b041c93f0f3c890f1cd1426ff3d0a32f0d002bba810ffc
Remote builder SHA-256: 78ec31b92218789af4b041c93f0f3c890f1cd1426ff3d0a32f0d002bba810ffc
REMOTE BUILDER CHECKSUM: PASS
REMOTE BUILDER COMPILE: PASS
MUMBAI WORKSPACE CREATION
Workspace: /home/ubuntu/ghsl_multicity/mumbai_metro
Driver: /home/ubuntu/ghsl_multicity/mumbai_metro/batch/run_city_epochs.py
Configuration: /home/ubuntu/ghsl_multicity/mumbai_metro/city_config.json
Manifest: /home/ubuntu/ghsl_multicity/mumbai_metro/mumbai_workspace_manifest.json

Template parameterization: PASS
Template compilation: PASS
Driver parameterization: PASS
Driver compilation: PASS
Template checksum refresh: PASS
Saved checksum verification: PASS
Hyderabad leakage audit: PASS
Mumbai code-parameter audit: PASS
Mumbai configuration audit: PASS
Mumbai extent validation: PASS
Partial-build cleanup protection: PASS
Spark launched: NO
HDFS modified: NO
MUMBAI_WORKSPACE_BUILD_SUCCESS

RE

In [26]:
# ============================================================
# CELL 5: MUMBAI STATIC AND LEAKAGE AUDIT
#
# This version deliberately avoids:
#   - outer f-strings around embedded Python
#   - fragile grep checks for Python spacing
#   - creating __pycache__ inside the workspace
#
# It verifies:
#   - required Mumbai files
#   - Python compilation
#   - Mumbai configuration
#   - driver functions
#   - 4,900-cell expectation
#   - template checksums
#   - workspace-manifest checksums
#   - Mumbai paths and extent
#   - absence of Hyderabad leakage
#   - generalized endpoint logic
#   - generalized 2020 guards
#   - absence of stale launch state
#
# Read-only toward source files and HDFS.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_audit = r'''
set -e

ROOT="__MUMBAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
CONFIG="$ROOT/city_config.json"
MANIFEST="$ROOT/mumbai_workspace_manifest.json"

POPULATION_TEMPLATE="$ROOT/templates/population_epoch_template.py"
SMOD_TEMPLATE="$ROOT/templates/smod_epoch_template.py"
BUILT_TEMPLATE="$ROOT/templates/built_surface_epoch_template.py"

COMPILE_ROOT="/tmp/mumbai_static_compile_check"


echo "============================================================"
echo "MUMBAI STATIC AND LEAKAGE AUDIT"
echo "============================================================"


# ============================================================
# REQUIRED FILES
# ============================================================

for required_file in \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE"

do
    if [ ! -f "$required_file" ]; then
        echo "ERROR: Required file is missing:"
        echo "$required_file"
        exit 20
    fi
done

echo "Required Mumbai files: PASS"


# ============================================================
# COMPILE OUTSIDE THE WORKSPACE
# ============================================================

rm -rf "$COMPILE_ROOT"

mkdir -p "$COMPILE_ROOT"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$DRIVER"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$POPULATION_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$SMOD_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$BUILT_TEMPLATE"

echo "Mumbai Python compilation: PASS"


# ============================================================
# STRUCTURAL PYTHON AUDIT
#
# All paths are command-line arguments. No shell or notebook
# interpolation occurs inside this Python source.
# ============================================================

python3 - \
    "$ROOT" \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE" <<'PY'
from pathlib import Path
import ast
import hashlib
import json
import sys


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])
config_path = Path(sys.argv[3])
manifest_path = Path(sys.argv[4])

population_template = Path(sys.argv[5])
smod_template = Path(sys.argv[6])
built_template = Path(sys.argv[7])


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# ============================================================
# CONFIGURATION AUDIT
# ============================================================

config = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

expected_config = {
    "city_slug": "mumbai_metro",
    "city_name": "Mumbai Metropolitan Region",
    "remote_work_dir":
        "/home/ubuntu/ghsl_multicity/mumbai_metro",
    "hdfs_output_base":
        "hdfs://localhost:9000"
        "/data/ghsl/curated/cities/"
        "city=mumbai_metro",
    "expected_grid_cells": 4900,
    "expected_complete_cells": 4900,
    "expected_partial_cells": 0,
    "target_crs": "ESRI:54009",
}

for key, expected_value in expected_config.items():
    actual_value = config.get(key)

    if actual_value != expected_value:
        raise RuntimeError(
            "Configuration mismatch for "
            + key
            + ": expected "
            + repr(expected_value)
            + ", found "
            + repr(actual_value)
        )


expected_extent = {
    "min_x": 7018000.0,
    "min_y": 2306000.0,
    "max_x": 7088000.0,
    "max_y": 2376000.0,
}

actual_extent = config.get(
    "extent_mollweide"
)

if actual_extent != expected_extent:
    raise RuntimeError(
        "Mumbai extent mismatch. Expected "
        + repr(expected_extent)
        + ", found "
        + repr(actual_extent)
    )

if (
    expected_extent["max_x"]
    - expected_extent["min_x"]
    != 70000.0
):
    raise RuntimeError(
        "Mumbai X extent is not 70 km."
    )

if (
    expected_extent["max_y"]
    - expected_extent["min_y"]
    != 70000.0
):
    raise RuntimeError(
        "Mumbai Y extent is not 70 km."
    )

print("Mumbai configuration: PASS")


# ============================================================
# DRIVER STRUCTURE
# ============================================================

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)

function_names = set()

for node in driver_tree.body:
    if isinstance(
        node,
        ast.FunctionDef,
    ):
        function_names.add(
            node.name
        )


required_functions = {
    "generate_population_program",
    "generate_smod_program",
    "generate_built_program",
    "ensure_stage",
    "verify_template",
    "replace_exact_count",
}

missing_functions = sorted(
    required_functions
    - function_names
)

if missing_functions:
    raise RuntimeError(
        "Required driver functions are missing: "
        + repr(missing_functions)
    )

print("Required production functions: PASS")


# ============================================================
# TOP-LEVEL ASSIGNMENTS
# ============================================================

assignments = {}

for node in driver_tree.body:
    if not isinstance(
        node,
        ast.Assign,
    ):
        continue

    try:
        assignment_value = ast.literal_eval(
            node.value
        )
    except Exception:
        continue

    for target in node.targets:
        if isinstance(
            target,
            ast.Name,
        ):
            assignments[
                target.id
            ] = assignment_value


# Find a structural expected row/cell constant equal to 4900.

grid_expectations = {}

for name, value in assignments.items():
    uppercase_name = name.upper()

    if "EXPECTED" not in uppercase_name:
        continue

    if (
        "ROW" not in uppercase_name
        and "CELL" not in uppercase_name
    ):
        continue

    grid_expectations[name] = value


if 4900 not in grid_expectations.values():
    raise RuntimeError(
        "No expected row/cell constant equal "
        "to 4900 was found. Candidates: "
        + repr(grid_expectations)
    )

print(
    "Expected 4,900-cell constant: PASS",
    grid_expectations,
)


# ============================================================
# TEMPLATE CHECKSUM AUDIT
# ============================================================

driver_checksums = assignments.get(
    "EXPECTED_TEMPLATE_SHA256"
)

if not isinstance(
    driver_checksums,
    dict,
):
    raise RuntimeError(
        "EXPECTED_TEMPLATE_SHA256 is missing "
        "or is not a dictionary."
    )


actual_checksums = {
    "population": sha256_file(
        population_template
    ),
    "smod": sha256_file(
        smod_template
    ),
    "built_surface": sha256_file(
        built_template
    ),
}


if driver_checksums != actual_checksums:
    raise RuntimeError(
        "Driver template checksums do not match. "
        "Driver="
        + repr(driver_checksums)
        + ", actual="
        + repr(actual_checksums)
    )

print("Driver template checksums: PASS")


# ============================================================
# WORKSPACE MANIFEST AUDIT
# ============================================================

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

if (
    manifest.get("city_slug")
    != "mumbai_metro"
):
    raise RuntimeError(
        "Workspace manifest has the wrong slug."
    )

if (
    manifest.get("city_name")
    != "Mumbai Metropolitan Region"
):
    raise RuntimeError(
        "Workspace manifest has the wrong name."
    )

if (
    manifest.get("driver_sha256")
    != sha256_file(driver_path)
):
    raise RuntimeError(
        "Workspace driver checksum mismatch."
    )

if (
    manifest.get("template_sha256")
    != actual_checksums
):
    raise RuntimeError(
        "Workspace template checksum mismatch."
    )

print("Workspace manifest checksums: PASS")


# ============================================================
# CITY PATH AND LEAKAGE AUDIT
# ============================================================

combined_source = driver_source

combined_source += (
    population_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    smod_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    built_template.read_text(
        encoding="utf-8"
    )
)


required_mumbai_values = [
    "/home/ubuntu/ghsl_multicity/mumbai_metro",
    "city=mumbai_metro",
    "7018000",
    "2306000",
    "7088000",
    "2376000",
]

missing_mumbai_values = []

for value in required_mumbai_values:
    if value not in combined_source:
        missing_mumbai_values.append(
            value
        )

if missing_mumbai_values:
    raise RuntimeError(
        "Required Mumbai values are missing: "
        + repr(missing_mumbai_values)
    )


forbidden_hyderabad_values = [
    "/home/ubuntu/ghsl_multicity/hyderabad",
    "city=hyderabad",
    "Hyderabad Metropolitan Region",
]

remaining_hyderabad_values = []

for value in forbidden_hyderabad_values:
    if value in combined_source:
        remaining_hyderabad_values.append(
            value
        )

if remaining_hyderabad_values:
    raise RuntimeError(
        "Hyderabad leakage remains: "
        + repr(
            remaining_hyderabad_values
        )
    )

print("Mumbai paths and extent in code: PASS")
print("Hyderabad leakage audit: PASS")


# ============================================================
# GENERALIZED TEMPLATE LOGIC
# ============================================================

population_source = (
    population_template.read_text(
        encoding="utf-8"
    )
)

smod_source = (
    smod_template.read_text(
        encoding="utf-8"
    )
)

built_source = (
    built_template.read_text(
        encoding="utf-8"
    )
)


if (
    ".parquet(population_1975_path)"
    in population_source
):
    raise RuntimeError(
        "Population template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    ".parquet(SMOD_1975_PATH)"
    in smod_source
):
    raise RuntimeError(
        "SMOD template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    "product=population/"
    not in built_source
):
    raise RuntimeError(
        "Built template does not contain "
        "the population-grid reference."
    )

print("Generalized template logic: PASS")


# ============================================================
# GENERALIZED 2020 GUARDS
# ============================================================

def is_epoch_not_2020(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Name,
        )
        and node.left.id == "epoch"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.NotEq,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Constant,
        )
        and node.comparators[0].value
        == 2020
    )


def is_active_2020_check(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Constant,
        )
        and node.left.value
        == "EPOCH = 2020"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.In,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Name,
        )
        and node.comparators[0].id
        == "program"
    )


guards_checked = 0

for node in ast.walk(
    driver_tree
):
    if not isinstance(
        node,
        ast.If,
    ):
        continue

    messages = []

    for child in ast.walk(node):
        if (
            isinstance(
                child,
                ast.Constant,
            )
            and isinstance(
                child.value,
                str,
            )
        ):
            messages.append(
                child.value
            )

    if not any(
        "retains active 2020"
        in message
        for message in messages
    ):
        continue

    if not (
        isinstance(
            node.test,
            ast.BoolOp,
        )
        and isinstance(
            node.test.op,
            ast.And,
        )
    ):
        raise RuntimeError(
            "A retained-active-2020 guard "
            "is not generalized."
        )

    has_epoch_condition = any(
        is_epoch_not_2020(
            condition
        )
        for condition
        in node.test.values
    )

    has_program_condition = any(
        is_active_2020_check(
            condition
        )
        for condition
        in node.test.values
    )

    if not (
        has_epoch_condition
        and has_program_condition
    ):
        raise RuntimeError(
            "A 2020 guard is missing one "
            "of its required conditions."
        )

    guards_checked += 1


if guards_checked == 0:
    raise RuntimeError(
        "No generalized 2020 guards were found."
    )

print(
    "Generalized 2020 guards: PASS",
    guards_checked,
)


print()
print("MUMBAI_STRUCTURAL_AUDIT_SUCCESS")
PY


# ============================================================
# LAUNCH-STATE AUDIT
# ============================================================

if [ -e "$ROOT/batch/city.pid" ]; then
    echo "ERROR: Mumbai city.pid already exists."
    exit 30
fi

if [ -e "$ROOT/batch/city.log" ]; then
    echo "ERROR: Mumbai city.log already exists."
    exit 31
fi

echo "No stale Mumbai launch state: PASS"


# ============================================================
# CLEAN TEMPORARY COMPILATION OUTPUT
# ============================================================

rm -rf "$COMPILE_ROOT"


echo
echo "MUMBAI_STATIC_AUDIT_SUCCESS"
'''


# Replace a literal placeholder instead of using an f-string.

cmd = remote_audit.replace(
    "__MUMBAI_ROOT__",
    root,
)


audit_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5 RESULT")
print("=" * 80)

print(
    "Return code:",
    audit_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Mumbai source files modified: NO")
print("CELL 5: SUCCESS")

MUMBAI STATIC AND LEAKAGE AUDIT
Required Mumbai files: PASS
Mumbai Python compilation: PASS
Mumbai configuration: PASS
Required production functions: PASS
Expected 4,900-cell constant: PASS {'EXPECTED_ROWS': 4900}
Driver template checksums: PASS
Workspace manifest checksums: PASS
Mumbai paths and extent in code: PASS
Hyderabad leakage audit: PASS
Generalized template logic: PASS
Generalized 2020 guards: PASS 3

MUMBAI_STRUCTURAL_AUDIT_SUCCESS
No stale Mumbai launch state: PASS

MUMBAI_STATIC_AUDIT_SUCCESS

REMOTE STDERR:


CELL 5 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Mumbai source files modified: NO
CELL 5: SUCCESS


In [10]:
# ============================================================
# CELL 5A: MUMBAI SELECTED-TILE COUNT PROBE
#
# Purpose:
#   - Generates a temporary Mumbai population-1975 program
#   - Runs only through source discovery and tile selection
#   - Stops immediately after printing selected tile count
#
# Does not:
#   - write Mumbai output to HDFS
#   - modify Mumbai templates or driver
#   - modify the workspace manifest
#   - launch the full production batch
#
# Temporary probe files are removed after execution.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_probe = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

ROOT="__MUMBAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"

PROBE_ROOT="/tmp/mumbai_tile_count_probe"
PROBE_PROGRAM="$PROBE_ROOT/population_1975_probe.py"


echo "============================================================"
echo "MUMBAI SELECTED-TILE COUNT PROBE"
echo "============================================================"


if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Mumbai driver is missing:"
    echo "$DRIVER"
    exit 20
fi


rm -rf "$PROBE_ROOT"

mkdir -p "$PROBE_ROOT"


# ============================================================
# GENERATE A TEMPORARY 1975 POPULATION PROGRAM
# ============================================================

python3 - \
    "$DRIVER" \
    "$PROBE_ROOT" \
    "$PROBE_PROGRAM" <<'PY'
from pathlib import Path
import ast
import sys


driver_path = Path(sys.argv[1])
probe_root = Path(sys.argv[2])
probe_program = Path(sys.argv[3])


driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)


# ============================================================
# LOAD DRIVER DEFINITIONS WITHOUT RUNNING THE BATCH LOOP
# ============================================================

allowed_node_types = (
    ast.Import,
    ast.ImportFrom,
    ast.Assign,
    ast.AnnAssign,
    ast.FunctionDef,
    ast.AsyncFunctionDef,
    ast.ClassDef,
)

definition_nodes = [
    node
    for node in driver_tree.body
    if isinstance(
        node,
        allowed_node_types,
    )
]

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


namespace = {
    "__file__": str(driver_path),
    "__name__": "mumbai_tile_count_probe",
}

exec(
    compile(
        definition_module,
        str(driver_path),
        "exec",
    ),
    namespace,
)


factory = namespace.get(
    "generate_population_program"
)

if not callable(factory):
    raise RuntimeError(
        "generate_population_program is unavailable."
    )


generated_program = factory(
    1975,
    probe_root,
)

generated_program = Path(
    generated_program
)


if not generated_program.is_file():
    raise RuntimeError(
        "The population factory did not create "
        "the 1975 program."
    )


source_lines = generated_program.read_text(
    encoding="utf-8"
).splitlines(
    keepends=True
)


# ============================================================
# LOCATE THE SELECTED-TILE REPORTING BLOCK
# ============================================================

message_index = None

for index, line in enumerate(
    source_lines
):
    if (
        "Selected Bengaluru tiles:"
        in line
    ):
        message_index = index
        break


if message_index is None:
    raise RuntimeError(
        "Could not locate the selected-tile "
        "reporting block."
    )


# Find the opening print statement. The message line is more
# deeply indented than the enclosing print statement.

print_index = None

for index in range(
    message_index,
    max(
        -1,
        message_index - 10,
    ),
    -1,
):
    if (
        source_lines[index]
        .lstrip()
        .startswith("print(")
    ):
        print_index = index
        break


if print_index is None:
    raise RuntimeError(
        "Could not locate the opening print statement "
        "for the selected-tile report."
    )


# Find the closing parenthesis of that print statement.

closing_index = None

for index in range(
    message_index + 1,
    min(
        len(source_lines),
        message_index + 15,
    ),
):
    if (
        source_lines[index].strip()
        == ")"
    ):
        closing_index = index
        break


if closing_index is None:
    raise RuntimeError(
        "Could not locate the end of the "
        "selected-tile reporting block."
    )


# Use the indentation of the opening print statement, not the
# more deeply indented string argument.

indent = (
    source_lines[
        print_index
    ][
        :
        len(
            source_lines[
                print_index
            ]
        )
        - len(
            source_lines[
                print_index
            ].lstrip()
        )
    ]
)


# ============================================================
# INSERT A CLEAN EXIT AFTER TILE-COUNT PRINT
# ============================================================

exit_block = [
    "\n",
    indent
    + 'print("MUMBAI_TILE_COUNT_PROBE_SUCCESS")\n',
    indent
    + "raise SystemExit(0)\n",
    "\n",
]


source_lines[
    closing_index + 1:
    closing_index + 1
] = exit_block


probe_source = "".join(
    source_lines
)


# ============================================================
# ENSURE THE PROBE EXITS BEFORE THE HDFS WRITE
# ============================================================

exit_position = probe_source.find(
    "MUMBAI_TILE_COUNT_PROBE_SUCCESS"
)

write_position = probe_source.find(
    ".parquet(OUTPUT_PATH)"
)

if exit_position < 0:
    raise RuntimeError(
        "Probe exit marker is missing."
    )

if (
    write_position >= 0
    and exit_position >= write_position
):
    raise RuntimeError(
        "Probe exit occurs after the HDFS write."
    )


compile(
    probe_source,
    str(probe_program),
    "exec",
)


probe_program.write_text(
    probe_source,
    encoding="utf-8",
)


print(
    "Temporary probe program:",
    probe_program,
)

print(
    "Probe exits before HDFS write: PASS"
)

print(
    "Probe compilation: PASS"
)
PY


# ============================================================
# RUN THE TEMPORARY PROBE
# ============================================================

set +e

spark-submit \
    --master local[1] \
    --driver-memory 5g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
    org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
    https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "$PROBE_PROGRAM"

PROBE_RETURN_CODE="$?"

set -e


if [ "$PROBE_RETURN_CODE" -ne 0 ]; then
    echo "ERROR: Mumbai tile-count probe failed."
    echo "Probe return code: $PROBE_RETURN_CODE"

    rm -rf "$PROBE_ROOT"

    exit 30
fi


# ============================================================
# CLEAN TEMPORARY FILES
# ============================================================

rm -rf "$PROBE_ROOT"


echo
echo "Temporary probe files removed: PASS"
echo "Mumbai workspace modified: NO"
echo "HDFS modified: NO"
echo "Full production batch launched: NO"
echo "MUMBAI_TILE_COUNT_PROBE_COMPLETE"
'''


cmd = remote_probe.replace(
    "__MUMBAI_ROOT__",
    root,
)


probe_result = run_remote(
    cmd,
    timeout=900,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5A RESULT")
print("=" * 80)

print(
    "Return code:",
    probe_result.returncode,
)

print("Mumbai workspace modified: NO")
print("HDFS modified: NO")
print("Full production batch launched: NO")
print("CELL 5A: SUCCESS")

MUMBAI SELECTED-TILE COUNT PROBE
Temporary probe program: /tmp/mumbai_tile_count_probe/population_1975_probe.py
Probe exits before HDFS write: PASS
Probe compilation: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
BATCH: BENGALURU 1975 POPULATION
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Output: hdfs://localhost:9000/data/ghsl/curated/cities/city=mumbai_metro/product=population/epoch=1975/grid_1km
1975 Bengaluru source shards: 2
-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 filename     | ghsl_india_population_1975_100m_r2023a-0000008192-0000000000.tif                                                                 
 path         | hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a

RuntimeError: Remote command failed with return code 30.

In [11]:
# ============================================================
# CELL 5B: ALLOW MULTIPLE MUMBAI POPULATION SOURCE SHARDS
#
# Confirmed Mumbai condition:
#   - The 70 km extent intersects two population GeoTIFF shards
#   - The inherited template requires exactly one shard
#
# This cell changes only the population-template guard:
#
#   source_shard_count != 1
#               to
#   source_shard_count < 1
#
# It also:
#   - creates a rollback backup
#   - refreshes the population-template checksum in the driver
#   - refreshes the Mumbai workspace manifest
#   - resets stale generated programs
#   - performs Python compilation checks
#
# It does not:
#   - modify the frozen baseline
#   - modify HDFS
#   - launch Spark
#   - modify other city workspaces
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    WORKSPACE_MANIFEST = (
        ROOT
        / "mumbai_workspace_manifest.json"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    BACKUP_ROOT = (
        ROOT
        / "multi_shard_patch_backup"
    )

    PATCH_VERSION = (
        "mumbai_population_multi_shard_v1"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    required_files = [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
        WORKSPACE_MANIFEST,
    ]

    for required_file in required_files:
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    # ========================================================
    # REFUSE TO PATCH AFTER A PRODUCTION LAUNCH
    # ========================================================

    pid_file = (
        ROOT
        / "batch"
        / "city.pid"
    )

    log_file = (
        ROOT
        / "batch"
        / "city.log"
    )

    if pid_file.exists():
        raise RuntimeError(
            "Mumbai city.pid already exists. "
            "Do not patch after production launch."
        )

    if log_file.exists():
        raise RuntimeError(
            "Mumbai city.log already exists. "
            "Do not patch after production launch."
        )


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        WORKSPACE_MANIFEST,
        BACKUP_ROOT
        / WORKSPACE_MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / WORKSPACE_MANIFEST.name,
            WORKSPACE_MANIFEST,
        )


    try:
        # ====================================================
        # PATCH THE POPULATION SHARD-COUNT GUARD
        # ====================================================

        population_source = (
            POPULATION_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        if (
            "MUMBAI MULTI-SHARD SOURCE SUPPORT"
            in population_source
        ):
            raise RuntimeError(
                "Mumbai multi-shard patch is already applied."
            )


        population_tree = ast.parse(
            population_source
        )

        target_if = None

        for node in ast.walk(
            population_tree
        ):
            if not isinstance(
                node,
                ast.If,
            ):
                continue

            test = node.test

            if not isinstance(
                test,
                ast.Compare,
            ):
                continue

            if not isinstance(
                test.left,
                ast.Name,
            ):
                continue

            if (
                test.left.id
                != "source_shard_count"
            ):
                continue

            if len(test.ops) != 1:
                continue

            if not isinstance(
                test.ops[0],
                ast.NotEq,
            ):
                continue

            if len(test.comparators) != 1:
                continue

            comparator = (
                test.comparators[0]
            )

            if not (
                isinstance(
                    comparator,
                    ast.Constant,
                )
                and comparator.value == 1
            ):
                continue

            messages = [
                child.value
                for child in ast.walk(node)
                if (
                    isinstance(
                        child,
                        ast.Constant,
                    )
                    and isinstance(
                        child.value,
                        str,
                    )
                )
            ]

            if not any(
                "Expected exactly one"
                in message
                and "population shard"
                in message
                for message in messages
            ):
                continue

            if target_if is not None:
                raise RuntimeError(
                    "Multiple population shard-count "
                    "guards matched."
                )

            target_if = node


        if target_if is None:
            raise RuntimeError(
                "Could not locate the exact-one "
                "population shard-count guard."
            )


        source_lines = (
            population_source.splitlines(
                keepends=True
            )
        )

        start_index = (
            target_if.lineno - 1
        )

        first_line = (
            source_lines[start_index]
        )

        leading_whitespace = (
            first_line[
                :
                len(first_line)
                - len(
                    first_line.lstrip()
                )
            ]
        )

        old_if_text = "".join(
            source_lines[
                start_index:
                target_if.end_lineno
            ]
        )

        old_condition = (
            "source_shard_count != 1"
        )

        if old_condition not in old_if_text:
            compact_text = (
                old_if_text.replace(
                    "\n",
                    " "
                )
            )

            compact_text = " ".join(
                compact_text.split()
            )

            if (
                old_condition
                not in compact_text
            ):
                raise RuntimeError(
                    "The located shard guard did not "
                    "contain the expected condition."
                )


        new_if_lines = [
            (
                leading_whitespace
                + "# MUMBAI MULTI-SHARD SOURCE SUPPORT\n"
            ),
            (
                leading_whitespace
                + "# The Mumbai extent may legitimately "
                + "intersect more than one\n"
            ),
            (
                leading_whitespace
                + "# raster shard. Fail only when no "
                + "source shard intersects.\n"
            ),
            (
                leading_whitespace
                + "if source_shard_count < 1:\n"
            ),
            (
                leading_whitespace
                + "    raise RuntimeError(\n"
            ),
            (
                leading_whitespace
                + '        "No population source shard "\n'
            ),
            (
                leading_whitespace
                + '        "intersects the Mumbai extent."\n'
            ),
            (
                leading_whitespace
                + "    )\n"
            ),
        ]


        source_lines[
            start_index:
            target_if.end_lineno
        ] = new_if_lines

        population_source = "".join(
            source_lines
        )


        # Ensure exactly one active guard now exists.

        if (
            population_source.count(
                "MUMBAI MULTI-SHARD SOURCE SUPPORT"
            )
            != 1
        ):
            raise RuntimeError(
                "Unexpected multi-shard patch marker count."
            )

        if (
            "if source_shard_count < 1:"
            not in population_source
        ):
            raise RuntimeError(
                "The at-least-one-shard guard "
                "was not installed."
            )


        compile(
            population_source,
            str(POPULATION_TEMPLATE),
            "exec",
        )

        POPULATION_TEMPLATE.write_text(
            population_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(POPULATION_TEMPLATE),
            doraise=True,
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # UPDATE CHECKSUM DICTIONARY IN DRIVER
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                checksum_node = node
                break


        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found in the driver."
            )


        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                template_checksums
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH WORKSPACE MANIFEST
        # ====================================================

        manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        manifest[
            "template_sha256"
        ] = template_checksums

        manifest[
            "population_multi_shard_patch"
        ] = {
            "version": PATCH_VERSION,
            "minimum_source_shards": 1,
            "multiple_intersecting_shards_allowed": True,
            "confirmed_intersecting_shards": 2,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        WORKSPACE_MANIFEST.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL VERIFICATION
        # ====================================================

        for source_file in [
            DRIVER,
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )


        saved_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        saved_checksums = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                saved_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break


        if (
            saved_checksums
            != template_checksums
        ):
            raise RuntimeError(
                "Saved driver checksums do not "
                "match the templates."
            )


        saved_manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_manifest.get(
                "driver_sha256"
            )
            != sha256_file(DRIVER)
        ):
            raise RuntimeError(
                "Workspace driver checksum "
                "was not refreshed."
            )

        if (
            saved_manifest.get(
                "template_sha256"
            )
            != template_checksums
        ):
            raise RuntimeError(
                "Workspace template checksums "
                "were not refreshed."
            )


    except Exception:
        restore_backup()

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )

        raise


    print("=" * 80)
    print("MUMBAI POPULATION MULTI-SHARD PATCH")
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Confirmed intersecting shards: 2")
    print("Minimum required shards: 1")
    print("Multiple intersecting shards allowed: YES")
    print("Population template modified: YES")
    print("SMOD template modified: NO")
    print("Built-surface template modified: NO")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")
    print("Generated programs reset: PASS")
    print("Python compilation: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")
    print(
        "MUMBAI MULTI-SHARD PATCH: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# LOCAL PATCHER VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_mumbai_multi_shard.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI MULTI-SHARD PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_mumbai_multi_shard.py"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 5B: SUCCESS")

LOCAL MUMBAI MULTI-SHARD PATCHER COMPILE: PASS
Expected patcher SHA-256: 3bff9cec0334e928234830f332945cb4c02fcb1254e1159af10be8cae32af61d
Remote patcher SHA-256: 3bff9cec0334e928234830f332945cb4c02fcb1254e1159af10be8cae32af61d
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/patch_mumbai_multi_shard.py", line 322, in <module>
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: Could not locate the exact-one population shard-count guard.



RuntimeError: Remote command failed with return code 1.

In [12]:
# ============================================================
# CELL 5B-INSPECT: LOCATE EXACT MUMBAI SHARD CHECK
#
# Read-only:
#   - Does not modify templates
#   - Does not modify the driver
#   - Does not modify the workspace manifest
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


cmd = r'''
set -e

ROOT="__MUMBAI_ROOT__"

TEMPLATE="$ROOT/templates/population_epoch_template.py"

if [ ! -f "$TEMPLATE" ]; then
    echo "ERROR: Population template is missing:"
    echo "$TEMPLATE"
    exit 20
fi

echo "============================================================"
echo "MUMBAI POPULATION SHARD-CHECK INSPECTION"
echo "============================================================"

python3 - "$TEMPLATE" <<'PY'
from pathlib import Path
import ast
import sys


template_path = Path(
    sys.argv[1]
)

source = template_path.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


print(
    "Template:",
    template_path,
)

print()


patterns = [
    "Expected exactly one",
    "population shard",
    "source_shard",
    "shard_count",
    "source_count",
    "catalog_count",
]


matched_lines = set()

for line_number, line in enumerate(
    lines,
    start=1,
):
    if any(
        pattern in line
        for pattern in patterns
    ):
        matched_lines.add(
            line_number
        )


if not matched_lines:
    raise RuntimeError(
        "No population shard-check text was found."
    )


ranges = []

for line_number in sorted(
    matched_lines
):
    start = max(
        1,
        line_number - 25,
    )

    end = min(
        len(lines),
        line_number + 25,
    )

    if (
        ranges
        and start
        <= ranges[-1][1] + 1
    ):
        ranges[-1] = (
            ranges[-1][0],
            max(
                ranges[-1][1],
                end,
            ),
        )

    else:
        ranges.append(
            (
                start,
                end,
            )
        )


for start, end in ranges:
    print(
        "=" * 80
    )

    print(
        "SOURCE LINES "
        + str(start)
        + "-"
        + str(end)
    )

    print(
        "=" * 80
    )

    for line_number in range(
        start,
        end + 1,
    ):
        print(
            str(line_number).rjust(5)
            + ": "
            + lines[
                line_number - 1
            ]
        )

    print()


tree = ast.parse(
    source
)


print(
    "=" * 80
)

print(
    "IF CONDITIONS CONTAINING SHARD-RELATED TEXT"
)

print(
    "=" * 80
)


condition_count = 0

for node in ast.walk(
    tree
):
    if not isinstance(
        node,
        ast.If,
    ):
        continue

    node_text = ast.get_source_segment(
        source,
        node,
    )

    if node_text is None:
        continue

    lowered = node_text.lower()

    if (
        "shard"
        not in lowered
        and "source_count"
        not in lowered
        and "catalog_count"
        not in lowered
    ):
        continue

    condition_count += 1

    print(
        "IF line:",
        node.lineno,
    )

    print(
        "Condition:",
        ast.unparse(
            node.test
        ),
    )

    print(
        "Block:"
    )

    print(
        node_text
    )

    print(
        "-" * 80
    )


if condition_count == 0:
    raise RuntimeError(
        "No shard-related IF condition was found."
    )


print(
    "Shard-related IF conditions:",
    condition_count,
)

print(
    "MUMBAI_SHARD_CHECK_INSPECTION_SUCCESS"
)
PY

echo
echo "Mumbai workspace modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CELL_5B_INSPECT_REMOTE_SUCCESS"
'''


cmd = cmd.replace(
    "__MUMBAI_ROOT__",
    root,
)


inspection_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B-INSPECT RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Mumbai workspace modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 5B-INSPECT: SUCCESS")

MUMBAI POPULATION SHARD-CHECK INSPECTION
Template: /home/ubuntu/ghsl_multicity/mumbai_metro/templates/population_epoch_template.py

SOURCE LINES 72-232
   72:     print("Catalog:", HDFS_CATALOG)
   73:     print("Output:", OUTPUT_PATH)
   74: 
   75: 
   76:     # ====================================================
   77:     # DISCOVER THE 2020 POPULATION SHARD
   78:     #
   79:     # First filter the catalog to the 15 population files.
   80:     # Then use the catalog envelopes and Sedona to select
   81:     # the shard intersecting Bengaluru.
   82:     # ====================================================
   83: 
   84:     catalog_df = (
   85:         spark.read
   86:         .parquet(HDFS_CATALOG)
   87:         .filter(
   88:             F.lower(F.col("product"))
   89:             == F.lit(PRODUCT)
   90:         )
   91:         .filter(
   92:             F.col("temporal_value").cast("int")
   93:             == F.lit(EPOCH)
   94:         )
   95:     )
   96: 
   9

In [23]:
# ============================================================
# CELL 5B-BUILT: MUMBAI BUILT-SURFACE MULTI-SHARD PATCH
#
# Confirmed failure:
#   Expected exactly one Bengaluru source shard, found 2.
#
# Corrected behavior:
#   - Allows one or more intersecting built-surface shards
#   - Loads every selected source path
#   - Expects one decoded GeoTIFF per selected shard
#   - Expects source tiles per shard multiplied by shard count
#
# Safety:
#   - Patches only the Mumbai built-surface template
#   - Preserves the existing population multi-shard patch
#   - Preserves the coastal canonical-grid correction
#   - Refreshes driver template checksums
#   - Refreshes the Mumbai workspace manifest
#   - Archives the failed production log
#   - Removes only the stopped/stale PID file
#   - Removes generated programs for safe regeneration
#   - Rolls back source files if any patch check fails
#
# Does not:
#   - modify the frozen baseline
#   - delete valid Mumbai HDFS outputs
#   - modify HDFS
#   - launch Spark
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    MANIFEST = (
        ROOT
        / "mumbai_workspace_manifest.json"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    PID_FILE = (
        ROOT
        / "batch"
        / "city.pid"
    )

    LOG_FILE = (
        ROOT
        / "batch"
        / "city.log"
    )

    BACKUP_ROOT = (
        ROOT
        / "built_multi_shard_patch_backup"
    )

    PATCH_VERSION = (
        "mumbai_built_surface_multi_shard_v1"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def replace_once(
        source,
        old_text,
        new_text,
        description,
    ):
        count = source.count(
            old_text
        )

        if count != 1:
            raise RuntimeError(
                description
                + ": expected one occurrence, found "
                + str(count)
                + "\nExpected text:\n"
                + old_text
            )

        return source.replace(
            old_text,
            new_text,
            1,
        )


    required_files = [
        DRIVER,
        MANIFEST,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
    ]

    for required_file in (
        required_files
    ):
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    # ========================================================
    # REFUSE TO PATCH A RUNNING JOB
    # ========================================================

    recorded_pid = None

    if PID_FILE.is_file():
        recorded_pid = (
            PID_FILE.read_text(
                encoding="utf-8"
            ).strip()
        )

        if (
            recorded_pid
            and (
                Path("/proc")
                / recorded_pid
            ).exists()
        ):
            raise RuntimeError(
                "Mumbai production is still running "
                "with PID "
                + recorded_pid
            )


    # ========================================================
    # VERIFY EXISTING REQUIRED CORRECTIONS
    # ========================================================

    population_source = (
        POPULATION_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    required_population_fragments = [
        "MUMBAI MULTI-SHARD SOURCE SUPPORT",
        "selected_source_paths",
        ".load(selected_source_paths)",
        "decoded_source_count",
        "expected_all_tile_count",
        "COASTAL CANONICAL-GRID COMPLETION",
        "has_valid_population_coverage",
    ]

    missing_population_fragments = [
        fragment
        for fragment
        in required_population_fragments
        if fragment
        not in population_source
    ]

    if missing_population_fragments:
        raise RuntimeError(
            "Required Mumbai population corrections "
            "are missing: "
            + repr(
                missing_population_fragments
            )
        )


    smod_source = (
        SMOD_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    if (
        "COASTAL CANONICAL-GRID COMPLETION"
        not in smod_source
        or "has_valid_smod_coverage"
        not in smod_source
    ):
        raise RuntimeError(
            "Mumbai SMOD coastal correction "
            "is missing."
        )


    original_built_source = (
        BUILT_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    if (
        "COASTAL CANONICAL-GRID COMPLETION"
        not in original_built_source
        or "has_valid_built_coverage"
        not in original_built_source
    ):
        raise RuntimeError(
            "Mumbai built-surface coastal correction "
            "is missing."
        )


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        MANIFEST,
        BACKUP_ROOT
        / MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / MANIFEST.name,
            MANIFEST,
        )


    try:
        built_source = (
            original_built_source
        )


        # ====================================================
        # PATCH 1: ALLOW ONE OR MORE SOURCE SHARDS
        # ====================================================

        old_source_count_guard = (
            "    if selected_source_count != 1:\n"
            "        raise RuntimeError(\n"
            '            "Expected exactly one Bengaluru source shard, found "\n'
            '            f"{selected_source_count}."\n'
            "        )\n"
        )

        new_source_count_guard = (
            "    # ====================================================\n"
            "    # MUMBAI BUILT-SURFACE MULTI-SHARD SUPPORT\n"
            "    # ====================================================\n"
            "\n"
            "    if selected_source_count < 1:\n"
            "        raise RuntimeError(\n"
            '            "No built-surface source shard intersects "\n'
            '            "the Mumbai extent."\n'
            "        )\n"
        )

        built_source = replace_once(
            built_source,
            old_source_count_guard,
            new_source_count_guard,
            "Built source-count guard",
        )


        # ====================================================
        # PATCH 2: COLLECT ALL SELECTED SOURCE PATHS
        #
        # Keep selected_source and source_path because the
        # inherited template uses the first source record for
        # its existing metadata checks and display text.
        # Add the complete path list for actual raster loading.
        # ====================================================

        old_source_selection = (
            "    selected_source = (\n"
            "        selected_sources_df.first()\n"
            "    )\n"
            "\n"
            "    source_path = selected_source[\"path\"]\n"
        )

        new_source_selection = (
            "    selected_source_records = (\n"
            "        selected_sources_df.collect()\n"
            "    )\n"
            "\n"
            "    selected_source_paths = [\n"
            "        record[\"path\"]\n"
            "        for record\n"
            "        in selected_source_records\n"
            "    ]\n"
            "\n"
            "    if (\n"
            "        len(selected_source_paths)\n"
            "        != selected_source_count\n"
            "    ):\n"
            "        raise RuntimeError(\n"
            '            "Collected built-source path count "\n'
            '            "does not match the selected shard count."\n'
            "        )\n"
            "\n"
            "    if (\n"
            "        len(set(selected_source_paths))\n"
            "        != selected_source_count\n"
            "    ):\n"
            "        raise RuntimeError(\n"
            '            "Duplicate built-surface source paths "\n'
            '            "were selected."\n'
            "        )\n"
            "\n"
            "    selected_source = (\n"
            "        selected_source_records[0]\n"
            "    )\n"
            "\n"
            "    source_path = selected_source[\"path\"]\n"
        )

        built_source = replace_once(
            built_source,
            old_source_selection,
            new_source_selection,
            "Built source-path selection",
        )


        # ====================================================
        # PATCH 3: LOAD EVERY INTERSECTING SOURCE PATH
        # ====================================================

        built_source = replace_once(
            built_source,
            "        .load(source_path)\n",
            "        .load(selected_source_paths)\n",
            "Built binaryFile source loading",
        )


        # ====================================================
        # PATCH 4: EXPECT ONE DECODED GEOTIFF PER SHARD
        # ====================================================

        old_source_df_check = (
            "    if source_df.count() != 1:\n"
            "        raise RuntimeError(\n"
            '            "Expected exactly one source GeoTIFF."\n'
            "        )\n"
        )

        new_source_df_check = (
            "    decoded_source_count = (\n"
            "        source_df.count()\n"
            "    )\n"
            "\n"
            "    if (\n"
            "        decoded_source_count\n"
            "        != selected_source_count\n"
            "    ):\n"
            "        raise RuntimeError(\n"
            '            "Decoded built-surface GeoTIFF count "\n'
            '            "does not match the selected shard count. "\n'
            '            f"Selected={selected_source_count}, "\n'
            '            f"decoded={decoded_source_count}."\n'
            "        )\n"
        )

        built_source = replace_once(
            built_source,
            old_source_df_check,
            new_source_df_check,
            "Built decoded-source count",
        )


        # ====================================================
        # PATCH 5: EXPECT SOURCE TILES PER SELECTED SHARD
        # ====================================================

        old_tile_count_check = (
            "    if all_tile_count != EXPECTED_SOURCE_TILES:\n"
            "        raise RuntimeError(\n"
            '            "Expected 1024 source tiles, found "\n'
            '            f"{all_tile_count}."\n'
            "        )\n"
        )

        new_tile_count_check = (
            "    expected_all_tile_count = (\n"
            "        EXPECTED_SOURCE_TILES\n"
            "        * selected_source_count\n"
            "    )\n"
            "\n"
            "    if (\n"
            "        all_tile_count\n"
            "        != expected_all_tile_count\n"
            "    ):\n"
            "        raise RuntimeError(\n"
            '            "Unexpected built-surface source-tile count. "\n'
            '            f"Expected {expected_all_tile_count} "\n'
            '            f"from {selected_source_count} source "\n'
            '            f"shard(s), found {all_tile_count}."\n'
            "        )\n"
            "\n"
            "    print(\n"
            '        "Built-surface source shards tiled:",\n'
            "        selected_source_count,\n"
            "    )\n"
            "\n"
            "    print(\n"
            '        "Total built-surface source tiles:",\n'
            "        all_tile_count,\n"
            "    )\n"
        )

        built_source = replace_once(
            built_source,
            old_tile_count_check,
            new_tile_count_check,
            "Built source-tile count",
        )


        # ====================================================
        # VERIFY REQUIRED BUILT MULTI-SHARD LOGIC
        # ====================================================

        required_built_fragments = [
            "MUMBAI BUILT-SURFACE MULTI-SHARD SUPPORT",
            "selected_source_records",
            "selected_source_paths",
            ".load(selected_source_paths)",
            "decoded_source_count",
            "expected_all_tile_count",
            (
                "EXPECTED_SOURCE_TILES\n"
                "        * selected_source_count"
            ),
            "COASTAL CANONICAL-GRID COMPLETION",
            "has_valid_built_coverage",
        ]

        missing_built_fragments = [
            fragment
            for fragment
            in required_built_fragments
            if fragment
            not in built_source
        ]

        if missing_built_fragments:
            raise RuntimeError(
                "Required built multi-shard logic "
                "is missing: "
                + repr(
                    missing_built_fragments
                )
            )


        forbidden_built_fragments = [
            "if selected_source_count != 1:",
            ".load(source_path)",
            "Expected exactly one source GeoTIFF.",
            (
                "if all_tile_count "
                "!= EXPECTED_SOURCE_TILES:"
            ),
        ]

        remaining_forbidden = [
            fragment
            for fragment
            in forbidden_built_fragments
            if fragment
            in built_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Old single-shard built logic remains: "
                + repr(
                    remaining_forbidden
                )
            )


        compile(
            built_source,
            str(BUILT_TEMPLATE),
            "exec",
        )

        BUILT_TEMPLATE.write_text(
            built_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(BUILT_TEMPLATE),
            doraise=True,
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # REFRESH EXPECTED_TEMPLATE_SHA256 IN DRIVER
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found in the driver."
            )

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                template_checksums
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH MUMBAI WORKSPACE MANIFEST
        # ====================================================

        workspace_manifest = json.loads(
            MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        workspace_manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        workspace_manifest[
            "template_sha256"
        ] = template_checksums

        workspace_manifest[
            "built_surface_multi_shard_patch"
        ] = {
            "version": PATCH_VERSION,
            "minimum_source_shards": 1,
            "multiple_intersecting_shards_allowed": True,
            "confirmed_intersecting_shards": 2,
            "all_intersecting_paths_loaded": True,
            "source_tiles_per_shard": 1024,
            "expected_total_source_tiles": 2048,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        MANIFEST.write_text(
            json.dumps(
                workspace_manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # PRESERVE FAILED LOG AND REMOVE STOPPED PID
        # ====================================================

        log_preserved = False
        preserved_log_path = None

        if LOG_FILE.is_file():
            preserved_log_path = (
                ROOT
                / "batch"
                / (
                    "city.log.before_built_multi_shard_patch."
                    + str(int(time.time()))
                )
            )

            LOG_FILE.replace(
                preserved_log_path
            )

            log_preserved = True

        stale_pid_removed = False

        if PID_FILE.is_file():
            PID_FILE.unlink()

            stale_pid_removed = True


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        programs_removed = False

        if PROGRAMS_ROOT.exists():
            programs_removed = any(
                PROGRAMS_ROOT.iterdir()
            )

            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL VERIFICATION
        # ====================================================

        for source_file in [
            DRIVER,
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )


        saved_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        saved_checksums = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                saved_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if (
            saved_checksums
            != template_checksums
        ):
            raise RuntimeError(
                "Saved driver template checksums "
                "do not match the templates."
            )


        saved_manifest = json.loads(
            MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_manifest.get(
                "driver_sha256"
            )
            != sha256_file(DRIVER)
        ):
            raise RuntimeError(
                "Workspace driver checksum mismatch "
                "after built multi-shard patch."
            )

        if (
            saved_manifest.get(
                "template_sha256"
            )
            != template_checksums
        ):
            raise RuntimeError(
                "Workspace template checksum mismatch "
                "after built multi-shard patch."
            )


    except Exception:
        restore_backup()

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )

        raise


    print("=" * 80)
    print(
        "MUMBAI BUILT-SURFACE MULTI-SHARD PATCH"
    )
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Confirmed intersecting source shards: 2")
    print("Minimum required source shards: 1")
    print("All selected source paths loaded: YES")
    print("Source tiles per shard: 1024")
    print("Expected total source tiles: 2048")
    print("Built-surface template modified: YES")
    print("Population template modified: NO")
    print("SMOD template modified: NO")
    print("Population multi-shard support: PRESERVED")
    print("Coastal canonical-grid patch: PRESERVED")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")

    print(
        "Failed log preserved:",
        log_preserved,
    )

    if preserved_log_path is not None:
        print(
            "Preserved failed log:",
            preserved_log_path,
        )

    print(
        "Stopped PID removed:",
        stale_pid_removed,
    )

    print(
        "Generated programs previously present:",
        programs_removed,
    )

    print("Generated programs reset: PASS")
    print("Existing HDFS outputs deleted: NO")
    print("HDFS modified: NO")
    print("Spark launched: NO")

    print(
        "MUMBAI BUILT MULTI-SHARD PATCH: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# LOCAL PATCHER VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_mumbai_built_multi_shard.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI BUILT MULTI-SHARD "
    "PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

PATCHER="/tmp/patch_mumbai_built_multi_shard.py"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B-BUILT RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("Existing HDFS outputs deleted: NO")
print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 5B-BUILT: SUCCESS")

LOCAL MUMBAI BUILT MULTI-SHARD PATCHER COMPILE: PASS
Expected patcher SHA-256: c8c5a00df416e4fe212242b94fe99085b29ccf0862bc1b3adbd98984b4dfaebf
Remote patcher SHA-256: c8c5a00df416e4fe212242b94fe99085b29ccf0862bc1b3adbd98984b4dfaebf
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS

REMOTE STDERR:
Traceback (most recent call last):
  File "/tmp/patch_mumbai_built_multi_shard.py", line 350, in <module>
    built_source = replace_once(
        built_source,
    ...<2 lines>...
        "Built source-count guard",
    )
  File "/tmp/patch_mumbai_built_multi_shard.py", line 95, in replace_once
    raise RuntimeError(
    ...<5 lines>...
    )
RuntimeError: Built source-count guard: expected one occurrence, found 0
Expected text:
    if selected_source_count != 1:
        raise RuntimeError(
            "Expected exactly one Bengaluru source shard, found "
            f"{selected_source_count}."
        )




RuntimeError: Remote command failed with return code 1.

In [24]:
# ============================================================
# CELL 5B-BUILT-INSPECT:
# EXACT MUMBAI BUILT-SURFACE SOURCE-SHARD INSPECTION
#
# Read-only:
#   - Does not modify the built-surface template
#   - Does not modify the driver or workspace manifest
#   - Does not modify generated programs
#   - Does not modify HDFS
#   - Does not launch Spark
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


cmd = r'''
set -e

ROOT="__MUMBAI_ROOT__"

TEMPLATE="$ROOT/templates/built_surface_epoch_template.py"

if [ ! -f "$TEMPLATE" ]; then
    echo "ERROR: Built-surface template is missing:"
    echo "$TEMPLATE"
    exit 20
fi


echo "============================================================"
echo "MUMBAI BUILT-SURFACE SHARD-LOGIC INSPECTION"
echo "============================================================"


python3 - "$TEMPLATE" <<'PY'
from pathlib import Path
import ast
import sys


template_path = Path(
    sys.argv[1]
)

source = template_path.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


print(
    "Template:",
    template_path,
)

print()


# ============================================================
# FIND RELEVANT SOURCE LINES
# ============================================================

patterns = [
    "Expected exactly one",
    "source shard",
    "selected_source_count",
    "selected_sources_df",
    "selected_source",
    "source_path",
    "source_df",
    "source GeoTIFF",
    "source_raster",
    "source_metadata",
    "all_tile_count",
    "EXPECTED_SOURCE_TILES",
    "EXPECTED_SELECTED_TILES",
    "selected_tile_count",
    "RS_FromGeoTiff",
    "RS_Tile",
]


matched_lines = set()

for line_number, line in enumerate(
    lines,
    start=1,
):
    if any(
        pattern in line
        for pattern in patterns
    ):
        matched_lines.add(
            line_number
        )


if not matched_lines:
    raise RuntimeError(
        "No built-surface shard-related "
        "source text was found."
    )


# ============================================================
# MERGE NEARBY SOURCE RANGES
# ============================================================

ranges = []

for line_number in sorted(
    matched_lines
):
    start = max(
        1,
        line_number - 25,
    )

    end = min(
        len(lines),
        line_number + 35,
    )

    if (
        ranges
        and start
        <= ranges[-1][1] + 1
    ):
        ranges[-1] = (
            ranges[-1][0],
            max(
                ranges[-1][1],
                end,
            ),
        )

    else:
        ranges.append(
            (
                start,
                end,
            )
        )


# ============================================================
# PRINT EXACT SOURCE WITH LINE NUMBERS
# ============================================================

for start, end in ranges:
    print(
        "=" * 80
    )

    print(
        "SOURCE LINES "
        + str(start)
        + "-"
        + str(end)
    )

    print(
        "=" * 80
    )

    for line_number in range(
        start,
        end + 1,
    ):
        print(
            str(line_number).rjust(5)
            + ": "
            + lines[
                line_number - 1
            ]
        )

    print()


# ============================================================
# PRINT RELEVANT IF CONDITIONS
# ============================================================

tree = ast.parse(
    source
)


print(
    "=" * 80
)

print(
    "BUILT-SURFACE SHARD-RELATED IF CONDITIONS"
)

print(
    "=" * 80
)


if_count = 0

for node in ast.walk(
    tree
):
    if not isinstance(
        node,
        ast.If,
    ):
        continue

    node_text = ast.get_source_segment(
        source,
        node,
    )

    if node_text is None:
        continue

    lowered = node_text.lower()

    relevant_terms = [
        "shard",
        "source_count",
        "selected_source",
        "geotiff",
        "tile_count",
        "all_tile_count",
    ]

    if not any(
        term in lowered
        for term in relevant_terms
    ):
        continue

    if_count += 1

    print(
        "IF line:",
        node.lineno,
    )

    print(
        "Condition:",
        ast.unparse(
            node.test
        ),
    )

    print(
        "Block:"
    )

    print(
        node_text
    )

    print(
        "-" * 80
    )


if if_count == 0:
    raise RuntimeError(
        "No built-surface shard-related "
        "IF condition was found."
    )


# ============================================================
# PRINT RELEVANT ASSIGNMENTS
# ============================================================

print()

print(
    "=" * 80
)

print(
    "BUILT-SURFACE RELEVANT ASSIGNMENTS"
)

print(
    "=" * 80
)


assignment_count = 0

relevant_assignment_names = {
    "selected_source_count",
    "selected_source",
    "source_path",
    "source_df",
    "source_metadata",
    "tiles_df",
    "all_tile_count",
    "selected_tiles_df",
    "selected_tile_count",
}

for node in ast.walk(
    tree
):
    if not isinstance(
        node,
        ast.Assign,
    ):
        continue

    assigned_names = []

    for target in node.targets:
        if isinstance(
            target,
            ast.Name,
        ):
            assigned_names.append(
                target.id
            )

    if not any(
        name in relevant_assignment_names
        for name in assigned_names
    ):
        continue

    assignment_count += 1

    print(
        "Assignment line:",
        node.lineno,
    )

    print(
        "Names:",
        assigned_names,
    )

    print(
        ast.get_source_segment(
            source,
            node,
        )
    )

    print(
        "-" * 80
    )


print(
    "Shard-related IF conditions:",
    if_count,
)

print(
    "Relevant assignments:",
    assignment_count,
)

print(
    "MUMBAI_BUILT_SHARD_INSPECTION_SUCCESS"
)
PY


echo
echo "Mumbai workspace modified: NO"
echo "Generated programs modified: NO"
echo "HDFS modified: NO"
echo "Spark launched: NO"
echo "CELL_5B_BUILT_INSPECT_REMOTE_SUCCESS"
'''


cmd = cmd.replace(
    "__MUMBAI_ROOT__",
    root,
)


inspection_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B-BUILT-INSPECT RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Mumbai workspace modified: NO")
print("Generated programs modified: NO")
print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 5B-BUILT-INSPECT: SUCCESS")

MUMBAI BUILT-SURFACE SHARD-LOGIC INSPECTION
Template: /home/ubuntu/ghsl_multicity/mumbai_metro/templates/built_surface_epoch_template.py

SOURCE LINES 24-85
   24:     "hdfs://localhost:9000"
   25:     "/data/ghsl/curated/cities/city=mumbai_metro/"
   26:     "product=built_surface/"
   27:     "epoch=2020/"
   28:     "grid_1km"
   29: )
   30: 
   31: CITY = "mumbai_metro"
   32: PRODUCT = "built_surface"
   33: EPOCH = 2020
   34: 
   35: SOURCE_WIDTH = 8192
   36: SOURCE_HEIGHT = 8192
   37: SOURCE_BANDS = 2
   38: 
   39: SOURCE_RESOLUTION_M = 100.0
   40: TARGET_RESOLUTION_M = 1000.0
   41: 
   42: SOURCE_PIXEL_AREA_M2 = 10000.0
   43: COMPLETE_GRID_AREA_M2 = 1000000.0
   44: 
   45: TILE_WIDTH = 256
   46: TILE_HEIGHT = 256
   47: TILE_COLUMNS = SOURCE_WIDTH // TILE_WIDTH
   48: 
   49: EXPECTED_SOURCE_TILES = 1024
   50: EXPECTED_SELECTED_TILES = 16
   51: EXPECTED_GRID_CELLS = 4900
   52: EXPECTED_COMPLETE_PIXELS = 100
   53: 
   54: NUMERICAL_TOLERANCE = 1e-6
   55: 
   56: 

In [25]:
# ============================================================
# CELL 5B-BUILT:
# MUMBAI BUILT-SURFACE MULTI-SHARD PATCH V2
#
# Confirmed Mumbai condition:
#   - Mumbai intersects 2 built-surface source GeoTIFF shards
#   - The inherited template assumes exactly 1 source shard
#
# Corrected behavior:
#   - Requires at least one intersecting source shard
#   - Loads every intersecting source GeoTIFF
#   - Validates every catalog source record
#   - Validates every decoded raster
#   - Tiles all intersecting source rasters
#   - Expects:
#       EXPECTED_SOURCE_TILES * selected_source_count
#   - Keeps EXPECTED_SELECTED_TILES unchanged at 16
#   - Uses the existing Mumbai spatial tile filter
#
# Also:
#   - creates a transactional rollback backup
#   - preserves population multi-shard support
#   - preserves all coastal canonical-grid corrections
#   - refreshes driver template checksums
#   - refreshes the Mumbai workspace manifest
#   - preserves the failed production log
#   - removes only a stopped/stale PID file
#   - resets generated programs
#
# Does not:
#   - modify the frozen baseline
#   - modify or delete valid HDFS outputs
#   - launch Spark
# ============================================================

import base64
import hashlib
import json
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


# ============================================================
# EXACT PATCH FRAGMENT 1:
# SOURCE-SELECTION AND CATALOG-METADATA LOGIC
# ============================================================

old_source_selection_block = r'''
    selected_source_count = selected_sources_df.count()

    if selected_source_count != 1:
        raise RuntimeError(
            "Expected exactly one Bengaluru source shard, "
            f"found {selected_source_count}."
        )

    selected_source = selected_sources_df.first()
    source_path = selected_source["path"]

    if (
        int(selected_source["width"]) != SOURCE_WIDTH
        or int(selected_source["height"]) != SOURCE_HEIGHT
        or int(selected_source["numBands"]) != SOURCE_BANDS
    ):
        raise RuntimeError(
            "Unexpected built-surface source structure."
        )

    print("Selected source:", source_path)
'''.strip("\n")


new_source_selection_block = r'''
    selected_source_count = selected_sources_df.count()

    # ====================================================
    # MUMBAI BUILT-SURFACE MULTI-SHARD SUPPORT
    #
    # Mumbai crosses a built-surface source-shard boundary.
    # Every catalog shard intersecting the 70 km extent must
    # therefore be loaded and processed.
    # ====================================================

    if selected_source_count < 1:
        raise RuntimeError(
            "No built-surface source shard intersects "
            "the Mumbai extent."
        )

    selected_source_records = (
        selected_sources_df.collect()
    )

    selected_source_paths = [
        record["path"]
        for record
        in selected_source_records
    ]

    if (
        len(selected_source_records)
        != selected_source_count
    ):
        raise RuntimeError(
            "Collected built-surface source-record "
            "count does not match the selected "
            "source-shard count."
        )

    if (
        len(selected_source_paths)
        != selected_source_count
    ):
        raise RuntimeError(
            "Collected built-surface source-path "
            "count does not match the selected "
            "source-shard count."
        )

    if (
        len(set(selected_source_paths))
        != selected_source_count
    ):
        raise RuntimeError(
            "Duplicate built-surface source paths "
            "were selected."
        )

    for selected_source in (
        selected_source_records
    ):
        if (
            int(selected_source["width"])
            != SOURCE_WIDTH
            or int(selected_source["height"])
            != SOURCE_HEIGHT
            or int(selected_source["numBands"])
            != SOURCE_BANDS
        ):
            raise RuntimeError(
                "Unexpected built-surface source "
                "structure for "
                + str(
                    selected_source["path"]
                )
            )

    print(
        "Selected built-surface sources:",
        selected_source_paths,
    )
'''.strip("\n")


# ============================================================
# EXACT PATCH FRAGMENT 2:
# LOAD ALL SELECTED GEOTIFFS
# ============================================================

old_source_load_block = r'''
    source_df = (
        spark.read
        .format("binaryFile")
        .load(source_path)
        .select(
            "path",
            F.col("length").alias("source_file_bytes"),
            F.expr(
                "RS_FromGeoTiff(content)"
            ).alias("source_raster"),
        )
    )

    if source_df.count() != 1:
        raise RuntimeError(
            "Expected exactly one source GeoTIFF."
        )

    source_df.createOrReplaceTempView(
        "built_surface_source"
    )
'''.strip("\n")


new_source_load_block = r'''
    source_df = (
        spark.read
        .format("binaryFile")
        .load(selected_source_paths)
        .select(
            "path",
            F.col("length").alias("source_file_bytes"),
            F.expr(
                "RS_FromGeoTiff(content)"
            ).alias("source_raster"),
        )
    )

    decoded_source_count = (
        source_df.count()
    )

    if (
        decoded_source_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Decoded built-surface GeoTIFF count "
            "does not match the selected source-shard "
            "count. "
            f"Selected={selected_source_count}, "
            f"decoded={decoded_source_count}."
        )

    decoded_source_path_count = (
        source_df
        .select("path")
        .distinct()
        .count()
    )

    if (
        decoded_source_path_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Decoded built-surface source paths "
            "are not unique."
        )

    source_df.createOrReplaceTempView(
        "built_surface_source"
    )
'''.strip("\n")


# ============================================================
# EXACT PATCH FRAGMENT 3:
# VALIDATE EVERY DECODED SOURCE RASTER
# ============================================================

old_metadata_block = r'''
    metadata = spark.sql(
        """
        SELECT
            RS_Width(source_raster)
                AS source_width,
            RS_Height(source_raster)
                AS source_height,
            RS_NumBands(source_raster)
                AS source_band_count,
            RS_ScaleX(source_raster)
                AS source_scale_x,
            RS_ScaleY(source_raster)
                AS source_scale_y
        FROM built_surface_source
        """
    ).first()

    if (
        int(metadata["source_width"]) != SOURCE_WIDTH
        or int(metadata["source_height"]) != SOURCE_HEIGHT
        or int(metadata["source_band_count"])
        != SOURCE_BANDS
    ):
        raise RuntimeError(
            "Decoded metadata validation failed."
        )

    if (
        abs(
            float(metadata["source_scale_x"])
            - SOURCE_RESOLUTION_M
        )
        > 1e-9
        or abs(
            abs(float(metadata["source_scale_y"]))
            - SOURCE_RESOLUTION_M
        )
        > 1e-9
    ):
        raise RuntimeError(
            "Built-surface resolution is not 100 m."
        )

    print("Decoded source metadata: PASS")
'''.strip("\n")


new_metadata_block = r'''
    metadata_df = spark.sql(
        """
        SELECT
            path,
            RS_Width(source_raster)
                AS source_width,
            RS_Height(source_raster)
                AS source_height,
            RS_NumBands(source_raster)
                AS source_band_count,
            RS_ScaleX(source_raster)
                AS source_scale_x,
            RS_ScaleY(source_raster)
                AS source_scale_y
        FROM built_surface_source
        """
    )

    metadata_count = (
        metadata_df.count()
    )

    if (
        metadata_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Decoded built-surface metadata count "
            "does not match the selected "
            "source-shard count."
        )

    invalid_metadata_count = (
        metadata_df
        .filter(
            (
                F.col("source_width")
                != F.lit(SOURCE_WIDTH)
            )
            | (
                F.col("source_height")
                != F.lit(SOURCE_HEIGHT)
            )
            | (
                F.col("source_band_count")
                != F.lit(SOURCE_BANDS)
            )
            | (
                F.abs(
                    F.col("source_scale_x")
                    - F.lit(
                        float(
                            SOURCE_RESOLUTION_M
                        )
                    )
                )
                > F.lit(1e-9)
            )
            | (
                F.abs(
                    F.abs(
                        F.col("source_scale_y")
                    )
                    - F.lit(
                        float(
                            SOURCE_RESOLUTION_M
                        )
                    )
                )
                > F.lit(1e-9)
            )
        )
        .count()
    )

    if invalid_metadata_count != 0:
        raise RuntimeError(
            "One or more decoded built-surface "
            "rasters failed metadata validation."
        )

    print(
        "Decoded built-surface metadata records:",
        metadata_count,
    )

    print("Decoded source metadata: PASS")
'''.strip("\n")


# ============================================================
# EXACT PATCH FRAGMENT 4:
# SOURCE-TILE COUNT FOR ALL SHARDS
# ============================================================

old_source_tile_block = r'''
    source_tile_count = tiles_df.count()

    if source_tile_count != EXPECTED_SOURCE_TILES:
        raise RuntimeError(
            "Expected 1024 source tiles, found "
            f"{source_tile_count}."
        )
'''.strip("\n")


new_source_tile_block = r'''
    source_tile_count = tiles_df.count()

    expected_source_tile_count = (
        EXPECTED_SOURCE_TILES
        * selected_source_count
    )

    if (
        source_tile_count
        != expected_source_tile_count
    ):
        raise RuntimeError(
            "Unexpected built-surface source-tile "
            "count. "
            f"Expected {expected_source_tile_count} "
            f"from {selected_source_count} source "
            f"shard(s), found {source_tile_count}."
        )

    print(
        "Built-surface source shards tiled:",
        selected_source_count,
    )

    print(
        "Total built-surface source tiles:",
        source_tile_count,
    )
'''.strip("\n")


# ============================================================
# EXACT PATCH FRAGMENT 5:
# DYNAMIC SELECTED-TILE ERROR MESSAGE
#
# The actual constant remains unchanged:
#   EXPECTED_SELECTED_TILES = 16
# ============================================================

old_selected_tile_block = r'''
    if selected_tile_count != EXPECTED_SELECTED_TILES:
        raise RuntimeError(
            "Expected 12 Bengaluru tiles, found "
            f"{selected_tile_count}."
        )
'''.strip("\n")


new_selected_tile_block = r'''
    if selected_tile_count != EXPECTED_SELECTED_TILES:
        raise RuntimeError(
            "Expected "
            + str(EXPECTED_SELECTED_TILES)
            + " selected built-surface tiles, found "
            + str(selected_tile_count)
            + "."
        )
'''.strip("\n")


# ============================================================
# ENCODE PATCH PAYLOAD
# ============================================================

patch_payload = {
    "old_source_selection_block":
        old_source_selection_block,
    "new_source_selection_block":
        new_source_selection_block,
    "old_source_load_block":
        old_source_load_block,
    "new_source_load_block":
        new_source_load_block,
    "old_metadata_block":
        old_metadata_block,
    "new_metadata_block":
        new_metadata_block,
    "old_source_tile_block":
        old_source_tile_block,
    "new_source_tile_block":
        new_source_tile_block,
    "old_selected_tile_block":
        old_selected_tile_block,
    "new_selected_tile_block":
        new_selected_tile_block,
}

payload_json = json.dumps(
    patch_payload,
    sort_keys=True,
)

payload_b64 = base64.b64encode(
    payload_json.encode("utf-8")
).decode("ascii")

payload_sha256 = hashlib.sha256(
    payload_json.encode("utf-8")
).hexdigest()


# ============================================================
# REMOTE TRANSACTIONAL PATCHER
# ============================================================

remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    PAYLOAD_PATH = Path(
        sys.argv[2]
    )

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    WORKSPACE_MANIFEST = (
        ROOT
        / "mumbai_workspace_manifest.json"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    PID_FILE = (
        ROOT
        / "batch"
        / "city.pid"
    )

    LOG_FILE = (
        ROOT
        / "batch"
        / "city.log"
    )

    BACKUP_ROOT = (
        ROOT
        / "built_multi_shard_patch_backup"
    )

    PATCH_VERSION = (
        "mumbai_built_surface_multi_shard_v2"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def replace_exactly_once(
        source,
        old_text,
        new_text,
        description,
    ):
        actual_count = source.count(
            old_text
        )

        if actual_count != 1:
            raise RuntimeError(
                description
                + ": expected exactly one old block, "
                + "found "
                + str(actual_count)
                + "\nExpected block:\n"
                + old_text
            )

        return source.replace(
            old_text,
            new_text,
            1,
        )


    required_files = [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
        WORKSPACE_MANIFEST,
        PAYLOAD_PATH,
    ]

    for required_file in (
        required_files
    ):
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    # ========================================================
    # REFUSE TO PATCH A RUNNING JOB
    # ========================================================

    recorded_pid = None

    if PID_FILE.is_file():
        recorded_pid = (
            PID_FILE.read_text(
                encoding="utf-8"
            ).strip()
        )

        if (
            recorded_pid
            and (
                Path("/proc")
                / recorded_pid
            ).exists()
        ):
            raise RuntimeError(
                "Mumbai production is still running "
                "with PID "
                + recorded_pid
            )


    # ========================================================
    # VERIFY EXISTING POPULATION AND COASTAL CORRECTIONS
    # ========================================================

    population_source = (
        POPULATION_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    required_population_fragments = [
        "MUMBAI MULTI-SHARD SOURCE SUPPORT",
        "selected_source_paths",
        ".load(selected_source_paths)",
        "decoded_source_count",
        "expected_all_tile_count",
        "COASTAL CANONICAL-GRID COMPLETION",
        "has_valid_population_coverage",
    ]

    missing_population_fragments = [
        fragment
        for fragment
        in required_population_fragments
        if fragment
        not in population_source
    ]

    if missing_population_fragments:
        raise RuntimeError(
            "Required Mumbai population corrections "
            "are missing: "
            + repr(
                missing_population_fragments
            )
        )


    smod_source = (
        SMOD_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    required_smod_fragments = [
        "COASTAL CANONICAL-GRID COMPLETION",
        "has_valid_smod_coverage",
    ]

    missing_smod_fragments = [
        fragment
        for fragment
        in required_smod_fragments
        if fragment not in smod_source
    ]

    if missing_smod_fragments:
        raise RuntimeError(
            "Required Mumbai SMOD coastal "
            "correction is missing: "
            + repr(
                missing_smod_fragments
            )
        )


    built_source_before = (
        BUILT_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    required_existing_built_fragments = [
        "COASTAL CANONICAL-GRID COMPLETION",
        "has_valid_built_coverage",
        "EXPECTED_SELECTED_TILES = 16",
    ]

    missing_existing_built_fragments = [
        fragment
        for fragment
        in required_existing_built_fragments
        if fragment
        not in built_source_before
    ]

    if missing_existing_built_fragments:
        raise RuntimeError(
            "Required existing Mumbai built-surface "
            "logic is missing: "
            + repr(
                missing_existing_built_fragments
            )
        )


    # ========================================================
    # LOAD PATCH PAYLOAD
    # ========================================================

    payload = json.loads(
        PAYLOAD_PATH.read_text(
            encoding="utf-8"
        )
    )


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        WORKSPACE_MANIFEST,
        BACKUP_ROOT
        / WORKSPACE_MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / WORKSPACE_MANIFEST.name,
            WORKSPACE_MANIFEST,
        )


    try:
        # ====================================================
        # PATCH BUILT-SURFACE TEMPLATE
        # ====================================================

        built_source = (
            built_source_before
        )

        if (
            "MUMBAI BUILT-SURFACE MULTI-SHARD SUPPORT"
            in built_source
        ):
            raise RuntimeError(
                "Mumbai built-surface multi-shard "
                "patch is already applied."
            )


        replacement_pairs = [
            (
                payload[
                    "old_source_selection_block"
                ],
                payload[
                    "new_source_selection_block"
                ],
                "Built source-selection block",
            ),
            (
                payload[
                    "old_source_load_block"
                ],
                payload[
                    "new_source_load_block"
                ],
                "Built source-load block",
            ),
            (
                payload[
                    "old_metadata_block"
                ],
                payload[
                    "new_metadata_block"
                ],
                "Built metadata-validation block",
            ),
            (
                payload[
                    "old_source_tile_block"
                ],
                payload[
                    "new_source_tile_block"
                ],
                "Built source-tile block",
            ),
            (
                payload[
                    "old_selected_tile_block"
                ],
                payload[
                    "new_selected_tile_block"
                ],
                "Built selected-tile message block",
            ),
        ]


        for (
            old_text,
            new_text,
            description,
        ) in replacement_pairs:
            built_source = (
                replace_exactly_once(
                    built_source,
                    old_text,
                    new_text,
                    description,
                )
            )


        # ====================================================
        # VERIFY REQUIRED NEW BUILT-SURFACE LOGIC
        # ====================================================

        required_built_fragments = [
            "MUMBAI BUILT-SURFACE MULTI-SHARD SUPPORT",
            "selected_source_records",
            "selected_source_paths",
            ".load(selected_source_paths)",
            "decoded_source_count",
            "decoded_source_path_count",
            "metadata_df",
            "metadata_count",
            "invalid_metadata_count",
            "expected_source_tile_count",
            (
                "EXPECTED_SOURCE_TILES\n"
                "        * selected_source_count"
            ),
            "EXPECTED_SELECTED_TILES = 16",
            "COASTAL CANONICAL-GRID COMPLETION",
            "has_valid_built_coverage",
        ]

        missing_built_fragments = [
            fragment
            for fragment
            in required_built_fragments
            if fragment
            not in built_source
        ]

        if missing_built_fragments:
            raise RuntimeError(
                "Required built-surface multi-shard "
                "logic is missing: "
                + repr(
                    missing_built_fragments
                )
            )


        forbidden_built_fragments = [
            "if selected_source_count != 1:",
            "selected_sources_df.first()",
            ".load(source_path)",
            "if source_df.count() != 1:",
            "Expected exactly one source GeoTIFF.",
            (
                "if source_tile_count "
                "!= EXPECTED_SOURCE_TILES:"
            ),
            "Expected 12 Bengaluru tiles",
        ]

        remaining_forbidden = [
            fragment
            for fragment
            in forbidden_built_fragments
            if fragment
            in built_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Old single-shard built-surface "
                "logic remains: "
                + repr(
                    remaining_forbidden
                )
            )


        compile(
            built_source,
            str(BUILT_TEMPLATE),
            "exec",
        )

        BUILT_TEMPLATE.write_text(
            built_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(BUILT_TEMPLATE),
            doraise=True,
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # REFRESH DRIVER CHECKSUM DICTIONARY
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                checksum_node = node
                break


        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found in the driver."
            )


        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                template_checksums
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH WORKSPACE MANIFEST
        # ====================================================

        manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        manifest[
            "template_sha256"
        ] = template_checksums

        manifest[
            "built_surface_multi_shard_patch"
        ] = {
            "version": PATCH_VERSION,
            "minimum_source_shards": 1,
            "multiple_intersecting_shards_allowed": True,
            "confirmed_intersecting_shards": 2,
            "all_intersecting_paths_loaded": True,
            "source_tiles_per_shard": 1024,
            "expected_total_source_tiles": 2048,
            "expected_selected_tiles": 16,
            "coastal_grid_preserved": True,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        WORKSPACE_MANIFEST.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # PRESERVE FAILED LOG AND REMOVE STOPPED PID
        # ====================================================

        log_preserved = False
        preserved_log_path = None

        if LOG_FILE.is_file():
            preserved_log_path = (
                ROOT
                / "batch"
                / (
                    "city.log.before_built_multi_shard_patch."
                    + str(int(time.time()))
                )
            )

            LOG_FILE.replace(
                preserved_log_path
            )

            log_preserved = True


        stale_pid_removed = False

        if PID_FILE.is_file():
            PID_FILE.unlink()

            stale_pid_removed = True


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        programs_removed = False

        if PROGRAMS_ROOT.exists():
            programs_removed = any(
                PROGRAMS_ROOT.iterdir()
            )

            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL STATIC VERIFICATION
        # ====================================================

        for source_file in [
            DRIVER,
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )


        final_population_source = (
            POPULATION_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        for fragment in (
            required_population_fragments
        ):
            if (
                fragment
                not in final_population_source
            ):
                raise RuntimeError(
                    "Population correction was lost: "
                    + fragment
                )


        final_smod_source = (
            SMOD_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        for fragment in (
            required_smod_fragments
        ):
            if fragment not in final_smod_source:
                raise RuntimeError(
                    "SMOD correction was lost: "
                    + fragment
                )


        final_built_source = (
            BUILT_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        for fragment in required_built_fragments:
            if fragment not in final_built_source:
                raise RuntimeError(
                    "Final built-surface template "
                    "is missing: "
                    + fragment
                )


        saved_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        saved_checksums = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                saved_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break


        if (
            saved_checksums
            != template_checksums
        ):
            raise RuntimeError(
                "Saved driver checksums do not "
                "match the patched templates."
            )


        saved_manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_manifest.get(
                "driver_sha256"
            )
            != sha256_file(DRIVER)
        ):
            raise RuntimeError(
                "Workspace driver checksum was "
                "not refreshed."
            )

        if (
            saved_manifest.get(
                "template_sha256"
            )
            != template_checksums
        ):
            raise RuntimeError(
                "Workspace template checksums "
                "were not refreshed."
            )


    except Exception:
        restore_backup()

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )

        raise


    print("=" * 80)
    print(
        "MUMBAI BUILT-SURFACE MULTI-SHARD PATCH"
    )
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Confirmed intersecting source shards: 2")
    print("Minimum required source shards: 1")
    print("All selected source paths loaded: YES")
    print("Decoded metadata validated per shard: YES")
    print("Source tiles per shard: 1024")
    print("Expected total source tiles: 2048")
    print("Expected selected tiles: 16")
    print("Built-surface template modified: YES")
    print("Population template modified: NO")
    print("SMOD template modified: NO")
    print("Population multi-shard support: PRESERVED")
    print("Coastal canonical-grid patch: PRESERVED")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")

    print(
        "Failed log preserved:",
        log_preserved,
    )

    if preserved_log_path is not None:
        print(
            "Preserved failed log:",
            preserved_log_path,
        )

    print(
        "Stopped PID removed:",
        stale_pid_removed,
    )

    print(
        "Generated programs previously present:",
        programs_removed,
    )

    print("Generated programs reset: PASS")
    print("Existing HDFS outputs deleted: NO")
    print("HDFS modified: NO")
    print("Spark launched: NO")

    print(
        "MUMBAI BUILT MULTI-SHARD PATCH: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# LOCAL PATCHER VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_mumbai_built_multi_shard_v2.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI BUILT MULTI-SHARD "
    "PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

PATCHER="/tmp/patch_mumbai_built_multi_shard_v2.py"
PAYLOAD="/tmp/mumbai_built_multi_shard_payload.json"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"
EXPECTED_PAYLOAD_SHA256="{payload_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

echo "{payload_b64}" \
| base64 -d \
> "${{PAYLOAD}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

ACTUAL_PAYLOAD_SHA256="$(
    sha256sum "${{PAYLOAD}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"
echo "Remote payload SHA-256: ${{ACTUAL_PAYLOAD_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

if [ "${{ACTUAL_PAYLOAD_SHA256}}" != "${{EXPECTED_PAYLOAD_SHA256}}" ]; then
    echo "ERROR: Payload checksum mismatch."
    exit 21
fi

echo "REMOTE PATCHER CHECKSUM: PASS"
echo "REMOTE PAYLOAD CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}" \
    "${{PAYLOAD}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B-BUILT RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("Existing HDFS outputs deleted: NO")
print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 5B-BUILT: SUCCESS")

LOCAL MUMBAI BUILT MULTI-SHARD PATCHER COMPILE: PASS
Expected patcher SHA-256: 307a998aeaae5bd8c49cceac5b9f01153c29d0ed580c8f1cc38e174fa7bc9473
Remote patcher SHA-256: 307a998aeaae5bd8c49cceac5b9f01153c29d0ed580c8f1cc38e174fa7bc9473
Remote payload SHA-256: dc9816e0703f9621b9794c7d622d9e6b120c268e549261935cff8df83a378f1d
REMOTE PATCHER CHECKSUM: PASS
REMOTE PAYLOAD CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
MUMBAI BUILT-SURFACE MULTI-SHARD PATCH
Backup: /home/ubuntu/ghsl_multicity/mumbai_metro/built_multi_shard_patch_backup

Confirmed intersecting source shards: 2
Minimum required source shards: 1
All selected source paths loaded: YES
Decoded metadata validated per shard: YES
Source tiles per shard: 1024
Expected total source tiles: 2048
Expected selected tiles: 16
Built-surface template modified: YES
Population template modified: NO
SMOD template modified: NO
Population multi-shard support: PRESERVED
Coastal canonical-grid patch: PRESERVED
Driver checksums refreshed: PASS
Workspace ma

In [13]:
# ============================================================
# CELL 5C: INSPECT MUMBAI POPULATION RASTER-TO-TILE PATH
#
# Read-only:
#   - Does not modify the workspace
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


cmd = r'''
set -e

ROOT="__MUMBAI_ROOT__"
TEMPLATE="$ROOT/templates/population_epoch_template.py"

if [ ! -f "$TEMPLATE" ]; then
    echo "ERROR: Population template is missing:"
    echo "$TEMPLATE"
    exit 20
fi

echo "============================================================"
echo "MUMBAI POPULATION RASTER-TO-TILE INSPECTION"
echo "============================================================"

sed -n '145,540p' "$TEMPLATE"

echo
echo "============================================================"
echo "KEY RASTER AND TILE REFERENCES"
echo "============================================================"

grep -n \
    -E \
    'selected_source|source_path|source_df|population_source|RS_FromGeoTiff|RS_Tile|selected_tiles_df|selected_tile_count|EXPECTED_SELECTED_TILES|source_metadata' \
    "$TEMPLATE"

echo
echo "Mumbai workspace modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "MUMBAI_RASTER_TILE_INSPECTION_SUCCESS"
'''


cmd = cmd.replace(
    "__MUMBAI_ROOT__",
    root,
)


inspection_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5C RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Mumbai workspace modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 5C: SUCCESS")

MUMBAI POPULATION RASTER-TO-TILE INSPECTION
    selected_source_count = (
        selected_sources_df.count()
    )

    print(
        "2020 Bengaluru source shards:",
        selected_source_count,
    )

    selected_sources_df.show(
        10,
        truncate=False,
        vertical=True,
    )

    if selected_source_count != 1:
        raise RuntimeError(
            "Expected exactly one 2020 population shard "
            "for Bengaluru, found "
            f"{selected_source_count}."
        )

    selected_source = (
        selected_sources_df.first()
    )

    source_path = selected_source["path"]

    if (
        int(selected_source["width"])
        != SOURCE_WIDTH
        or int(selected_source["height"])
        != SOURCE_HEIGHT
    ):
        raise RuntimeError(
            "Unexpected 2020 population shard dimensions."
        )

    print("Selected source:", source_path)


    # ====================================================
    # READ AND DECODE THE SELECT

In [14]:
# ============================================================
# CELL 5B: ENABLE MULTI-SHARD MUMBAI POPULATION PROCESSING
#
# Confirmed Mumbai condition:
#   - Mumbai intersects 2 population source GeoTIFF shards
#   - The frozen template assumes exactly 1 source shard
#
# Corrected behavior:
#   - Requires at least one intersecting source shard
#   - Loads every intersecting source GeoTIFF
#   - Validates every decoded raster
#   - Tiles all intersecting source rasters
#   - Expects:
#       EXPECTED_SOURCE_TILES * selected_source_count
#   - Leaves the existing Mumbai spatial tile filter unchanged
#
# Also:
#   - Creates a transactional rollback backup
#   - Refreshes template checksums in the driver
#   - Refreshes the Mumbai workspace manifest
#   - Resets generated programs
#   - Removes generated bytecode
#
# Does not:
#   - modify the frozen baseline
#   - modify other city workspaces
#   - launch Spark
#   - modify HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


# ============================================================
# REPLACEMENT 1:
# SINGLE-SOURCE DISCOVERY/LOAD/METADATA
#                       TO
# MULTI-SOURCE DISCOVERY/LOAD/METADATA
# ============================================================

old_source_block = r'''
    if selected_source_count != 1:
        raise RuntimeError(
            "Expected exactly one 2020 population shard "
            "for Bengaluru, found "
            f"{selected_source_count}."
        )

    selected_source = (
        selected_sources_df.first()
    )

    source_path = selected_source["path"]

    if (
        int(selected_source["width"])
        != SOURCE_WIDTH
        or int(selected_source["height"])
        != SOURCE_HEIGHT
    ):
        raise RuntimeError(
            "Unexpected 2020 population shard dimensions."
        )

    print("Selected source:", source_path)


    # ====================================================
    # READ AND DECODE THE SELECTED GEOTIFF
    # ====================================================

    source_df = (
        spark.read
        .format("binaryFile")
        .load(source_path)
        .select(
            "path",
            F.col("length").alias(
                "source_file_bytes"
            ),
            F.expr(
                "RS_FromGeoTiff(content)"
            ).alias("source_raster"),
        )
    )

    if source_df.count() != 1:
        raise RuntimeError(
            "Expected exactly one source GeoTIFF."
        )

    source_df.createOrReplaceTempView(
        "population_source"
    )


    # ====================================================
    # VALIDATE SOURCE METADATA
    # ====================================================

    source_metadata = spark.sql(
        """
        SELECT
            RS_Width(source_raster)
                AS source_width,
            RS_Height(source_raster)
                AS source_height,
            RS_NumBands(source_raster)
                AS source_band_count,
            RS_ScaleX(source_raster)
                AS source_scale_x,
            RS_ScaleY(source_raster)
                AS source_scale_y
        FROM population_source
        """
    ).first()

    if (
        int(source_metadata["source_width"])
        != SOURCE_WIDTH
        or int(source_metadata["source_height"])
        != SOURCE_HEIGHT
        or int(source_metadata["source_band_count"])
        != 1
    ):
        raise RuntimeError(
            "2020 source metadata validation failed."
        )

    if (
        abs(
            float(source_metadata["source_scale_x"])
            - SOURCE_RESOLUTION
        )
        > 1e-9
        or abs(
            abs(
                float(
                    source_metadata["source_scale_y"]
                )
            )
            - SOURCE_RESOLUTION
        )
        > 1e-9
    ):
        raise RuntimeError(
            "2020 population resolution is not 100 m."
        )

    print("Source metadata: PASS")
'''.strip("\n")


new_source_block = r'''
    # ====================================================
    # MUMBAI MULTI-SHARD SOURCE SUPPORT
    #
    # Mumbai crosses a population-shard boundary. Therefore,
    # every catalogue shard intersecting the 70 km extent
    # must be loaded and processed.
    # ====================================================

    if selected_source_count < 1:
        raise RuntimeError(
            "No population source shard intersects "
            "the Mumbai extent."
        )

    selected_source_records = (
        selected_sources_df.collect()
    )

    selected_source_paths = [
        record["path"]
        for record
        in selected_source_records
    ]

    if (
        len(selected_source_paths)
        != selected_source_count
    ):
        raise RuntimeError(
            "Collected source-path count does not match "
            "the selected source-shard count."
        )

    if (
        len(set(selected_source_paths))
        != selected_source_count
    ):
        raise RuntimeError(
            "Duplicate population source paths were "
            "selected."
        )

    for selected_source in (
        selected_source_records
    ):
        if (
            int(selected_source["width"])
            != SOURCE_WIDTH
            or int(selected_source["height"])
            != SOURCE_HEIGHT
        ):
            raise RuntimeError(
                "Unexpected population source-shard "
                "dimensions for "
                + str(selected_source["path"])
            )

    print(
        "Selected source paths:",
        selected_source_paths,
    )


    # ====================================================
    # READ AND DECODE ALL SELECTED GEOTIFFS
    # ====================================================

    source_df = (
        spark.read
        .format("binaryFile")
        .load(selected_source_paths)
        .select(
            "path",
            F.col("length").alias(
                "source_file_bytes"
            ),
            F.expr(
                "RS_FromGeoTiff(content)"
            ).alias("source_raster"),
        )
    )

    decoded_source_count = (
        source_df.count()
    )

    if (
        decoded_source_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Decoded source GeoTIFF count does not "
            "match the selected source-shard count. "
            f"Selected={selected_source_count}, "
            f"decoded={decoded_source_count}."
        )

    decoded_path_count = (
        source_df
        .select("path")
        .distinct()
        .count()
    )

    if (
        decoded_path_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Decoded source paths are not unique."
        )

    source_df.createOrReplaceTempView(
        "population_source"
    )


    # ====================================================
    # VALIDATE EVERY SOURCE RASTER
    # ====================================================

    source_metadata_df = spark.sql(
        """
        SELECT
            path,
            RS_Width(source_raster)
                AS source_width,
            RS_Height(source_raster)
                AS source_height,
            RS_NumBands(source_raster)
                AS source_band_count,
            RS_ScaleX(source_raster)
                AS source_scale_x,
            RS_ScaleY(source_raster)
                AS source_scale_y
        FROM population_source
        """
    )

    source_metadata_count = (
        source_metadata_df.count()
    )

    if (
        source_metadata_count
        != selected_source_count
    ):
        raise RuntimeError(
            "Source metadata count does not match "
            "the selected source-shard count."
        )

    invalid_source_metadata_count = (
        source_metadata_df
        .filter(
            (
                F.col("source_width")
                != F.lit(SOURCE_WIDTH)
            )
            | (
                F.col("source_height")
                != F.lit(SOURCE_HEIGHT)
            )
            | (
                F.col("source_band_count")
                != F.lit(1)
            )
            | (
                F.abs(
                    F.col("source_scale_x")
                    - F.lit(
                        float(
                            SOURCE_RESOLUTION
                        )
                    )
                )
                > F.lit(1e-9)
            )
            | (
                F.abs(
                    F.abs(
                        F.col("source_scale_y")
                    )
                    - F.lit(
                        float(
                            SOURCE_RESOLUTION
                        )
                    )
                )
                > F.lit(1e-9)
            )
        )
        .count()
    )

    if (
        invalid_source_metadata_count
        != 0
    ):
        raise RuntimeError(
            "One or more Mumbai population source "
            "rasters failed metadata validation."
        )

    print(
        "Source metadata records:",
        source_metadata_count,
    )

    print("Source metadata: PASS")
'''.strip("\n")


# ============================================================
# REPLACEMENT 2:
# FIXED 1024-TILE EXPECTATION
#                       TO
# 1024 TILES PER SOURCE SHARD
# ============================================================

old_tile_check = r'''
    if all_tile_count != EXPECTED_SOURCE_TILES:
        raise RuntimeError(
            "Expected 1024 source tiles, found "
            f"{all_tile_count}."
        )
'''.strip("\n")


new_tile_check = r'''
    expected_all_tile_count = (
        EXPECTED_SOURCE_TILES
        * selected_source_count
    )

    if (
        all_tile_count
        != expected_all_tile_count
    ):
        raise RuntimeError(
            "Unexpected total source-tile count. "
            f"Expected {expected_all_tile_count} "
            f"from {selected_source_count} source "
            f"shard(s), found {all_tile_count}."
        )

    print(
        "Source shards tiled:",
        selected_source_count,
    )

    print(
        "Total source tiles:",
        all_tile_count,
    )
'''.strip("\n")


# ============================================================
# ENCODE PATCH PAYLOAD
# ============================================================

patch_payload = {
    "old_source_block": old_source_block,
    "new_source_block": new_source_block,
    "old_tile_check": old_tile_check,
    "new_tile_check": new_tile_check,
}

payload_json = json.dumps(
    patch_payload,
    sort_keys=True,
)

payload_b64 = base64.b64encode(
    payload_json.encode("utf-8")
).decode("ascii")

payload_sha256 = hashlib.sha256(
    payload_json.encode("utf-8")
).hexdigest()


# ============================================================
# REMOTE TRANSACTIONAL PATCHER
# ============================================================

remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    PAYLOAD_PATH = Path(
        sys.argv[2]
    )

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    WORKSPACE_MANIFEST = (
        ROOT
        / "mumbai_workspace_manifest.json"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    PID_FILE = (
        ROOT
        / "batch"
        / "city.pid"
    )

    LOG_FILE = (
        ROOT
        / "batch"
        / "city.log"
    )

    BACKUP_ROOT = (
        ROOT
        / "multi_shard_patch_backup"
    )

    PATCH_VERSION = (
        "mumbai_population_multi_shard_v2"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def replace_exactly_once(
        source,
        old_text,
        new_text,
        description,
    ):
        actual_count = source.count(
            old_text
        )

        if actual_count != 1:
            raise RuntimeError(
                description
                + ": expected exactly one old block, "
                + "found "
                + str(actual_count)
            )

        return source.replace(
            old_text,
            new_text,
            1,
        )


    required_files = [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
        WORKSPACE_MANIFEST,
        PAYLOAD_PATH,
    ]

    for required_file in (
        required_files
    ):
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    # ========================================================
    # REFUSE TO PATCH AFTER PRODUCTION LAUNCH
    # ========================================================

    if PID_FILE.exists():
        raise RuntimeError(
            "Mumbai city.pid already exists. "
            "Do not patch after production launch."
        )

    if LOG_FILE.exists():
        raise RuntimeError(
            "Mumbai city.log already exists. "
            "Do not patch after production launch."
        )


    # ========================================================
    # LOAD PATCH PAYLOAD
    # ========================================================

    payload = json.loads(
        PAYLOAD_PATH.read_text(
            encoding="utf-8"
        )
    )

    old_source_block = payload[
        "old_source_block"
    ]

    new_source_block = payload[
        "new_source_block"
    ]

    old_tile_check = payload[
        "old_tile_check"
    ]

    new_tile_check = payload[
        "new_tile_check"
    ]


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        WORKSPACE_MANIFEST,
        BACKUP_ROOT
        / WORKSPACE_MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / WORKSPACE_MANIFEST.name,
            WORKSPACE_MANIFEST,
        )


    try:
        # ====================================================
        # PATCH POPULATION TEMPLATE
        # ====================================================

        population_source = (
            POPULATION_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        if (
            "MUMBAI MULTI-SHARD SOURCE SUPPORT"
            in population_source
        ):
            raise RuntimeError(
                "Mumbai multi-shard support is "
                "already present."
            )

        population_source = (
            replace_exactly_once(
                population_source,
                old_source_block,
                new_source_block,
                "Population source-load block",
            )
        )

        population_source = (
            replace_exactly_once(
                population_source,
                old_tile_check,
                new_tile_check,
                "Population source-tile check",
            )
        )


        # Confirm old single-source operations are gone.

        forbidden_fragments = [
            "selected_sources_df.first()",
            ".load(source_path)",
            "Expected exactly one source GeoTIFF.",
            "if selected_source_count != 1:",
        ]

        remaining_forbidden = [
            fragment
            for fragment
            in forbidden_fragments
            if fragment
            in population_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Single-source population logic remains: "
                + repr(
                    remaining_forbidden
                )
            )


        required_fragments = [
            "MUMBAI MULTI-SHARD SOURCE SUPPORT",
            "selected_source_paths",
            ".load(selected_source_paths)",
            "decoded_source_count",
            "source_metadata_df",
            "expected_all_tile_count",
            (
                "EXPECTED_SOURCE_TILES\n"
                "        * selected_source_count"
            ),
        ]

        missing_required = [
            fragment
            for fragment
            in required_fragments
            if fragment
            not in population_source
        ]

        if missing_required:
            raise RuntimeError(
                "Required multi-shard logic is missing: "
                + repr(
                    missing_required
                )
            )


        compile(
            population_source,
            str(POPULATION_TEMPLATE),
            "exec",
        )

        POPULATION_TEMPLATE.write_text(
            population_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(POPULATION_TEMPLATE),
            doraise=True,
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # REFRESH DRIVER TEMPLATE CHECKSUM DICTIONARY
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                checksum_node = node
                break


        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 was "
                "not found in the driver."
            )


        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                template_checksums
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH WORKSPACE MANIFEST
        # ====================================================

        manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        manifest[
            "template_sha256"
        ] = template_checksums

        manifest[
            "population_multi_shard_patch"
        ] = {
            "version": PATCH_VERSION,
            "minimum_source_shards": 1,
            "multiple_intersecting_shards_allowed": True,
            "confirmed_intersecting_shards": 2,
            "source_tiles_per_shard": 1024,
            "expected_total_source_tiles": 2048,
            "all_intersecting_paths_loaded": True,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        WORKSPACE_MANIFEST.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        programs_removed = False

        if PROGRAMS_ROOT.exists():
            programs_removed = any(
                PROGRAMS_ROOT.iterdir()
            )

            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL STATIC VERIFICATION
        # ====================================================

        for source_file in [
            DRIVER,
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )


        saved_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        saved_checksums = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                saved_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break


        if (
            saved_checksums
            != template_checksums
        ):
            raise RuntimeError(
                "Saved driver checksums do not "
                "match the patched templates."
            )


        saved_manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_manifest.get(
                "driver_sha256"
            )
            != sha256_file(DRIVER)
        ):
            raise RuntimeError(
                "Workspace driver checksum was "
                "not refreshed."
            )

        if (
            saved_manifest.get(
                "template_sha256"
            )
            != template_checksums
        ):
            raise RuntimeError(
                "Workspace template checksums "
                "were not refreshed."
            )


    except Exception:
        restore_backup()

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )

        raise


    print("=" * 80)
    print("MUMBAI POPULATION MULTI-SHARD PATCH")
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Confirmed intersecting source shards: 2")
    print("All selected source paths loaded: YES")
    print("Source metadata validated per shard: YES")
    print("Source tiles per shard: 1024")
    print("Expected total source tiles: 2048")
    print("Population template modified: YES")
    print("SMOD template modified: NO")
    print("Built-surface template modified: NO")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")

    print(
        "Generated programs previously present:",
        programs_removed,
    )

    print("Generated programs reset: PASS")
    print("Python compilation: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")
    print(
        "MUMBAI MULTI-SHARD PATCH: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# LOCAL PATCHER VALIDATION
# ============================================================

compile(
    remote_patcher,
    "patch_mumbai_multi_shard_v2.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI MULTI-SHARD PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

PATCHER="/tmp/patch_mumbai_multi_shard_v2.py"
PAYLOAD="/tmp/mumbai_multi_shard_payload.json"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"
EXPECTED_PAYLOAD_SHA256="{payload_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

echo "{payload_b64}" \
| base64 -d \
> "${{PAYLOAD}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

ACTUAL_PAYLOAD_SHA256="$(
    sha256sum "${{PAYLOAD}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"
echo "Remote payload SHA-256: ${{ACTUAL_PAYLOAD_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

if [ "${{ACTUAL_PAYLOAD_SHA256}}" != "${{EXPECTED_PAYLOAD_SHA256}}" ]; then
    echo "ERROR: Payload checksum mismatch."
    exit 21
fi

echo "REMOTE PATCHER CHECKSUM: PASS"
echo "REMOTE PAYLOAD CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}" \
    "${{PAYLOAD}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5B RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 5B: SUCCESS")

LOCAL MUMBAI MULTI-SHARD PATCHER COMPILE: PASS
Expected patcher SHA-256: 70881238797946d6f883bd9ba5b4048f94babc562ccfcc16d668224a130c845a
Remote patcher SHA-256: 70881238797946d6f883bd9ba5b4048f94babc562ccfcc16d668224a130c845a
Remote payload SHA-256: 665f5baa10ab340eeedf9d233e583bc39f0438a7f40d9f3edb56c52a8bb1936e
REMOTE PATCHER CHECKSUM: PASS
REMOTE PAYLOAD CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
MUMBAI POPULATION MULTI-SHARD PATCH
Backup: /home/ubuntu/ghsl_multicity/mumbai_metro/multi_shard_patch_backup

Confirmed intersecting source shards: 2
All selected source paths loaded: YES
Source metadata validated per shard: YES
Source tiles per shard: 1024
Expected total source tiles: 2048
Population template modified: YES
SMOD template modified: NO
Built-surface template modified: NO
Driver checksums refreshed: PASS
Workspace manifest refreshed: PASS
Generated programs previously present: False
Generated programs reset: PASS
Python compilation: PASS
HDFS modified: NO
Spark launched: NO

In [16]:
# ============================================================
# CELL 5A: MUMBAI SELECTED-TILE COUNT PROBE
#
# Purpose:
#   - Generates a temporary Mumbai population-1975 program
#   - Uses the patched multi-shard population template
#   - Runs through source discovery, decoding and tile selection
#   - Prints Mumbai's actual selected-tile count
#   - Exits before the inherited expected-tile validation
#
# Does not:
#   - write Mumbai output to HDFS
#   - modify Mumbai templates or driver
#   - modify the workspace manifest
#   - launch the full production batch
#
# Temporary probe files are removed after execution.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_probe = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

ROOT="__MUMBAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"

PROBE_ROOT="/tmp/mumbai_tile_count_probe"
PROBE_PROGRAM="$PROBE_ROOT/population_1975_probe.py"


echo "============================================================"
echo "MUMBAI SELECTED-TILE COUNT PROBE"
echo "============================================================"


if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Mumbai driver is missing:"
    echo "$DRIVER"
    exit 20
fi


rm -rf "$PROBE_ROOT"

mkdir -p "$PROBE_ROOT"


# ============================================================
# GENERATE A TEMPORARY 1975 POPULATION PROGRAM
# ============================================================

python3 - \
    "$DRIVER" \
    "$PROBE_ROOT" \
    "$PROBE_PROGRAM" <<'PY'
from pathlib import Path
import ast
import sys


driver_path = Path(
    sys.argv[1]
)

probe_root = Path(
    sys.argv[2]
)

probe_program = Path(
    sys.argv[3]
)


driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)


# ============================================================
# LOAD DRIVER DEFINITIONS WITHOUT RUNNING THE BATCH LOOP
# ============================================================

allowed_node_types = (
    ast.Import,
    ast.ImportFrom,
    ast.Assign,
    ast.AnnAssign,
    ast.FunctionDef,
    ast.AsyncFunctionDef,
    ast.ClassDef,
)

definition_nodes = [
    node
    for node in driver_tree.body
    if isinstance(
        node,
        allowed_node_types,
    )
]

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


namespace = {
    "__file__": str(
        driver_path
    ),
    "__name__": (
        "mumbai_tile_count_probe"
    ),
}

exec(
    compile(
        definition_module,
        str(driver_path),
        "exec",
    ),
    namespace,
)


factory = namespace.get(
    "generate_population_program"
)

if not callable(factory):
    raise RuntimeError(
        "generate_population_program "
        "is unavailable."
    )


generated_program = factory(
    1975,
    probe_root,
)

generated_program = Path(
    generated_program
)


if not generated_program.is_file():
    raise RuntimeError(
        "The population factory did not "
        "create the 1975 program."
    )


probe_source = (
    generated_program.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# CONFIRM MULTI-SHARD SUPPORT IS PRESENT
# ============================================================

required_multi_shard_fragments = [
    "MUMBAI MULTI-SHARD SOURCE SUPPORT",
    "selected_source_paths",
    ".load(selected_source_paths)",
    "decoded_source_count",
    "expected_all_tile_count",
]

missing_multi_shard_fragments = [
    fragment
    for fragment
    in required_multi_shard_fragments
    if fragment not in probe_source
]

if missing_multi_shard_fragments:
    raise RuntimeError(
        "Generated probe is missing Mumbai "
        "multi-shard logic: "
        + repr(
            missing_multi_shard_fragments
        )
    )


# ============================================================
# INSERT CLEAN EXIT AFTER SELECTED-TILE COUNT
#
# String concatenation is used here because remote_probe is
# already enclosed by triple-single quotes.
# ============================================================

tile_count_assignment = (
    "    selected_tile_count = (\n"
    "        selected_tiles_df.count()\n"
    "    )\n"
)

probe_exit = (
    "\n"
    "    print(\n"
    '        "MUMBAI ACTUAL SELECTED TILES:",\n'
    "        selected_tile_count,\n"
    "    )\n"
    "\n"
    "    print(\n"
    '        "MUMBAI_TILE_COUNT_PROBE_SUCCESS"\n'
    "    )\n"
    "\n"
    "    raise SystemExit(0)\n"
)


assignment_count = probe_source.count(
    tile_count_assignment
)

if assignment_count != 1:
    raise RuntimeError(
        "Expected exactly one selected-tile-count "
        "assignment, found "
        + str(
            assignment_count
        )
    )


probe_source = probe_source.replace(
    tile_count_assignment,
    tile_count_assignment
    + probe_exit,
    1,
)


# ============================================================
# VERIFY EXIT POSITION
# ============================================================

exit_position = probe_source.find(
    "MUMBAI_TILE_COUNT_PROBE_SUCCESS"
)

expected_tile_check_position = (
    probe_source.find(
        "selected_tile_count\n"
        "        != EXPECTED_SELECTED_TILES"
    )
)

hdfs_write_position = (
    probe_source.find(
        ".parquet(OUTPUT_PATH)"
    )
)


if exit_position < 0:
    raise RuntimeError(
        "Probe exit marker is missing."
    )


if expected_tile_check_position < 0:
    raise RuntimeError(
        "The inherited selected-tile "
        "validation was not found."
    )


if (
    exit_position
    >= expected_tile_check_position
):
    raise RuntimeError(
        "Probe exit does not occur before "
        "the selected-tile validation."
    )


if (
    hdfs_write_position >= 0
    and exit_position
    >= hdfs_write_position
):
    raise RuntimeError(
        "Probe exit does not occur before "
        "the HDFS write."
    )


compile(
    probe_source,
    str(probe_program),
    "exec",
)


probe_program.write_text(
    probe_source,
    encoding="utf-8",
)


print(
    "Temporary probe program:",
    probe_program,
)

print(
    "Mumbai multi-shard logic present: PASS"
)

print(
    "Probe exits before selected-tile "
    "validation: PASS"
)

print(
    "Probe exits before HDFS write: PASS"
)

print(
    "Probe compilation: PASS"
)
PY


# ============================================================
# RUN THE TEMPORARY PROBE
# ============================================================

set +e

spark-submit \
    --master local[1] \
    --driver-memory 5g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
    org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
    https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "$PROBE_PROGRAM"

PROBE_RETURN_CODE="$?"

set -e


if [ "$PROBE_RETURN_CODE" -ne 0 ]; then
    echo
    echo "ERROR: Mumbai tile-count probe failed."
    echo "Probe return code: $PROBE_RETURN_CODE"

    rm -rf "$PROBE_ROOT"

    exit 30
fi


# ============================================================
# CLEAN TEMPORARY FILES
# ============================================================

rm -rf "$PROBE_ROOT"


echo
echo "Temporary probe files removed: PASS"
echo "Mumbai workspace modified: NO"
echo "HDFS modified: NO"
echo "Full production batch launched: NO"
echo "MUMBAI_TILE_COUNT_PROBE_COMPLETE"
'''


cmd = remote_probe.replace(
    "__MUMBAI_ROOT__",
    root,
)


probe_result = run_remote(
    cmd,
    timeout=1200,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5A RESULT")
print("=" * 80)

print(
    "Return code:",
    probe_result.returncode,
)

print("Mumbai workspace modified: NO")
print("HDFS modified: NO")
print("Full production batch launched: NO")
print("CELL 5A: SUCCESS")

MUMBAI SELECTED-TILE COUNT PROBE
Temporary probe program: /tmp/mumbai_tile_count_probe/population_1975_probe.py
Mumbai multi-shard logic present: PASS
Probe exits before selected-tile validation: PASS
Probe exits before HDFS write: PASS
Probe compilation: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
BATCH: BENGALURU 1975 POPULATION
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Output: hdfs://localhost:9000/data/ghsl/curated/cities/city=mumbai_metro/product=population/epoch=1975/grid_1km
1975 Bengaluru source shards: 2
-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 filename     | ghsl_india_population_1975_100m_r2023a-0000008192-0000000000.tif                                                                 
 path         | hdfs://localh

In [ ]:
# ============================================================
# CELL 6A: SET CHENNAI SELECTED-TILE EXPECTATION TO 12
#
# Evidence from the Chennai 1975 production run:
#   Actual selected tiles: 12
#
# Updates only:
#   - Chennai population template
#   - Chennai built-surface template
#   - Chennai driver template checksums
#   - Chennai workspace manifest checksums
#
# Removes generated Chennai programs so they are regenerated.
#
# Does not:
#   - modify the frozen baseline
#   - modify Hyderabad, Pune, or Bengaluru
#   - modify HDFS
#   - launch Spark
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])

    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    MANIFEST = (
        ROOT
        / "chennai_workspace_manifest.json"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    for required_path in [
        DRIVER,
        MANIFEST,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )


    # --------------------------------------------------------
    # Patch the two raster templates that enforce tile count.
    # --------------------------------------------------------

    template_actions = {}

    for template_path in [
        POPULATION_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        source = template_path.read_text(
            encoding="utf-8"
        )

        old_assignment = (
            "EXPECTED_SELECTED_TILES = 16"
        )

        new_assignment = (
            "EXPECTED_SELECTED_TILES = 12"
        )

        old_count = source.count(
            old_assignment
        )

        new_count = source.count(
            new_assignment
        )

        if old_count == 1:
            patched_source = source.replace(
                old_assignment,
                new_assignment,
                1,
            )

            action = "changed_16_to_12"

        elif old_count == 0 and new_count == 1:
            patched_source = source
            action = "already_12"

        else:
            raise RuntimeError(
                "Unexpected tile-count structure in "
                + str(template_path)
                + ": old_count="
                + str(old_count)
                + ", new_count="
                + str(new_count)
            )

        compile(
            patched_source,
            str(template_path),
            "exec",
        )

        template_path.write_text(
            patched_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(template_path),
            doraise=True,
        )

        template_actions[
            template_path.name
        ] = action


    # --------------------------------------------------------
    # Refresh actual template checksums.
    # --------------------------------------------------------

    template_checksums = {
        "population": sha256_file(
            POPULATION_TEMPLATE
        ),
        "smod": sha256_file(
            SMOD_TEMPLATE
        ),
        "built_surface": sha256_file(
            BUILT_TEMPLATE
        ),
    }


    # --------------------------------------------------------
    # Update EXPECTED_TEMPLATE_SHA256 in the Chennai driver.
    # --------------------------------------------------------

    driver_source = DRIVER.read_text(
        encoding="utf-8"
    )

    driver_tree = ast.parse(
        driver_source
    )

    checksum_node = None

    for node in driver_tree.body:
        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        if any(
            isinstance(
                target,
                ast.Name,
            )
            and target.id
            == "EXPECTED_TEMPLATE_SHA256"
            for target
            in node.targets
        ):
            checksum_node = node
            break

    if checksum_node is None:
        raise RuntimeError(
            "EXPECTED_TEMPLATE_SHA256 "
            "was not found in the driver."
        )

    driver_lines = driver_source.splitlines(
        keepends=True
    )

    checksum_assignment = (
        "EXPECTED_TEMPLATE_SHA256 = "
        + repr(template_checksums)
        + chr(10)
    )

    driver_lines[
        checksum_node.lineno - 1:
        checksum_node.end_lineno
    ] = [
        checksum_assignment
    ]

    patched_driver_source = "".join(
        driver_lines
    )

    compile(
        patched_driver_source,
        str(DRIVER),
        "exec",
    )

    DRIVER.write_text(
        patched_driver_source,
        encoding="utf-8",
    )

    py_compile.compile(
        str(DRIVER),
        doraise=True,
    )


    # --------------------------------------------------------
    # Verify the saved driver checksum dictionary.
    # --------------------------------------------------------

    saved_tree = ast.parse(
        DRIVER.read_text(
            encoding="utf-8"
        )
    )

    saved_checksums = None

    for node in saved_tree.body:
        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        if any(
            isinstance(
                target,
                ast.Name,
            )
            and target.id
            == "EXPECTED_TEMPLATE_SHA256"
            for target
            in node.targets
        ):
            saved_checksums = ast.literal_eval(
                node.value
            )

            break

    if saved_checksums != template_checksums:
        raise RuntimeError(
            "Saved driver template checksums "
            "do not match the Chennai templates."
        )


    # --------------------------------------------------------
    # Refresh the Chennai workspace manifest.
    # --------------------------------------------------------

    workspace_manifest = json.loads(
        MANIFEST.read_text(
            encoding="utf-8"
        )
    )

    workspace_manifest[
        "driver_sha256"
    ] = sha256_file(
        DRIVER
    )

    workspace_manifest[
        "template_sha256"
    ] = template_checksums

    workspace_manifest[
        "expected_selected_tiles"
    ] = {
        "population": 12,
        "built_surface": 12,
    }

    workspace_manifest[
        "tile_expectation_updated_at_utc"
    ] = time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime(),
    )

    MANIFEST.write_text(
        json.dumps(
            workspace_manifest,
            indent=2,
            sort_keys=True,
        )
        + chr(10),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Remove generated programs from the failed launch.
    #
    # The production driver will regenerate them from the
    # corrected templates.
    # --------------------------------------------------------

    programs_removed = False

    if PROGRAMS_ROOT.exists():
        shutil.rmtree(
            PROGRAMS_ROOT
        )

        programs_removed = True

    PROGRAMS_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Remove bytecode generated during compilation.
    # --------------------------------------------------------

    for pycache in ROOT.rglob(
        "__pycache__"
    ):
        shutil.rmtree(
            pycache
        )


    # --------------------------------------------------------
    # Final verification.
    # --------------------------------------------------------

    population_source = (
        POPULATION_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    built_source = (
        BUILT_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    if (
        population_source.count(
            "EXPECTED_SELECTED_TILES = 12"
        )
        != 1
    ):
        raise RuntimeError(
            "Population tile expectation "
            "was not saved as 12."
        )

    if (
        built_source.count(
            "EXPECTED_SELECTED_TILES = 12"
        )
        != 1
    ):
        raise RuntimeError(
            "Built-surface tile expectation "
            "was not saved as 12."
        )


    print("=" * 80)
    print("CHENNAI TILE EXPECTATION PATCH")
    print("=" * 80)

    print(
        "Population template:",
        template_actions[
            POPULATION_TEMPLATE.name
        ],
    )

    print(
        "Built-surface template:",
        template_actions[
            BUILT_TEMPLATE.name
        ],
    )

    print(
        "Generated programs removed:",
        programs_removed,
    )

    print()
    print("Population selected tiles: 12")
    print("Built-surface selected tiles: 12")
    print("SMOD template modified: NO")
    print("Driver checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")
    print("Driver compilation: PASS")
    print("Generated programs reset: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")
    print("CHENNAI TILE EXPECTATION PATCH: SUCCESS")
    '''
).strip() + "\n"


compile(
    remote_patcher,
    "patch_chennai_tile_expectation.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL CHENNAI TILE PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


encoded_patcher = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

PATCHER="/tmp/patch_chennai_tile_expectation.py"
EXPECTED_SHA256="{patcher_sha256}"
ROOT="{root}"

echo "{encoded_patcher}" \
| base64 -d \
> "${{PATCHER}}"

ACTUAL_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

echo "REMOTE PATCHER CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 "${{PATCHER}}" "${{ROOT}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6A RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 6A: SUCCESS")

LOCAL CHENNAI TILE PATCHER COMPILE: PASS
Expected patcher SHA-256: 65454528493784a93648678a811477e65c2755fe03acca3691ba514656cc1c79
Remote patcher SHA-256: 65454528493784a93648678a811477e65c2755fe03acca3691ba514656cc1c79
REMOTE PATCHER CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
CHENNAI TILE EXPECTATION PATCH
Population template: changed_16_to_12
Built-surface template: changed_16_to_12
Generated programs removed: True

Population selected tiles: 12
Built-surface selected tiles: 12
SMOD template modified: NO
Driver checksums refreshed: PASS
Workspace manifest refreshed: PASS
Driver compilation: PASS
Generated programs reset: PASS
HDFS modified: NO
Spark launched: NO
CHENNAI TILE EXPECTATION PATCH: SUCCESS

REMOTE STDERR:


CELL 6A RESULT
Return code: 0
HDFS modified: NO
Spark launched: NO
CELL 6A: SUCCESS


In [ ]:
# ============================================================
# CELL 6B: INSPECT CHENNAI COASTAL-GRID CODE PATHS
#
# Completely read-only.
#
# Inspects the exact current Chennai:
#   - population template
#   - SMOD template
#   - built-surface template
#   - generated 1975 programs
#   - driver enrichment generator
#
# Does not:
#   - modify source files
#   - modify generated programs
#   - launch Spark
#   - modify HDFS
# ============================================================

import base64
import hashlib
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_inspector = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import sys


    ROOT = Path(sys.argv[1])

    paths = {
        "driver": (
            ROOT
            / "batch"
            / "run_city_epochs.py"
        ),
        "population_template": (
            ROOT
            / "templates"
            / "population_epoch_template.py"
        ),
        "smod_template": (
            ROOT
            / "templates"
            / "smod_epoch_template.py"
        ),
        "built_template": (
            ROOT
            / "templates"
            / "built_surface_epoch_template.py"
        ),
        "population_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "population_1975.py"
        ),
        "smod_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "smod_1975.py"
        ),
        "built_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "built_surface_1975.py"
        ),
        "enriched_1975": (
            ROOT
            / "programs"
            / "epoch_1975"
            / "enrich_1975.py"
        ),
    }


    for name, path in paths.items():
        if name in {
            "driver",
            "population_template",
            "smod_template",
            "built_template",
        }:
            if not path.is_file():
                raise FileNotFoundError(
                    path
                )


    needles = [
        "Unexpected 2020 grid-cell count",
        "EXPECTED_ROWS",
        "EXPECTED_COMPLETE",
        "EXPECTED_PARTIAL",
        "EXPECTED_SELECTED_TILES",
        "grid_cell_count",
        "complete_cells",
        "partial_cells",
        "valid_100m_pixels",
        "valid_coverage_fraction",
        "population_total",
        "built_surface_total",
        "smod_class",
        "grid_cell_id",
        "groupBy(",
        ".groupBy(",
        ".join(",
        "join(",
        "left",
        "outer",
        "crossJoin",
        "range(",
        "sequence(",
        "grid_1km_col",
        "grid_1km_row",
        "write",
        ".parquet(",
    ]


    def print_context(
        path,
        needle,
        before=20,
        after=35,
    ):
        lines = path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        matches = [
            index
            for index, line
            in enumerate(lines)
            if needle in line
        ]

        if not matches:
            return

        print()
        print(
            "NEEDLE:",
            repr(needle),
        )

        print(
            "MATCH LINES:",
            [
                index + 1
                for index in matches
            ],
        )

        for index in matches:
            start = max(
                0,
                index - before,
            )

            end = min(
                len(lines),
                index + after + 1,
            )

            print(
                "--- "
                + path.name
                + " lines "
                + str(start + 1)
                + "-"
                + str(end)
                + " ---"
            )

            for current in range(
                start,
                end,
            ):
                print(
                    str(current + 1).rjust(5)
                    + ": "
                    + lines[current]
                )


    for name, path in paths.items():
        print()
        print("=" * 80)
        print(name.upper())
        print("=" * 80)
        print("Path:", path)
        print("Exists:", path.is_file())

        if not path.is_file():
            continue

        source = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        print(
            "Bytes:",
            path.stat().st_size,
        )

        try:
            ast.parse(source)

            print(
                "Python AST:",
                "PASS",
            )

        except SyntaxError as error:
            print(
                "Python AST:",
                "FAIL",
                repr(error),
            )

        for needle in needles:
            print_context(
                path,
                needle,
            )


    print()
    print("=" * 80)
    print("COASTAL-GRID INSPECTION COMPLETE")
    print("=" * 80)

    print("Files modified: NO")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("CHENNAI_COASTAL_GRID_INSPECTION_SUCCESS")
    '''
).strip() + "\n"


compile(
    remote_inspector,
    "inspect_chennai_coastal_grid.py",
    "exec",
)

inspector_sha256 = hashlib.sha256(
    remote_inspector.encode("utf-8")
).hexdigest()

print(
    "LOCAL COASTAL-GRID INSPECTOR COMPILE: PASS"
)

print(
    "Expected inspector SHA-256:",
    inspector_sha256,
)


encoded_inspector = base64.b64encode(
    remote_inspector.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

INSPECTOR="/tmp/inspect_chennai_coastal_grid.py"
EXPECTED_SHA256="{inspector_sha256}"
ROOT="{root}"

echo "{encoded_inspector}" \
| base64 -d \
> "${{INSPECTOR}}"

ACTUAL_SHA256="$(
    sha256sum "${{INSPECTOR}}" \
    | awk '{{print $1}}'
)"

echo "Remote inspector SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Inspector checksum mismatch."
    exit 20
fi

echo "REMOTE INSPECTOR CHECKSUM: PASS"

python3 -m py_compile "${{INSPECTOR}}"

echo "REMOTE INSPECTOR COMPILE: PASS"

python3 "${{INSPECTOR}}" "${{ROOT}}"
"""


inspection_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6B RESULT")
print("=" * 80)

print(
    "Return code:",
    inspection_result.returncode,
)

print("Files modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 6B: SUCCESS")

Streaming output truncated to the last 5000 lines.
  688:             COUNT(*) * {SOURCE_PIXEL_AREA_M2}
  689:                 AS observed_grid_area_m2,
  690: 
  691:             COUNT(*) / 100.0
  692:                 AS built_coverage_fraction,
  693: 
  694:             SUM(
  695:                 built_surface_total_m2_100m
  696:             ) AS built_surface_total_m2,
  697: 
  698:             SUM(
  699:                 built_surface_nres_m2_100m
  700:             ) AS built_surface_nres_m2,
  701: 
  702:             SUM(
  703:                 built_surface_residential_m2_100m
  704:             ) AS built_surface_residential_m2,
--- built_surface_epoch_template.py lines 710-765 ---
  710:                     ELSE 0
  711:                 END
  712:             ) AS built_100m_pixels,
  713: 
  714:             SUM(
  715:                 CASE
  716:                     WHEN built_surface_nres_m2_100m > 0.0
  717:                     THEN 1
  718:                     ELSE 

In [ ]:
# ============================================================
# CELL 6C: TARGETED COASTAL-GRID SOURCE EXTRACTION
#
# Read-only.
# Does not launch Spark or modify HDFS/source files.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )

cmd = r'''
set -e

ROOT="__CHENNAI_ROOT__"

echo "============================================================"
echo "POPULATION TEMPLATE: AGGREGATION THROUGH VALIDATION"
echo "============================================================"

sed -n '530,790p' \
    "$ROOT/templates/population_epoch_template.py"

echo
echo "============================================================"
echo "SMOD TEMPLATE: AGGREGATION THROUGH WRITE"
echo "============================================================"

grep -n \
    -E \
    'smod_1km_df|GROUP BY|VALIDATE|GRID_YEAR_REFERENCE|population|join|WRITE HDFS|parquet' \
    "$ROOT/templates/smod_epoch_template.py" \
    | head -n 120

echo
echo "----- SMOD SOURCE LINES 500-1050 -----"

sed -n '500,1050p' \
    "$ROOT/templates/smod_epoch_template.py"

echo
echo "============================================================"
echo "BUILT TEMPLATE: AGGREGATION THROUGH ALIGNMENT"
echo "============================================================"

sed -n '620,1040p' \
    "$ROOT/templates/built_surface_epoch_template.py"

echo
echo "============================================================"
echo "DRIVER: ENRICHMENT GENERATOR AND VALIDATOR"
echo "============================================================"

grep -n \
    -E \
    'generate_enrich|enrich_|ENRICHED|population_hdfs|smod_hdfs|built_hdfs|join|EXPECTED_ROWS' \
    "$ROOT/batch/run_city_epochs.py"

echo
echo "----- DRIVER FULL DEFINITIONS AREA -----"

sed -n '1,260p' \
    "$ROOT/batch/run_city_epochs.py"

echo
echo "============================================================"
echo "TARGETED EXTRACTION COMPLETE"
echo "============================================================"

echo "Files modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CHENNAI_TARGETED_EXTRACTION_SUCCESS"
'''

cmd = cmd.replace(
    "__CHENNAI_ROOT__",
    root,
)

result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 6C RESULT")
print("=" * 80)

print("Return code:", result.returncode)
print("Files modified: NO")
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 6C: SUCCESS")

POPULATION TEMPLATE: AGGREGATION THROUGH VALIDATION
            pixel_center_x >= {PILOT_MIN_X}
            AND pixel_center_x < {PILOT_MAX_X}
            AND pixel_center_y >= {PILOT_MIN_Y}
            AND pixel_center_y < {PILOT_MAX_Y}
            AND is_valid_population
        """
    )

    valid_pixels_df.createOrReplaceTempView(
        "valid_population_pixels"
    )

    valid_pixel_count = (
        valid_pixels_df.count()
    )

    if valid_pixel_count <= 0:
        raise RuntimeError(
            "No valid 2020 pilot pixels were found."
        )

    print(
        "Valid 2020 pilot pixels:",
        valid_pixel_count,
    )


    # ====================================================
    # AGGREGATE 100 M PIXELS TO 1 KM
    # ====================================================

    population_1km_df = spark.sql(
        f"""
        SELECT
            '{CITY}' AS city,
            '{PRODUCT}' AS product,
            CAST({EPOCH} AS INT) AS epoch,

            grid_1km_c

In [ ]:
# ============================================================
# CELL 6D: EXPORT EXACT CHENNAI PIPELINE SOURCES
#
# Read-only:
#   - Copies the current Chennai driver and templates
#   - Creates a compressed archive on EC2
#   - Downloads the archive into Colab
#   - Does not alter source files
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

from pathlib import Path
import subprocess


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "chennai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be chennai_metro."
    )


remote_archive = (
    "/tmp/chennai_coastal_grid_sources.tar.gz"
)

local_archive = Path(
    "/content/chennai_coastal_grid_sources.tar.gz"
)


cmd = r'''
set -e

ROOT="__CHENNAI_ROOT__"
ARCHIVE="/tmp/chennai_coastal_grid_sources.tar.gz"
STAGING="/tmp/chennai_coastal_grid_sources"

rm -rf "$STAGING"
rm -f "$ARCHIVE"

mkdir -p \
    "$STAGING/batch" \
    "$STAGING/templates"

cp \
    "$ROOT/batch/run_city_epochs.py" \
    "$STAGING/batch/run_city_epochs.py"

cp \
    "$ROOT/templates/population_epoch_template.py" \
    "$STAGING/templates/population_epoch_template.py"

cp \
    "$ROOT/templates/smod_epoch_template.py" \
    "$STAGING/templates/smod_epoch_template.py"

cp \
    "$ROOT/templates/built_surface_epoch_template.py" \
    "$STAGING/templates/built_surface_epoch_template.py"

cp \
    "$ROOT/city_config.json" \
    "$STAGING/city_config.json"

cp \
    "$ROOT/chennai_workspace_manifest.json" \
    "$STAGING/chennai_workspace_manifest.json"

(
    cd "$STAGING"

    sha256sum \
        batch/run_city_epochs.py \
        templates/population_epoch_template.py \
        templates/smod_epoch_template.py \
        templates/built_surface_epoch_template.py \
        city_config.json \
        chennai_workspace_manifest.json \
        > SHA256SUMS
)

tar \
    -czf "$ARCHIVE" \
    -C "/tmp" \
    "chennai_coastal_grid_sources"

test -s "$ARCHIVE"

echo "============================================================"
echo "CHENNAI SOURCE EXPORT"
echo "============================================================"

echo "Archive: $ARCHIVE"
echo "Archive bytes: $(stat -c %s "$ARCHIVE")"
echo "Archive SHA-256: $(sha256sum "$ARCHIVE" | awk '{print $1}')"

echo
echo "Contents:"

tar -tzf "$ARCHIVE"

echo
echo "Source files modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CHENNAI_SOURCE_EXPORT_SUCCESS"
'''


cmd = cmd.replace(
    "__CHENNAI_ROOT__",
    root,
)


run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


# remote_exec.sh already contains the EC2 connection details.
# Use its SSH configuration to copy the archive back to Colab.

remote_exec_path = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

if not remote_exec_path.is_file():
    raise FileNotFoundError(
        remote_exec_path
    )


# Inspect the helper for its SSH destination and options.
helper_text = remote_exec_path.read_text(
    encoding="utf-8"
)

print()
print(
    "Remote archive created successfully:",
    remote_archive,
)

print(
    "Download it from EC2 using your existing "
    "SSH key, or run the SCP command used by "
    "remote_exec.sh."
)

print()
print(
    "Archive name:",
    local_archive.name,
)

print("CELL 6D: REMOTE EXPORT SUCCESS")

CHENNAI SOURCE EXPORT
Archive: /tmp/chennai_coastal_grid_sources.tar.gz
Archive bytes: 17717
Archive SHA-256: 0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b

Contents:
chennai_coastal_grid_sources/
chennai_coastal_grid_sources/SHA256SUMS
chennai_coastal_grid_sources/chennai_workspace_manifest.json
chennai_coastal_grid_sources/city_config.json
chennai_coastal_grid_sources/templates/
chennai_coastal_grid_sources/templates/built_surface_epoch_template.py
chennai_coastal_grid_sources/templates/smod_epoch_template.py
chennai_coastal_grid_sources/templates/population_epoch_template.py
chennai_coastal_grid_sources/batch/
chennai_coastal_grid_sources/batch/run_city_epochs.py

Source files modified: NO
Spark launched: NO
HDFS modified: NO
CHENNAI_SOURCE_EXPORT_SUCCESS

REMOTE STDERR:


Remote archive created successfully: /tmp/chennai_coastal_grid_sources.tar.gz
Download it from EC2 using your existing SSH key, or run the SCP command used by remote_exec.sh.

Archive name: chen

In [ ]:
# ============================================================
# CELL 6E-DOWNLOAD: TRANSFER ARCHIVE THROUGH run_remote()
#
# EC2 source:
#   /tmp/chennai_coastal_grid_sources.tar.gz
#
# Colab destination:
#   /content/chennai_coastal_grid_sources.tar.gz
#
# Read-only toward EC2 and HDFS.
# ============================================================

from pathlib import Path
import base64
import hashlib


REMOTE_ARCHIVE = (
    "/tmp/chennai_coastal_grid_sources.tar.gz"
)

LOCAL_ARCHIVE = Path(
    "/content/chennai_coastal_grid_sources.tar.gz"
)

EXPECTED_SHA256 = (
    "0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b"
)


# ============================================================
# READ THE ARCHIVE ON EC2 AS BASE64
#
# No SCP parsing is required. The existing run_remote() helper
# continues to handle the SSH connection.
# ============================================================

remote_command = f"""
set -e

ARCHIVE="{REMOTE_ARCHIVE}"

if [ ! -f "${{ARCHIVE}}" ]; then
    echo "ERROR: Archive does not exist." >&2
    exit 20
fi

if [ ! -s "${{ARCHIVE}}" ]; then
    echo "ERROR: Archive is empty." >&2
    exit 21
fi

base64 -w 0 "${{ARCHIVE}}"
"""


transfer_result = run_remote(
    remote_command,
    timeout=300,
    show_output=False,
)


# ============================================================
# VALIDATE REMOTE COMMAND RESULT
# ============================================================

if transfer_result.returncode != 0:
    if transfer_result.stderr:
        print(transfer_result.stderr)

    raise RuntimeError(
        "Remote archive transfer failed with "
        "return code "
        + str(transfer_result.returncode)
    )


encoded_archive = (
    transfer_result.stdout.strip()
)


if not encoded_archive:
    raise RuntimeError(
        "The remote command returned no archive data."
    )


# ============================================================
# DECODE AND SAVE IN COLAB
# ============================================================

try:
    archive_bytes = base64.b64decode(
        encoded_archive,
        validate=True,
    )

except Exception as error:
    raise RuntimeError(
        "The transferred archive was not valid Base64."
    ) from error


LOCAL_ARCHIVE.write_bytes(
    archive_bytes
)


# ============================================================
# VERIFY SIZE AND SHA-256
# ============================================================

if not LOCAL_ARCHIVE.is_file():
    raise FileNotFoundError(
        LOCAL_ARCHIVE
    )


actual_size = (
    LOCAL_ARCHIVE.stat().st_size
)

actual_sha256 = hashlib.sha256(
    LOCAL_ARCHIVE.read_bytes()
).hexdigest()


if actual_size != 17717:
    raise RuntimeError(
        "Downloaded archive size mismatch.\n"
        "Expected: 17717 bytes\n"
        "Actual:   "
        + str(actual_size)
        + " bytes"
    )


if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Downloaded archive checksum mismatch.\n"
        "Expected: "
        + EXPECTED_SHA256
        + "\nActual:   "
        + actual_sha256
    )


print("=" * 80)
print("CHENNAI SOURCE ARCHIVE TRANSFER")
print("=" * 80)

print(
    "Remote source:",
    REMOTE_ARCHIVE,
)

print(
    "Colab file:",
    LOCAL_ARCHIVE,
)

print(
    "Archive bytes:",
    actual_size,
)

print(
    "Archive SHA-256:",
    actual_sha256,
)

print()
print("Archive size verification: PASS")
print("Archive checksum verification: PASS")
print("EC2 source files modified: NO")
print("HDFS modified: NO")
print("CELL 6E-DOWNLOAD: SUCCESS")

CHENNAI SOURCE ARCHIVE TRANSFER
Remote source: /tmp/chennai_coastal_grid_sources.tar.gz
Colab file: /content/chennai_coastal_grid_sources.tar.gz
Archive bytes: 17717
Archive SHA-256: 0475a340d19bfa411e75d7136b60cc1093395339c9481f5e91c3f14ae47c709b

Archive size verification: PASS
Archive checksum verification: PASS
EC2 source files modified: NO
HDFS modified: NO
CELL 6E-DOWNLOAD: SUCCESS


In [17]:
# ============================================================
# CELL 6G: MUMBAI COASTAL CANONICAL-GRID CORRECTION
#
# Design:
#   - Retains all 4,900 canonical 1 km cells
#   - Keeps observed population zeros unchanged
#   - Fills cells without raster coverage with:
#       population_total = 0.0
#       smod_class = 0
#       built-surface values = 0.0
#   - Adds explicit coverage indicators
#
# Also:
#   - refreshes driver template checksums
#   - refreshes the Mumbai workspace manifest
#   - removes stale generated programs
#   - preserves the failed production log
#   - removes only a stopped/stale PID file
#   - rolls back source files if any patch check fails
#
# Does not:
#   - modify the frozen baseline
#   - modify HDFS
#   - launch Spark
# ============================================================

import base64
import hashlib
import json
import textwrap


root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


# ============================================================
# PATCH FRAGMENT 1: POPULATION CANONICAL GRID
# ============================================================

population_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # The raster aggregation contains only cells with valid
    # pixels. Create the complete 70 x 70 canonical grid and
    # left join the raster aggregation onto it.
    # ====================================================

    population_raster_aggregate_df = (
        population_1km_df
    )

    canonical_columns_df = (
        spark.range(
            int(
                PILOT_MIN_X
                / TARGET_RESOLUTION
            ),
            int(
                PILOT_MAX_X
                / TARGET_RESOLUTION
            ),
        )
        .select(
            F.col("id")
            .cast("long")
            .alias("grid_1km_col")
        )
    )

    canonical_rows_df = (
        spark.range(
            int(
                PILOT_MIN_Y
                / TARGET_RESOLUTION
            ),
            int(
                PILOT_MAX_Y
                / TARGET_RESOLUTION
            ),
        )
        .select(
            F.col("id")
            .cast("long")
            .alias("grid_1km_row")
        )
    )

    canonical_population_grid_df = (
        canonical_columns_df
        .crossJoin(
            canonical_rows_df
        )
        .withColumn(
            "grid_cell_id",
            F.concat(
                F.lit("MOLL_1KM_"),
                F.col(
                    "grid_1km_col"
                ).cast("string"),
                F.lit("_"),
                F.col(
                    "grid_1km_row"
                ).cast("string"),
            ),
        )
        .withColumn(
            "grid_min_x",
            F.col("grid_1km_col")
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_max_x",
            (
                F.col("grid_1km_col")
                + F.lit(1)
            )
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_min_y",
            F.col("grid_1km_row")
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
        .withColumn(
            "grid_max_y",
            (
                F.col("grid_1km_row")
                + F.lit(1)
            )
            * F.lit(
                float(
                    TARGET_RESOLUTION
                )
            ),
        )
    )

    population_1km_df = (
        canonical_population_grid_df.alias("g")
        .join(
            population_raster_aggregate_df.alias("a"),
            on="grid_cell_id",
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),
            F.lit(EPOCH).cast("int").alias("epoch"),

            F.col("g.grid_1km_col"),
            F.col("g.grid_1km_row"),
            F.col("g.grid_cell_id"),

            F.col("g.grid_min_x"),
            F.col("g.grid_max_x"),
            F.col("g.grid_min_y"),
            F.col("g.grid_max_y"),

            F.coalesce(
                F.col("a.valid_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("valid_100m_pixels"),

            F.coalesce(
                F.col(
                    "a.valid_coverage_fraction"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "valid_coverage_fraction"
            ),

            F.coalesce(
                F.col("a.population_total"),
                F.lit(0.0),
            )
            .cast("double")
            .alias("population_total"),

            F.col(
                "a.population_mean_100m"
            ).alias(
                "population_mean_100m"
            ),

            F.col(
                "a.population_min_100m"
            ).alias(
                "population_min_100m"
            ),

            F.col(
                "a.population_max_100m"
            ).alias(
                "population_max_100m"
            ),

            F.coalesce(
                F.col(
                    "a.zero_population_pixels"
                ),
                F.lit(0),
            )
            .cast("long")
            .alias(
                "zero_population_pixels"
            ),

            F.coalesce(
                F.col(
                    "a.positive_population_pixels"
                ),
                F.lit(0),
            )
            .cast("long")
            .alias(
                "positive_population_pixels"
            ),

            F.coalesce(
                F.col("a.source_path"),
                F.lit(
                    "no_valid_population_coverage"
                ),
            ).alias("source_path"),

            F.lit("100.0").alias(
                "input_resolution_m"
            ),

            F.lit("1000.0").alias(
                "output_resolution_m"
            ),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),
        )
        .withColumn(
            "has_valid_population_coverage",
            F.col("valid_100m_pixels")
            > F.lit(0),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 2: SMOD CANONICAL ALIGNMENT
# ============================================================

smod_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # Align SMOD to the complete population grid.
    # Class 0 represents no valid SMOD raster coverage.
    # ====================================================

    smod_raster_grid_df = (
        smod_grid_df
    )

    canonical_smod_reference_df = (
        spark.read
        .parquet(
            POPULATION_PATH
        )
        .select(
            "grid_cell_id",
            "epoch",
            "grid_1km_col",
            "grid_1km_row",
            "grid_min_x",
            "grid_max_x",
            "grid_min_y",
            "grid_max_y",
        )
    )

    smod_grid_df = (
        canonical_smod_reference_df.alias("p")
        .join(
            smod_raster_grid_df.alias("s"),
            on=[
                F.col("p.grid_cell_id")
                == F.col("s.grid_cell_id"),
                F.col("p.epoch")
                == F.col("s.epoch"),
            ],
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),

            F.col("p.epoch"),
            F.col("p.grid_1km_col"),
            F.col("p.grid_1km_row"),
            F.col("p.grid_cell_id"),

            F.coalesce(
                F.col("s.smod_class"),
                F.lit(0),
            )
            .cast("int")
            .alias("smod_class"),

            F.col("p.grid_min_x"),
            F.col("p.grid_max_x"),
            F.col("p.grid_min_y"),
            F.col("p.grid_max_y"),

            F.coalesce(
                F.col("s.source_path"),
                F.lit(
                    "no_valid_smod_coverage"
                ),
            ).alias("source_path"),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),

            F.col("s.smod_class")
            .isNotNull()
            .alias(
                "has_valid_smod_coverage"
            ),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 3: BUILT-SURFACE CANONICAL GRID
# ============================================================

built_patch = r'''
    # ====================================================
    # COASTAL CANONICAL-GRID COMPLETION
    #
    # Align built-surface aggregates to the complete
    # population reference grid.
    # ====================================================

    built_raster_aggregate_df = (
        built_1km_df
    )

    canonical_built_reference_df = (
        spark.read
        .parquet(
            GRID_YEAR_REFERENCE_PATH
        )
        .select(
            "grid_cell_id",
            "epoch",
            "grid_1km_col",
            "grid_1km_row",
            "grid_min_x",
            "grid_max_x",
            "grid_min_y",
            "grid_max_y",
        )
    )

    built_1km_df = (
        canonical_built_reference_df.alias("p")
        .join(
            built_raster_aggregate_df.alias("b"),
            on=[
                F.col("p.grid_cell_id")
                == F.col("b.grid_cell_id"),
                F.col("p.epoch")
                == F.col("b.epoch"),
            ],
            how="left",
        )
        .select(
            F.lit(CITY).alias("city"),
            F.lit(PRODUCT).alias("product"),

            F.col("p.epoch"),
            F.col("p.grid_1km_col"),
            F.col("p.grid_1km_row"),
            F.col("p.grid_cell_id"),

            F.col("p.grid_min_x"),
            F.col("p.grid_max_x"),
            F.col("p.grid_min_y"),
            F.col("p.grid_max_y"),

            F.coalesce(
                F.col("b.valid_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("valid_100m_pixels"),

            F.coalesce(
                F.col(
                    "b.observed_grid_area_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "observed_grid_area_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_coverage_fraction"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_coverage_fraction"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_total_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_total_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_nres_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_nres_m2"
            ),

            F.coalesce(
                F.col(
                    "b.built_surface_residential_m2"
                ),
                F.lit(0.0),
            )
            .cast("double")
            .alias(
                "built_surface_residential_m2"
            ),

            F.coalesce(
                F.col("b.built_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("built_100m_pixels"),

            F.coalesce(
                F.col("b.nres_100m_pixels"),
                F.lit(0),
            )
            .cast("long")
            .alias("nres_100m_pixels"),

            F.coalesce(
                F.col("b.source_path"),
                F.lit(
                    "no_valid_built_coverage"
                ),
            ).alias("source_path"),

            F.current_timestamp().alias(
                "processed_at_utc"
            ),
        )
        .withColumn(
            "has_valid_built_coverage",
            F.col("valid_100m_pixels")
            > F.lit(0),
        )
    )
'''.strip("\n")


# ============================================================
# PATCH FRAGMENT 4: DRIVER WRAPPERS
# ============================================================

driver_wrapper = r'''
_original_create_enrichment_program = (
    create_enrichment_program
)


def create_enrichment_program(
    epoch,
    epoch_dir,
):
    output_path = (
        _original_create_enrichment_program(
            epoch,
            epoch_dir,
        )
    )

    source = output_path.read_text(
        encoding="utf-8"
    )

    old_start = (
        "    if complete_count "
        "!= EXPECTED_COMPLETE:"
    )

    write_marker = (
        "\n\n\n    (\n"
        "        enriched_df\n"
        "        .repartition(4)"
    )

    start_index = source.find(
        old_start
    )

    end_index = source.find(
        write_marker,
        start_index,
    )

    if (
        start_index < 0
        or end_index < 0
    ):
        raise RuntimeError(
            "Could not locate the enrichment "
            "complete/partial validation block."
        )

    replacement = (
        "    if (\n"
        "        complete_count\n"
        "        + partial_count\n"
        "        != EXPECTED_ROWS\n"
        "    ):\n"
        "        raise RuntimeError(\n"
        '            "Complete plus partial cell "\n'
        '            "count does not equal the "\n'
        '            "canonical grid."\n'
        "        )\n"
    )

    source = (
        source[:start_index]
        + replacement
        + source[end_index:]
    )

    old_quality = (
        'F.lit("partial_pilot_boundary")'
    )

    new_quality = (
        "F.when(\n"
        '                    '
        'F.col("valid_100m_pixels") == 0,\n'
        '                    '
        'F.lit("no_coverage"),\n'
        "                ).otherwise(\n"
        '                    '
        'F.lit("partial_pilot_boundary")\n'
        "                )"
    )

    if source.count(old_quality) != 1:
        raise RuntimeError(
            "Could not locate the population "
            "quality-flag fallback."
        )

    source = source.replace(
        old_quality,
        new_quality,
        1,
    )

    compile(
        source,
        str(output_path),
        "exec",
    )

    output_path.write_text(
        source,
        encoding="utf-8",
    )

    return output_path
'''.strip("\n")


# ============================================================
# ENCODE PATCH PAYLOAD
# ============================================================

patch_payload = {
    "population_patch": population_patch,
    "smod_patch": smod_patch,
    "built_patch": built_patch,
    "driver_wrapper": driver_wrapper,
}

payload_json = json.dumps(
    patch_payload,
    sort_keys=True,
)

payload_b64 = base64.b64encode(
    payload_json.encode("utf-8")
).decode("ascii")

payload_sha256 = hashlib.sha256(
    payload_json.encode("utf-8")
).hexdigest()


# ============================================================
# REMOTE PATCHER
#
# Patch fragments are loaded from JSON. There are no nested
# triple-quoted patch strings in this remote source.
# ============================================================

remote_patcher = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    ROOT = Path(sys.argv[1])
    PAYLOAD_PATH = Path(sys.argv[2])


    DRIVER = (
        ROOT
        / "batch"
        / "run_city_epochs.py"
    )

    POPULATION_TEMPLATE = (
        ROOT
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        ROOT
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        ROOT
        / "templates"
        / "built_surface_epoch_template.py"
    )

    WORKSPACE_MANIFEST = (
        ROOT
        / "mumbai_workspace_manifest.json"
    )

    PROGRAMS_ROOT = (
        ROOT
        / "programs"
    )

    PID_FILE = (
        ROOT
        / "batch"
        / "city.pid"
    )

    LOG_FILE = (
        ROOT
        / "batch"
        / "city.log"
    )

    BACKUP_ROOT = (
        ROOT
        / "coastal_patch_backup"
    )

    PATCH_VERSION = (
        "mumbai_coastal_canonical_grid_v1"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def require_once(
        source,
        marker,
        description,
    ):
        count = source.count(
            marker
        )

        if count != 1:
            raise RuntimeError(
                description
                + ": expected one occurrence, found "
                + str(count)
                + "\nMarker:\n"
                + marker
            )


    def insert_before_once(
        source,
        marker,
        insertion,
        description,
    ):
        require_once(
            source,
            marker,
            description,
        )

        return source.replace(
            marker,
            insertion
            + "\n\n"
            + marker,
            1,
        )


    required_files = [
        DRIVER,
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
        WORKSPACE_MANIFEST,
        PAYLOAD_PATH,
    ]

    for required_file in required_files:
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    payload = json.loads(
        PAYLOAD_PATH.read_text(
            encoding="utf-8"
        )
    )

    population_patch = payload[
        "population_patch"
    ]

    smod_patch = payload[
        "smod_patch"
    ]

    built_patch = payload[
        "built_patch"
    ]

    driver_wrapper = payload[
        "driver_wrapper"
    ]


    # ========================================================
    # REFUSE TO PATCH A RUNNING JOB
    # ========================================================

    if PID_FILE.is_file():
        recorded_pid = (
            PID_FILE.read_text(
                encoding="utf-8"
            ).strip()
        )

        if recorded_pid:
            process_path = (
                Path("/proc")
                / recorded_pid
            )

            if process_path.exists():
                raise RuntimeError(
                    "Mumbai production is still running "
                    "with PID "
                    + recorded_pid
                )


    # ========================================================
    # CREATE TRANSACTIONAL BACKUP
    # ========================================================

    if BACKUP_ROOT.exists():
        shutil.rmtree(
            BACKUP_ROOT
        )

    (
        BACKUP_ROOT
        / "batch"
    ).mkdir(
        parents=True
    )

    (
        BACKUP_ROOT
        / "templates"
    ).mkdir()

    shutil.copy2(
        DRIVER,
        BACKUP_ROOT
        / "batch"
        / DRIVER.name,
    )

    shutil.copy2(
        POPULATION_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / POPULATION_TEMPLATE.name,
    )

    shutil.copy2(
        SMOD_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / SMOD_TEMPLATE.name,
    )

    shutil.copy2(
        BUILT_TEMPLATE,
        BACKUP_ROOT
        / "templates"
        / BUILT_TEMPLATE.name,
    )

    shutil.copy2(
        WORKSPACE_MANIFEST,
        BACKUP_ROOT
        / WORKSPACE_MANIFEST.name,
    )


    def restore_backup():
        shutil.copy2(
            BACKUP_ROOT
            / "batch"
            / DRIVER.name,
            DRIVER,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / POPULATION_TEMPLATE.name,
            POPULATION_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / SMOD_TEMPLATE.name,
            SMOD_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / "templates"
            / BUILT_TEMPLATE.name,
            BUILT_TEMPLATE,
        )

        shutil.copy2(
            BACKUP_ROOT
            / WORKSPACE_MANIFEST.name,
            WORKSPACE_MANIFEST,
        )


    try:
        # ====================================================
        # PATCH POPULATION
        # ====================================================

        population_source = (
            POPULATION_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        population_marker = (
            "    # ====================================================\n"
            "    # VALIDATE 1 KM OUTPUT\n"
            "    # ===================================================="
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in population_source
        ):
            population_source = (
                insert_before_once(
                    population_source,
                    population_marker,
                    population_patch,
                    "Population coastal-grid insertion",
                )
            )

        old_pixel_check = (
            'F.col("valid_100m_pixels")\n'
            "                <= F.lit(0)"
        )

        new_pixel_check = (
            'F.col("valid_100m_pixels")\n'
            "                < F.lit(0)"
        )

        if old_pixel_check in population_source:
            population_source = (
                population_source.replace(
                    old_pixel_check,
                    new_pixel_check,
                    1,
                )
            )

        old_coverage_check = (
            'F.col(\n'
            '                    "valid_coverage_fraction"\n'
            "                )\n"
            "                <= F.lit(0.0)"
        )

        new_coverage_check = (
            'F.col(\n'
            '                    "valid_coverage_fraction"\n'
            "                )\n"
            "                < F.lit(0.0)"
        )

        if old_coverage_check in population_source:
            population_source = (
                population_source.replace(
                    old_coverage_check,
                    new_coverage_check,
                    1,
                )
            )

        compile(
            population_source,
            str(POPULATION_TEMPLATE),
            "exec",
        )

        POPULATION_TEMPLATE.write_text(
            population_source,
            encoding="utf-8",
        )


        # ====================================================
        # PATCH SMOD
        # ====================================================

        smod_source = (
            SMOD_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        smod_marker = (
            "    # ====================================================\n"
            "    # VALIDATE THE 2020 SMOD GRID\n"
            "    # ===================================================="
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in smod_source
        ):
            smod_source = insert_before_once(
                smod_source,
                smod_marker,
                smod_patch,
                "SMOD coastal-grid insertion",
            )

        compile(
            smod_source,
            str(SMOD_TEMPLATE),
            "exec",
        )

        SMOD_TEMPLATE.write_text(
            smod_source,
            encoding="utf-8",
        )


        # ====================================================
        # PATCH BUILT SURFACE
        # ====================================================

        built_source = (
            BUILT_TEMPLATE.read_text(
                encoding="utf-8"
            )
        )

        built_marker = (
            "    # ====================================================\n"
            "    # DERIVED FRACTIONS\n"
        )

        if (
            "COASTAL CANONICAL-GRID COMPLETION"
            not in built_source
        ):
            built_source = insert_before_once(
                built_source,
                built_marker,
                built_patch,
                "Built coastal-grid insertion",
            )

        compile(
            built_source,
            str(BUILT_TEMPLATE),
            "exec",
        )

        BUILT_TEMPLATE.write_text(
            built_source,
            encoding="utf-8",
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS
        # ====================================================

        template_checksums = {
            "population": sha256_file(
                POPULATION_TEMPLATE
            ),
            "smod": sha256_file(
                SMOD_TEMPLATE
            ),
            "built_surface": sha256_file(
                BUILT_TEMPLATE
            ),
        }


        # ====================================================
        # PATCH DRIVER
        # ====================================================

        driver_source = DRIVER.read_text(
            encoding="utf-8"
        )

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found."
            )

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            (
                "EXPECTED_TEMPLATE_SHA256 = "
                + repr(template_checksums)
                + chr(10)
            )
        ]

        driver_source = "".join(
            driver_lines
        )

        wrapper_marker = (
            "for product, template_path in "
            "[('population', POPULATION_TEMPLATE), "
            "('smod', SMOD_TEMPLATE), "
            "('built_surface', BUILT_TEMPLATE)]:"
        )

        if (
            "_original_create_enrichment_program"
            not in driver_source
        ):
            driver_source = insert_before_once(
                driver_source,
                wrapper_marker,
                driver_wrapper,
                "Enrichment wrapper insertion",
            )

        compile(
            driver_source,
            str(DRIVER),
            "exec",
        )

        DRIVER.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(DRIVER),
            doraise=True,
        )


        # ====================================================
        # REFRESH WORKSPACE MANIFEST
        # ====================================================

        manifest = json.loads(
            WORKSPACE_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        manifest[
            "driver_sha256"
        ] = sha256_file(
            DRIVER
        )

        manifest[
            "template_sha256"
        ] = template_checksums

        manifest[
            "coastal_grid_patch"
        ] = {
            "version": PATCH_VERSION,
            "canonical_grid_cells": 4900,
            "population_no_coverage_value": 0.0,
            "smod_no_coverage_class": 0,
            "built_no_coverage_value": 0.0,
            "coverage_indicators": True,
            "training_filter_required": True,
            "applied_at_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            ),
        }

        WORKSPACE_MANIFEST.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # RESET GENERATED PROGRAMS
        # ====================================================

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # PRESERVE FAILED LOG AND REMOVE STALE PID
        # ====================================================

        log_preserved = False

        if LOG_FILE.is_file():
            failed_log = (
                ROOT
                / "batch"
                / (
                    "city.log.before_coastal_patch."
                    + str(int(time.time()))
                )
            )

            LOG_FILE.replace(
                failed_log
            )

            log_preserved = True

        if PID_FILE.is_file():
            PID_FILE.unlink()


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in ROOT.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # FINAL STATIC VERIFICATION
        # ====================================================

        for source_file in [
            POPULATION_TEMPLATE,
            SMOD_TEMPLATE,
            BUILT_TEMPLATE,
            DRIVER,
        ]:
            compile(
                source_file.read_text(
                    encoding="utf-8"
                ),
                str(source_file),
                "exec",
            )

        final_driver_tree = ast.parse(
            DRIVER.read_text(
                encoding="utf-8"
            )
        )

        final_checksums = None

        for node in final_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target in node.targets
            ):
                final_checksums = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if final_checksums != template_checksums:
            raise RuntimeError(
                "Driver template checksums do not "
                "match the patched templates."
            )


    except Exception:
        restore_backup()

        if PROGRAMS_ROOT.exists():
            shutil.rmtree(
                PROGRAMS_ROOT
            )

        PROGRAMS_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )

        raise


    print("=" * 80)
    print("MUMBAI COASTAL CANONICAL-GRID CORRECTION")
    print("=" * 80)

    print(
        "Backup:",
        BACKUP_ROOT,
    )

    print()
    print("Canonical grid cells: 4900")
    print("Population no-coverage value: 0.0")
    print("SMOD no-coverage class: 0")
    print("Built no-coverage value: 0.0")
    print("Population coverage indicator: ENABLED")
    print("SMOD coverage indicator: ENABLED")
    print("Built coverage indicator: ENABLED")
    print("Enrichment completeness logic: PATCHED")
    print("Template checksums refreshed: PASS")
    print("Workspace manifest refreshed: PASS")
    print("Generated programs reset: PASS")

    print(
        "Failed log preserved:",
        log_preserved,
    )

    print("Stale PID removed: PASS")
    print("Python compilation: PASS")
    print("HDFS modified: NO")
    print("Spark launched: NO")

    print(
        "MUMBAI COASTAL GRID CORRECTION: SUCCESS"
    )
    '''
).strip() + "\n"


# ============================================================
# VALIDATE AND ENCODE REMOTE PATCHER
# ============================================================

compile(
    remote_patcher,
    "patch_mumbai_coastal_grid.py",
    "exec",
)

patcher_sha256 = hashlib.sha256(
    remote_patcher.encode("utf-8")
).hexdigest()

print(
    "LOCAL MUMBAI COASTAL PATCHER COMPILE: PASS"
)

print(
    "Expected patcher SHA-256:",
    patcher_sha256,
)


patcher_b64 = base64.b64encode(
    remote_patcher.encode("utf-8")
).decode("ascii")


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

PATCHER="/tmp/patch_mumbai_coastal_grid.py"
PAYLOAD="/tmp/mumbai_coastal_patch_payload.json"
ROOT="{root}"

EXPECTED_PATCHER_SHA256="{patcher_sha256}"
EXPECTED_PAYLOAD_SHA256="{payload_sha256}"

echo "{patcher_b64}" \
| base64 -d \
> "${{PATCHER}}"

echo "{payload_b64}" \
| base64 -d \
> "${{PAYLOAD}}"

ACTUAL_PATCHER_SHA256="$(
    sha256sum "${{PATCHER}}" \
    | awk '{{print $1}}'
)"

ACTUAL_PAYLOAD_SHA256="$(
    sha256sum "${{PAYLOAD}}" \
    | awk '{{print $1}}'
)"

echo "Remote patcher SHA-256: ${{ACTUAL_PATCHER_SHA256}}"
echo "Remote payload SHA-256: ${{ACTUAL_PAYLOAD_SHA256}}"

if [ "${{ACTUAL_PATCHER_SHA256}}" != "${{EXPECTED_PATCHER_SHA256}}" ]; then
    echo "ERROR: Patcher checksum mismatch."
    exit 20
fi

if [ "${{ACTUAL_PAYLOAD_SHA256}}" != "${{EXPECTED_PAYLOAD_SHA256}}" ]; then
    echo "ERROR: Payload checksum mismatch."
    exit 21
fi

echo "REMOTE PATCHER CHECKSUM: PASS"
echo "REMOTE PAYLOAD CHECKSUM: PASS"

python3 -m py_compile "${{PATCHER}}"

echo "REMOTE PATCHER COMPILE: PASS"

python3 \
    "${{PATCHER}}" \
    "${{ROOT}}" \
    "${{PAYLOAD}}"
"""


patch_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6G RESULT")
print("=" * 80)

print(
    "Return code:",
    patch_result.returncode,
)

print("HDFS modified: NO")
print("Spark launched: NO")
print("CELL 6G: SUCCESS")

LOCAL MUMBAI COASTAL PATCHER COMPILE: PASS
Expected patcher SHA-256: 82c7b063e4c9c383931c94f669d918c0912d158d802fc05c7a61020bb2c5f84a
Remote patcher SHA-256: 82c7b063e4c9c383931c94f669d918c0912d158d802fc05c7a61020bb2c5f84a
Remote payload SHA-256: a73595b2a1c4366b7377fcd0230034f046584fe099b69992e0822d9fd13a33eb
REMOTE PATCHER CHECKSUM: PASS
REMOTE PAYLOAD CHECKSUM: PASS
REMOTE PATCHER COMPILE: PASS
MUMBAI COASTAL CANONICAL-GRID CORRECTION
Backup: /home/ubuntu/ghsl_multicity/mumbai_metro/coastal_patch_backup

Canonical grid cells: 4900
Population no-coverage value: 0.0
SMOD no-coverage class: 0
Built no-coverage value: 0.0
Population coverage indicator: ENABLED
SMOD coverage indicator: ENABLED
Built coverage indicator: ENABLED
Enrichment completeness logic: PATCHED
Template checksums refreshed: PASS
Workspace manifest refreshed: PASS
Generated programs reset: PASS
Failed log preserved: False
Stale PID removed: PASS
Python compilation: PASS
HDFS modified: NO
Spark launched: NO
MUMBAI COAS

In [27]:
# ============================================================
# CELL 6: ALL-EPOCH GENERATOR SMOKE TEST
#
# Generates and compiles:
#   10 population programs
#   10 SMOD programs
#   10 built-surface programs
#
# Total: 30 generated programs
#
# This cell does not:
#   - launch Spark
#   - execute generated programs
#   - read raster data
#   - write to HDFS
#   - modify the Mumbai workspace
#
# Generated programs are written only to /tmp and deleted.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_smoke_test = r'''
set -e

ROOT="__MUMBAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"

echo "============================================================"
echo "MUMBAI ALL-EPOCH GENERATOR SMOKE TEST"
echo "============================================================"

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Mumbai driver is missing:"
    echo "$DRIVER"
    exit 20
fi


python3 - "$ROOT" "$DRIVER" <<'PY'
from pathlib import Path
import ast
import py_compile
import shutil
import sys
import tempfile


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)


# ============================================================
# BUILD A DEFINITIONS-ONLY DRIVER MODULE
#
# Keep imports, constants, assignments, and functions.
# Exclude the top-level production loop so no Spark job starts.
# ============================================================

allowed_node_types = (
    ast.Import,
    ast.ImportFrom,
    ast.Assign,
    ast.AnnAssign,
    ast.FunctionDef,
    ast.AsyncFunctionDef,
    ast.ClassDef,
)

definition_nodes = [
    node
    for node in driver_tree.body
    if isinstance(
        node,
        allowed_node_types,
    )
]

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


namespace = {
    "__file__": str(driver_path),
    "__name__": "mumbai_generator_smoke_test",
}

exec(
    compile(
        definition_module,
        str(driver_path),
        "exec",
    ),
    namespace,
)


# ============================================================
# REQUIRED FACTORIES
# ============================================================

required_factories = {
    "population":
        "generate_population_program",
    "smod":
        "generate_smod_program",
    "built_surface":
        "generate_built_program",
}

factories = {}

for stage, function_name in (
    required_factories.items()
):
    factory = namespace.get(
        function_name
    )

    if not callable(factory):
        raise RuntimeError(
            "Required factory is unavailable: "
            + function_name
        )

    factories[stage] = factory


# ============================================================
# TEST CONFIGURATION
# ============================================================

epochs = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

expected_program_count = (
    len(epochs)
    * len(factories)
)

temporary_root = Path(
    tempfile.mkdtemp(
        prefix="mumbai_generator_smoke_"
    )
)

generated_records = []


# ============================================================
# GENERATE AND COMPILE ALL 30 PROGRAMS
# ============================================================

try:
    for epoch in epochs:
        epoch_directory = (
            temporary_root
            / ("epoch_" + str(epoch))
        )

        epoch_directory.mkdir()

        for stage, factory in (
            factories.items()
        ):
            generated_object = factory(
                epoch,
                epoch_directory,
            )

            # The production factories normally return a Path.
            # Support source text as a defensive fallback.
            if isinstance(
                generated_object,
                Path,
            ):
                program_path = generated_object

            elif (
                isinstance(
                    generated_object,
                    str,
                )
                and "\n"
                not in generated_object
            ):
                program_path = Path(
                    generated_object
                )

            elif isinstance(
                generated_object,
                str,
            ):
                program_path = (
                    epoch_directory
                    / (
                        stage
                        + "_"
                        + str(epoch)
                        + ".py"
                    )
                )

                program_path.write_text(
                    generated_object,
                    encoding="utf-8",
                )

            else:
                raise RuntimeError(
                    "Unexpected factory return type "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + type(
                        generated_object
                    ).__name__
                )


            if not program_path.is_file():
                raise RuntimeError(
                    "Factory did not create a program "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + str(program_path)
                )


            # Compile every generated program.
            py_compile.compile(
                str(program_path),
                doraise=True,
            )


            program_source = (
                program_path.read_text(
                    encoding="utf-8"
                )
            )


            # --------------------------------------------
            # Mumbai path validation
            # --------------------------------------------

            if (
                "city=mumbai_metro"
                not in program_source
            ):
                raise RuntimeError(
                    "Mumbai HDFS path is absent from "
                    + str(program_path)
                )

            if (
                "/home/ubuntu/ghsl_multicity/"
                "mumbai_metro"
                not in program_source
                and stage
                in {
                    "population",
                    "smod",
                    "built_surface",
                }
            ):
                # This is informational in most generated
                # programs, but reject only if neither Mumbai
                # workspace nor Mumbai HDFS identity appears.
                if (
                    "mumbai_metro"
                    not in program_source
                ):
                    raise RuntimeError(
                        "Mumbai identity is absent from "
                        + str(program_path)
                    )


            # --------------------------------------------
            # Hyderabad leakage validation
            # --------------------------------------------

            forbidden_values = [
                "city=hyderabad",
                "/home/ubuntu/ghsl_multicity/"
                "hyderabad",
                "Hyderabad Metropolitan Region",
            ]

            remaining_forbidden = [
                value
                for value
                in forbidden_values
                if value in program_source
            ]

            if remaining_forbidden:
                raise RuntimeError(
                    "Hyderabad leakage in "
                    + str(program_path)
                    + ": "
                    + repr(
                        remaining_forbidden
                    )
                )


            # --------------------------------------------
            # Active epoch validation
            # --------------------------------------------

            expected_epoch_assignment = (
                "EPOCH = "
                + str(epoch)
            )

            compact_program_source = (
                program_source.replace(
                    " ",
                    ""
                )
            )

            compact_expected_assignment = (
                expected_epoch_assignment.replace(
                    " ",
                    ""
                )
            )

            if (
                compact_expected_assignment
                not in compact_program_source
            ):
                raise RuntimeError(
                    "Active epoch assignment is "
                    "incorrect in "
                    + str(program_path)
                    + ". Expected "
                    + expected_epoch_assignment
                )


            # --------------------------------------------
            # Expected HDFS epoch path validation
            # --------------------------------------------

            expected_epoch_path = (
                "epoch="
                + str(epoch)
                + "/"
            )

            if (
                expected_epoch_path
                not in program_source
            ):
                raise RuntimeError(
                    "Expected epoch path is absent "
                    "from "
                    + str(program_path)
                    + ": "
                    + expected_epoch_path
                )


            generated_records.append(
                {
                    "epoch": epoch,
                    "stage": stage,
                    "path": str(
                        program_path
                    ),
                }
            )

            print(
                "PASS:",
                epoch,
                stage,
                program_path.name,
            )


    # ========================================================
    # FINAL COUNTS
    # ========================================================

    if (
        len(generated_records)
        != expected_program_count
    ):
        raise RuntimeError(
            "Expected "
            + str(
                expected_program_count
            )
            + " generated programs, found "
            + str(
                len(generated_records)
            )
        )


    stage_counts = {}

    for stage in factories:
        stage_counts[stage] = sum(
            1
            for record
            in generated_records
            if record["stage"]
            == stage
        )


    expected_stage_count = len(
        epochs
    )

    for stage, count in (
        stage_counts.items()
    ):
        if count != expected_stage_count:
            raise RuntimeError(
                "Expected "
                + str(
                    expected_stage_count
                )
                + " "
                + stage
                + " programs, found "
                + str(count)
            )


    print()

    print(
        "Population programs:",
        stage_counts["population"],
    )

    print(
        "SMOD programs:",
        stage_counts["smod"],
    )

    print(
        "Built-surface programs:",
        stage_counts[
            "built_surface"
        ],
    )

    print(
        "Total generated programs:",
        len(generated_records),
    )

    print()

    print(
        "ALL_30_GENERATED_PROGRAMS_COMPILE_PASS"
    )


finally:
    # No generated smoke-test artifacts persist.
    if temporary_root.exists():
        shutil.rmtree(
            temporary_root
        )


print(
    "Temporary smoke-test directory removed: PASS"
)

print(
    "ALL_EPOCH_GENERATOR_SMOKE_TEST_SUCCESS"
)
PY


echo
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "Mumbai workspace modified: NO"
echo "CELL_6_REMOTE_SUCCESS"
'''


# Replace a literal placeholder.
# Do not wrap the embedded Python in an outer f-string.

cmd = remote_smoke_test.replace(
    "__MUMBAI_ROOT__",
    root,
)


smoke_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6 RESULT")
print("=" * 80)

print(
    "Return code:",
    smoke_result.returncode,
)

print(
    "Expected generated programs:",
    30,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Mumbai workspace modified: NO")
print("CELL 6: SUCCESS")

MUMBAI ALL-EPOCH GENERATOR SMOKE TEST
PASS: 1975 population population_1975.py
PASS: 1975 smod smod_1975.py
PASS: 1975 built_surface built_surface_1975.py
PASS: 1980 population population_1980.py
PASS: 1980 smod smod_1980.py
PASS: 1980 built_surface built_surface_1980.py
PASS: 1985 population population_1985.py
PASS: 1985 smod smod_1985.py
PASS: 1985 built_surface built_surface_1985.py
PASS: 1990 population population_1990.py
PASS: 1990 smod smod_1990.py
PASS: 1990 built_surface built_surface_1990.py
PASS: 1995 population population_1995.py
PASS: 1995 smod smod_1995.py
PASS: 1995 built_surface built_surface_1995.py
PASS: 2000 population population_2000.py
PASS: 2000 smod smod_2000.py
PASS: 2000 built_surface built_surface_2000.py
PASS: 2005 population population_2005.py
PASS: 2005 smod smod_2005.py
PASS: 2005 built_surface built_surface_2005.py
PASS: 2010 population population_2010.py
PASS: 2010 smod smod_2010.py
PASS: 2010 built_surface built_surface_2010.py
PASS: 2015 population popu

In [28]:
# ============================================================
# CELL 7: DETACHED MUMBAI PRODUCTION LAUNCH
#
# Launches the full 1975-2020 Mumbai production pipeline.
#
# Safety:
#   - Refuses a duplicate active launch
#   - Loads Hadoop/Sedona environment
#   - Verifies HDFS daemons and safe mode
#   - Verifies Mumbai driver and template compilation
#   - Writes PID before returning
#   - Detects immediate startup failure
#   - Leaves the job detached on EC2
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_launch = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

ROOT="__MUMBAI_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"


echo "============================================================"
echo "MUMBAI PRODUCTION LAUNCH"
echo "============================================================"


# ------------------------------------------------------------
# Verify HDFS services.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Verify Mumbai production files.
# ------------------------------------------------------------

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Mumbai driver is missing."
    exit 20
fi

python3 -m py_compile "$DRIVER"

for template in "$ROOT"/templates/*.py; do
    python3 -m py_compile "$template"
done

echo "Mumbai Python compilation: PASS"


# ------------------------------------------------------------
# Refuse duplicate launch.
# ------------------------------------------------------------

if [ -f "$PID_FILE" ]; then
    OLD_PID="$(cat "$PID_FILE")"

    if kill -0 "$OLD_PID" 2>/dev/null; then
        echo "ERROR: Mumbai job is already running."
        echo "Active PID: $OLD_PID"
        exit 30
    fi

    echo "Removing stale PID file:"
    echo "$PID_FILE"

    rm -f "$PID_FILE"
fi


# ------------------------------------------------------------
# Preserve an old log if one unexpectedly exists.
# ------------------------------------------------------------

if [ -f "$LOG_FILE" ]; then
    LOG_BACKUP="$LOG_FILE.before_launch.$(
        date -u +%Y%m%dT%H%M%SZ
    )"

    mv "$LOG_FILE" "$LOG_BACKUP"

    echo "Previous log moved to:"
    echo "$LOG_BACKUP"
fi


# ------------------------------------------------------------
# Launch detached production driver.
# ------------------------------------------------------------

nohup python3 "$DRIVER" \
    >"$LOG_FILE" \
    2>&1 \
    </dev/null &

NEW_PID="$!"

echo "$NEW_PID" > "$PID_FILE"

echo "New PID: $NEW_PID"
echo "Log: $LOG_FILE"


# ------------------------------------------------------------
# Allow startup checks to run.
# ------------------------------------------------------------

sleep 5


# ------------------------------------------------------------
# Confirm process survived startup.
# ------------------------------------------------------------

if ! kill -0 "$NEW_PID" 2>/dev/null; then
    echo
    echo "ERROR: Mumbai driver exited during startup."

    echo
    echo "Latest 100 log lines:"

    if [ -f "$LOG_FILE" ]; then
        tail -n 100 "$LOG_FILE"
    fi

    exit 31
fi


echo

ps -p "$NEW_PID" \
    -o pid,etime,%cpu,%mem,cmd

echo
echo "DETACHED_LAUNCH_SUCCESS"
'''


cmd = remote_launch.replace(
    "__MUMBAI_ROOT__",
    root,
)


launch_result = run_remote(
    cmd,
    timeout=90,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7 RESULT")
print("=" * 80)

print(
    "Return code:",
    launch_result.returncode,
)

print("Safe to close Colab.")
print("Monitor with Cell 8.")
print("CELL 7: SUCCESS")

MUMBAI PRODUCTION LAUNCH
HDFS services: PASS
Mumbai Python compilation: PASS
New PID: 8730
Log: /home/ubuntu/ghsl_multicity/mumbai_metro/batch/city.log

    PID     ELAPSED %CPU %MEM CMD
   8730       00:05  0.3  0.2 python3 /home/ubuntu/ghsl_multicity/mumbai_metro/batch/run_city_epochs.py

DETACHED_LAUNCH_SUCCESS

REMOTE STDERR:


CELL 7 RESULT
Return code: 0
Safe to close Colab.
Monitor with Cell 8.
CELL 7: SUCCESS


In [39]:
# ============================================================
# CELL 8: STABLE MUMBAI MONITOR WITH CLEAR FINAL STATUS
#
# Read-only:
#   - Does not launch a job
#   - Does not stop a job
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]
city_name = ACTIVE_CITY["city_name"]

if ACTIVE_CITY_SLUG != "mumbai_metro":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be mumbai_metro."
    )


remote_monitor = r'''
set +e

source /home/ubuntu/ghsl-sedona-env.sh 2>/dev/null

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

ROOT="__MUMBAI_ROOT__"
CITY_NAME="__CITY_NAME__"

PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"
MANIFEST="$ROOT/batch/batch_manifest.tsv"


get_job_status() {
    RECORDED_PID="N/A"
    JOB_STATUS="UNKNOWN"

    if [ -f "$PID_FILE" ]; then
        RECORDED_PID="$(cat "$PID_FILE")"

        if kill -0 "$RECORDED_PID" 2>/dev/null; then
            JOB_STATUS="RUNNING"
        else
            JOB_STATUS="STOPPED"
        fi
    fi
}


# ============================================================
# INITIAL STATUS
# ============================================================

get_job_status

echo "============================================================"
echo "MUMBAI JOB STATUS"
echo "============================================================"

echo "City: $CITY_NAME"
echo "Recorded PID: $RECORDED_PID"
echo "STATUS=$JOB_STATUS"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd
fi


# ============================================================
# LATEST LOG
# ============================================================

echo
echo "============================================================"
echo "LATEST 150 LOG LINES"
echo "============================================================"

if [ -f "$LOG_FILE" ]; then
    tail -n 150 "$LOG_FILE"
else
    echo "Log file is missing."
fi


# ============================================================
# CURRENT MANIFEST
# ============================================================

echo
echo "============================================================"
echo "CURRENT MUMBAI MANIFEST"
echo "============================================================"

if [ -f "$MANIFEST" ]; then
    cat "$MANIFEST"
else
    echo "Manifest has not been created."
fi


# ============================================================
# SIMPLE PROGRESS COUNTS
# ============================================================

echo
echo "============================================================"
echo "CURRENT PROGRESS SUMMARY"
echo "============================================================"

COMPLETED_STAGES=0
COMPLETED_EPOCHS=0

if [ -f "$MANIFEST" ]; then
    COMPLETED_STAGES="$(
        grep -E \
            $'\t(built_and_validated|existing_valid_skipped)\t' \
            "$MANIFEST" \
            | wc -l \
            | tr -d ' '
    )"

    for epoch in \
        1975 1980 1985 1990 1995 \
        2000 2005 2010 2015 2020
    do
        EPOCH_RECORDS="$(
            awk -F '\t' \
                -v target_epoch="$epoch" \
                '$1 == target_epoch {print}' \
                "$MANIFEST" \
                | grep -E \
                    $'\t(built_and_validated|existing_valid_skipped)\t' \
                | wc -l \
                | tr -d ' '
        )"

        if [ "$EPOCH_RECORDS" = "4" ]; then
            COMPLETED_EPOCHS="$(
                expr "$COMPLETED_EPOCHS" + 1
            )"
        fi
    done
fi

echo "Completed stage records: $COMPLETED_STAGES / 40"
echo "Fully completed epochs: $COMPLETED_EPOCHS / 10"


# ============================================================
# ACTIVE JAVA PROCESSES
# ============================================================

echo
echo "============================================================"
echo "ACTIVE SPARK / JAVA PROCESSES"
echo "============================================================"

jps


# ============================================================
# FINAL STATUS
# ============================================================

get_job_status

COMPLETION_RESULT="UNKNOWN"
FINAL_MESSAGE="MUMBAI JOB STATUS UNKNOWN"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    COMPLETION_RESULT="IN_PROGRESS"
    FINAL_MESSAGE="MUMBAI JOB IS STILL RUNNING"
fi

if [ "$JOB_STATUS" = "STOPPED" ]; then
    if [ -f "$LOG_FILE" ]; then
        if grep -q "BATCH PIPELINE: SUCCESS" "$LOG_FILE"; then
            COMPLETION_RESULT="SUCCESS"
            FINAL_MESSAGE="MUMBAI JOB COMPLETED SUCCESSFULLY"
        elif grep -q "Traceback (most recent call last)" "$LOG_FILE"; then
            COMPLETION_RESULT="FAILED"
            FINAL_MESSAGE="MUMBAI JOB STOPPED WITH FAILURE"
        else
            COMPLETION_RESULT="UNKNOWN"
            FINAL_MESSAGE="MUMBAI JOB STOPPED, RESULT UNKNOWN"
        fi
    fi
fi


# ============================================================
# FINAL BLOCK
# ============================================================

echo
echo
echo "############################################################"
echo "############################################################"
echo
echo "              FINAL MUMBAI PIPELINE STATUS"
echo
echo "City: $CITY_NAME"
echo "PID: $RECORDED_PID"
echo "FINAL_STATUS=$JOB_STATUS"
echo "COMPLETION_RESULT=$COMPLETION_RESULT"
echo "COMPLETED_STAGES=$COMPLETED_STAGES/40"
echo "COMPLETED_EPOCHS=$COMPLETED_EPOCHS/10"
echo
echo ">>> $FINAL_MESSAGE <<<"
echo

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd

    echo
    echo "Action: Let the EC2 job continue."
    echo "Do not rerun Cell 7."
fi

if [ "$COMPLETION_RESULT" = "SUCCESS" ]; then
    echo "Action: Run Cell 9 final audit."
fi

if [ "$COMPLETION_RESULT" = "FAILED" ]; then
    echo "Latest failure markers:"

    grep -n \
        -E \
        "Traceback|RuntimeError|AnalysisException|Return code: 1" \
        "$LOG_FILE" \
        | tail -n 20

    echo
    echo "Action: Do not rerun Cell 7."
fi

echo
echo "############################################################"
echo "############################################################"

exit 0
'''


cmd = (
    remote_monitor
    .replace(
        "__MUMBAI_ROOT__",
        root,
    )
    .replace(
        "__CITY_NAME__",
        city_name,
    )
)


monitor_result = run_remote(
    cmd,
    timeout=180,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 8 RESULT")
print("=" * 80)

print(
    "Monitor return code:",
    monitor_result.returncode,
)

print("No new job was launched.")
print("No running process was stopped.")
print("No HDFS data was modified.")

MUMBAI JOB STATUS
City: Mumbai Metropolitan Region
Recorded PID: 8730
STATUS=STOPPED

LATEST 150 LOG LINES
VALIDATE NEW ENRICHED 2015
Command: spark-submit --master local[1] --driver-memory 1g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=2 --conf spark.default.parallelism=2 --conf spark.sql.adaptive.enabled=false --conf spark.sql.codegen.wholeStage=false --conf spark.driver.maxResultSize=256m /home/ubuntu/ghsl_multicity/mumbai_metro/programs/epoch_2015/validate_enriched_2015.py


Return code: 0
Elapsed seconds: 11.32
enriched 2015: BUILT AND VALIDATED

EPOCH 2015: COMPLETE
Epoch elapsed minutes: 12.44

################################################################################
EPOCH 2020 (10/10)
################################################################################

RUN POPULATION 2020
Command: spark-submit --master local[1] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=4 --conf spark.default.parallelism=2 --con

In [40]:
# CELL 9: FINAL 40-PATH AUDIT
root=ACTIVE_CITY['remote_work_dir']; slug=ACTIVE_CITY_SLUG
cmd=f"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH=${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}
grep -q 'BATCH PIPELINE: SUCCESS' {root}/batch/city.log
n=0
for e in 1975 1980 1985 1990 1995 2000 2005 2010 2015 2020; do
 for s in population smod built_surface; do hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/product=$s/epoch=$e/grid_1km; n=$((n+1)); done
 hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/enriched_grid_year/epoch=$e; n=$((n+1)); echo Epoch $e: 4/4 paths PASS
done
test $n -eq 40; echo ALL_40_PATHS_PASS; echo CITY_COMPLETE
"""
run_remote(cmd,timeout=300); print('CELL 9: SUCCESS')


Epoch 1975: 4/4 paths PASS
Epoch 1980: 4/4 paths PASS
Epoch 1985: 4/4 paths PASS
Epoch 1990: 4/4 paths PASS
Epoch 1995: 4/4 paths PASS
Epoch 2000: 4/4 paths PASS
Epoch 2005: 4/4 paths PASS
Epoch 2010: 4/4 paths PASS
Epoch 2015: 4/4 paths PASS
Epoch 2020: 4/4 paths PASS
ALL_40_PATHS_PASS
CITY_COMPLETE

REMOTE STDERR:

CELL 9: SUCCESS


The completed data is under:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/city=mumbai_metro/

The primary modelling inputs are:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/
city=mumbai_metro/enriched_grid_year/epoch=<YEAR>